# Name That LLM: Quad-Model Frontier Ensemble (V6 Production Pipeline)
Production pipeline for Kaggle 'Name That LLM' competition.


In [ ]:
"""
Name That LLM: Quad-Model Frontier Ensemble (V6 Production Pipeline)
DeBERTa-v3-large FT (GPU 0) + BGE-large-en-v1.5 FT (GPU 1) + Qwen 3 8B / 2.5 7B LoRA (Dual GPU) + CPU Multiview Linear (~850k).
Standalone, offline, self-contained Kaggle submission notebook.

Target Metric: Macro Balanced Accuracy across 18 classes (>0.70+).

Key Architectural Components:
  1. DeBERTa-v3-large Full FT -> cuda:0 (BS=16, Head+Tail L=448, LLRD xi=0.90, StylometricMultiMomentPooler)
  2. BGE-large-en-v1.5 Full FT -> cuda:1 (BS=20, Head+Tail L=448, LLRD xi=0.85, Multi-Moment Pooling)
  3. Qwen 3 8B / 2.5 7B LoRA  -> device_map="auto" sharded across dual GPUs (r=48, SpikeAndGapHierarchicalHead)
  4. CPU Multiview Dual-Linear -> ~850k sparse TF-IDF + 145 stylometrics + IntraFamilyCliqueSpecialist
  5. Zero-VRAM External Data   -> 24k passages with Kish ESS-capped mass conservation (0.000% prior skew)
  6. Meta-Combiner & Consensus -> Nested 5-fold CV CPU autograd L-BFGS + Hierarchical Soft-Tree + Sinkhorn balancing
"""
from collections import Counter
import concurrent.futures
import contextlib
import copy
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import queue
import re
import shutil
import subprocess
import sys
import threading
import time
import warnings

SCRIPT_T0 = time.perf_counter()

# 1. Disable Tokenizer Parallelism Deadlock & HF Internal Thread Contention & Broken Jupyter Widgets
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

# 2. Enforce Strict Offline & Disable Hugging Face Transfer / XET hangs
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

# 3. PyTorch CUDA memory allocator configuration to avoid fragmentation hangs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 4. Bypass Kaggle container bug where pre-installed torchao is 0.10.0 but peft requires >= 0.16.0
sys.modules['torchao'] = None

# 5. Ensure line-buffered stdout so logs stream immediately without buffering
try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass

# 6. Offline Dependency Bootstrap: Auto-upgrade transformers & tokenizers if wheels are attached
def bootstrap_offline_wheels():
    """
    Auto-discover and install offline wheel packages under /kaggle/input (e.g. transformers >= 4.50 for Qwen 3).
    Safe, silent no-op when running with standard pre-installed libraries or when no wheels are present.
    """
    input_roots = []
    local_data = os.environ.get("NTL_DATA_DIR")
    if local_data:
        input_roots.append(Path(local_data))
    input_roots.append(Path("/kaggle/input"))
    if os.environ.get("NTL_WHEELS_DIR"):
        input_roots.insert(0, Path(os.environ["NTL_WHEELS_DIR"]))

    wheel_dirs = set()
    for root in input_roots:
        if root.exists():
            try:
                root_depth = len(root.parts)
                for cur_root, dirs, files in os.walk(root):
                    cur_depth = len(Path(cur_root).parts) - root_depth
                    dirs[:] = [d for d in dirs if not d.startswith(".")] if cur_depth < 3 else []
                    for fname in files:
                        if fname.startswith("transformers") and fname.endswith(".whl"):
                            wheel_dirs.add(Path(cur_root))
            except Exception:
                pass

    if wheel_dirs:
        for wdir in sorted(wheel_dirs):
            print(f"[Offline Bootstrap] Found wheel directory at: {wdir}. Installing upgraded transformers...", flush=True)
            try:
                res = subprocess.run(
                    [sys.executable, "-m", "pip", "install", "--no-index", f"--find-links={wdir}", "transformers", "tokenizers", "peft"],
                    capture_output=True, text=True, timeout=120
                )
                if res.returncode == 0:
                    import importlib
                    importlib.invalidate_caches()
                    print(f"[Offline Bootstrap] Upgrade successful from {wdir}.", flush=True)
                    break
                else:
                    print(f"[Offline Bootstrap] pip returned {res.returncode}: {res.stderr[:160]}", flush=True)
            except Exception as e:
                print(f"[Offline Bootstrap] Installation error: {e}", flush=True)

bootstrap_offline_wheels()

# 7. Global Warning Controls
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import Nystroem
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)
try:
    import transformers
    transformers.logging.disable_progress_bar()
    transformers.logging.set_verbosity_error()
except Exception:
    pass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

# Enable cuDNN auto-tuning & high float32 matrix multiplication precision
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

class HeartbeatTimer:
    """Prints live elapsed seconds during long blocking operations (e.g. Kaggle virtual disk I/O)."""
    def __init__(self, message, interval=8.0):
        self.message = message
        self.interval = interval
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.start_time = None

    def __enter__(self):
        self.start_time = time.time()
        self.thread.start()
        return self

    def _run(self):
        while not self.stop_event.wait(self.interval):
            elapsed = time.time() - self.start_time
            print(f"    ... [Heartbeat] {self.message} ({elapsed:.0f}s elapsed - disk I/O active, not stuck) ...", flush=True)

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.stop_event.set()
        elapsed = time.time() - self.start_time
        print(f"    -> [Heartbeat] Completed in {elapsed:.1f}s.", flush=True)

def amp_autocast():
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda")
    return torch.cuda.amp.autocast()

def inference_context():
    """Inference context with PyTorch inference_mode for maximum dispatch throughput."""
    if hasattr(torch, "inference_mode"):
        return torch.inference_mode()
    return torch.no_grad()

def get_mem_status(device=None):
    """Returns live formatted VRAM and System RAM utilization."""
    vram_items = []
    if torch.cuda.is_available():
        if device is not None:
            try:
                d = torch.device(device)
                idx = d.index if d.index is not None else 0
                alloc = torch.cuda.memory_allocated(idx) / (1024 ** 3)
                res = torch.cuda.memory_reserved(idx) / (1024 ** 3)
                vram_items.append(f"{d}:{alloc:.1f}/{res:.1f}GB")
            except Exception:
                pass
        else:
            for i in range(torch.cuda.device_count()):
                alloc = torch.cuda.memory_allocated(i) / (1024 ** 3)
                res = torch.cuda.memory_reserved(i) / (1024 ** 3)
                vram_items.append(f"cuda:{i}:{alloc:.1f}/{res:.1f}GB")

    vram_str = "VRAM[" + " | ".join(vram_items) + "]" if vram_items else "CPU"
    ram_str = "RAM"
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram_str = f"RAM:{vm.used / (1024 ** 3):.1f}/{vm.total / (1024 ** 3):.1f}GB ({vm.percent:.0f}%)"
    except Exception:
        pass

    return f"[{vram_str} | {ram_str}]"

# --- 1. COMPETITION CONSTANTS ---

LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5",
    "claude-sonnet-5", "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",
]
LABEL_TO_IDX = {label: i for i, label in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)

FAMILY_MAP = {
    "gpt-5.5": "openai", "gpt-5.6-luna": "openai", "gpt-5.6-sol": "openai",
    "gpt-5.6-terra": "openai", "gpt-6-astra": "openai",
    "gemini-3.1-pro-preview": "google", "gemini-3.6-flash": "google",
    "gemini-3.7-flash": "google", "gemini-3.8-flash": "google",
    "claude-fable-5": "anthropic", "claude-opus-4-6": "anthropic",
    "claude-opus-4-8": "anthropic", "claude-opus-5": "anthropic", "claude-sonnet-5": "anthropic",
    "qwen3.8-max": "qwen",
    "glm-5.3": "zhipu", "glm-5.3-flash": "zhipu",
    "kimi-k3": "moonshot",
}
FAMILIES = sorted(set(FAMILY_MAP.values()))
FAMILY_TO_INDICES = {
    fam: [i for i, lab in enumerate(LABELS) if FAMILY_MAP[lab] == fam]
    for fam in FAMILIES
}
FAMILY_GROUPS = FAMILY_TO_INDICES  # Canonical alias matching hierarchical tree naming
FAMILY_TO_IDX = {fam: i for i, fam in enumerate(FAMILIES)}
LABEL_TO_FAMILY_IDX = np.array([FAMILY_TO_IDX[FAMILY_MAP[l]] for l in LABELS], dtype=np.int64)

# Sibling difficulty angular margins for FocalLoss (prioritizes hard sibling cliques over distinct outliers)
SIBLING_DIFFICULTY_MARGINS = [
    0.12,  # gpt-5.5
    0.25,  # gpt-5.6-luna (sibling clique)
    0.25,  # gpt-5.6-sol (sibling clique)
    0.25,  # gpt-5.6-terra (sibling clique)
    0.10,  # gpt-6-astra
    0.08,  # gemini-3.1-pro-preview (distinct architecture/style)
    0.32,  # gemini-3.6-flash (hardest sibling triplet)
    0.32,  # gemini-3.7-flash (hardest sibling triplet)
    0.32,  # gemini-3.8-flash (hardest sibling triplet)
    0.20,  # claude-fable-5
    0.20,  # claude-opus-4-6
    0.22,  # claude-opus-4-8
    0.12,  # claude-opus-5
    0.15,  # claude-sonnet-5
    0.08,  # qwen3.8-max (distinct vocabulary/tokenizer)
    0.24,  # glm-5.3
    0.24,  # glm-5.3-flash
    0.08,  # kimi-k3 (distinct Chinese/English reasoning patterns)
]

def compute_family_logits(model_logits: torch.Tensor, temp: float = 4.0) -> torch.Tensor:
    """
    Derive per-family logits by temperature-smoothed log-sum-exp pooling within each family.
    Scaling by temp=4.0 prevents hard max() saturation, ensuring family gradients distribute
    smoothly across all sibling models rather than starving non-top siblings.
    Pure functional construction avoids in-place slice mutation for kernel fusion & clean autograd.
    Shape: (N, 18) -> (N, n_families)
    """
    scaled = model_logits / max(float(temp), 1e-4)
    fam_cols = [
        float(temp) * torch.logsumexp(scaled[:, FAMILY_TO_INDICES[fam]], dim=1)
        for fam in FAMILIES
    ]
    return torch.stack(fam_cols, dim=1)

def seed_everything(seed: int = 42):
    """Seed Python, NumPy, PyTorch (CPU & CUDA) for full deterministic reproducibility."""
    import random
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def compute_smoothed_class_weights(y_indices, num_classes=18, gamma=0.5):
    """
    Computes smoothed class weights to prevent gradient spikes on rare
    classes while aligning optimization with Macro Balanced Accuracy.
    Caps max weight disparity at ~6.0x (clipping to [0.5, 4.5]).
    """
    counts = np.bincount(y_indices, minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    median_count = np.median(counts)
    weights = (median_count / counts) ** gamma
    weights = np.clip(weights, 0.5, 4.5)
    weights = weights / np.mean(weights)
    return weights

# --- 2. TEXT NORMALIZATION & STYLISTIC SIGNAL PRESERVATION ---

def clean_text(text: str) -> str:
    """Normalize whitespace and line endings while preserving casing, punctuation, and markdown."""
    if not isinstance(text, str):
        return ""
    s = text.replace("\r\n", "\n").replace("\r", "\n")
    s = s.replace("\u00a0", " ").replace("\u200b", "").replace("\ufeff", "")
    s = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", s)
    return s.strip()


def compute_fold_variance_capped_weights(
    y_comp_fold: np.ndarray,
    y_ext: np.ndarray,
    ext_n_models: np.ndarray,
    n_classes: int = 18,
    ext_mass_ratio: float = 0.55,
    max_single_ext_weight: float = 0.35,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Guarantees:
      1. Exact 0.000% class prior skew across all 18 classes: Sum_{i in c} (w_comp + w_ext) is constant.
      2. Bounded Estimator Variance: No single external sample weight ever exceeds max_single_ext_weight (0.35).
         Any mass deficit for starved classes (e.g. gemini-3.8-flash, N=71) is added directly to that
         class's official competition samples so the class prior stays 100% uniform!
      3. Regularization Invariance: Total weight sum equals len(y_comp_fold) so LinearSVC(C) and Ridge(alpha)
         penalties never shift when external data is attached.
    """
    n_comp = len(y_comp_fold)
    target_class_comp = n_comp / float(n_classes)

    w_comp = np.ones(n_comp, dtype=np.float64)
    w_ext = np.zeros(len(y_ext), dtype=np.float64)

    # Reward multi-model rows (2.5x) over single-model rows (1.0x)
    raw_intra = np.where(ext_n_models >= 2, 2.5, 1.0)
    total_target_per_class = target_class_comp * (1.0 + ext_mass_ratio)

    for c in range(n_classes):
        c_label = LABELS[c] if c < len(LABELS) else c
        comp_mask = (y_comp_fold == c) | (y_comp_fold == c_label)
        ext_mask = (y_ext == c) | (y_ext == c_label)
        n_c_comp = max(int(comp_mask.sum()), 1)

        desired_ext_mass = target_class_comp * ext_mass_ratio
        raw_c_sum = raw_intra[ext_mask].sum()

        if raw_c_sum > 0:
            w_c_ext = raw_intra[ext_mask] * (desired_ext_mass / raw_c_sum)
            # Cap individual external sample weights to prevent variance explosion on starved classes
            w_c_ext = np.minimum(w_c_ext, max_single_ext_weight)
            w_ext[ext_mask] = w_c_ext
            actual_ext_mass = w_c_ext.sum()
        else:
            actual_ext_mass = 0.0

        # Absorb any capped deficit into the official competition samples of class c
        rem_comp_mass = total_target_per_class - actual_ext_mass
        w_comp[comp_mask] = rem_comp_mass / float(n_c_comp)

    # Normalize global sum to equal n_comp so LinearSVC(C) and Ridge(alpha) stay invariant
    total_sum = w_comp.sum() + w_ext.sum()
    scale = float(n_comp) / max(total_sum, 1e-12)
    return w_comp * scale, w_ext * scale


def align_external_dense_moments(
    V_comp: np.ndarray,
    V_ext: np.ndarray,
    w_ext: np.ndarray,
    eps: float = 1e-6,
) -> np.ndarray:
    """
    Aligns weighted mean and std of external dense stylometric features to the competition fold
    so the classifier cannot split on domain covariate shift.
    """
    w_norm = w_ext[:, None] / max(float(w_ext.sum()), eps)
    mu_ext = np.sum(V_ext * w_norm, axis=0, keepdims=True)
    var_ext = np.sum(w_norm * (V_ext - mu_ext) ** 2, axis=0, keepdims=True)
    std_ext = np.maximum(np.sqrt(np.maximum(var_ext, 0.0)), eps)

    mu_comp = V_comp.mean(axis=0, keepdims=True)
    std_comp = np.maximum(V_comp.std(axis=0, keepdims=True), eps)

    return (((V_ext - mu_ext) / std_ext) * std_comp + mu_comp).astype(V_comp.dtype)


def build_mass_balanced_external_corpus(
    comp_train_df: pd.DataFrame,
    ext_wide_path: str | Path,
    labels: list[str],
    ext_mass_ratio: float = 0.55,
    prefer_multi_model: bool = True,
) -> tuple[pd.DataFrame, np.ndarray]:
    """
    Converts wide external data into a mass-conserved long dataframe where:
      Sum_{i in comp, class=c} w_i = N_comp_c
      Sum_{j in ext,  class=c} w_j = ext_mass_ratio * N_comp_c  (Identical across all 18 classes!)
    Guarantees 0.0% class prior skew, bounded individual variance (<=0.35),
    and exact regularization invariance for scikit-learn LinearSVC and Ridge.
    Namespaces external prompt IDs ('ext_{id}') to prevent prompt collisions.
    Strips NUL bytes and cleans whitespace.
    """
    ext_p = Path(ext_wide_path)
    suffix = ext_p.suffix.lower()
    if suffix == ".csv":
        wide = pd.read_csv(ext_p)
    elif suffix in (".jsonl", ".ndjson"):
        wide = pd.read_json(ext_p, lines=True)
    else:
        wide = pd.read_parquet(ext_p)
    model_cols = [c for c in labels if c in wide.columns]

    if len(model_cols) >= 2:
        # Wide format (1 row per prompt, 1 column per model)
        if "id" not in wide.columns:
            if "prompt_id" in wide.columns:
                wide["id"] = wide["prompt_id"].astype(str)
            else:
                wide["id"] = [f"wrow_{i}" for i in range(len(wide))]
        wide["_n_models"] = wide[model_cols].notna().sum(axis=1)
        ext_long = wide.melt(
            id_vars=["id", "_n_models"],
            value_vars=model_cols,
            var_name="model",
            value_name="text",
        ).dropna(subset=["text"]).reset_index(drop=True)
        ext_long["prompt_id"] = "ext_" + ext_long["id"].astype(str)
    elif "model" in wide.columns and "text" in wide.columns:
        # Long format fallback (e.g., clean_corpus_long.parquet)
        ext_long = wide[wide["model"].isin(labels)].dropna(subset=["text"]).copy().reset_index(drop=True)
        if "prompt_id" in ext_long.columns:
            raw_pid = ext_long["prompt_id"].fillna("").astype(str)
            pid_vals = [p if p else f"lrow_{i}" for i, p in enumerate(raw_pid)]
        elif "id" in ext_long.columns:
            pid_vals = ext_long["id"].astype(str).tolist()
        else:
            pid_vals = [f"lrow_{i}" for i in range(len(ext_long))]
        ext_long["prompt_id"] = ["ext_" + p for p in pid_vals]
        if "_n_models" not in ext_long.columns:
            ext_long["_n_models"] = ext_long.groupby("prompt_id")["model"].transform("nunique")
    else:
        raise ValueError(f"Unsupported external corpus schema at {ext_wide_path}: columns={list(wide.columns)[:12]}")

    del wide

    # Clean text and strip NUL bytes
    ext_long["text"] = ext_long["text"].apply(clean_text)
    ext_long = ext_long[ext_long["text"].str.len() > 0].reset_index(drop=True)
    if len(ext_long) > 50_000:
        # Filter oversized uncurated wide dumps (e.g., external_all_wide) to the competition 40-300 word window
        wc = ext_long["text"].str.split().str.len()
        ext_long = ext_long[(wc >= 40) & (wc <= 300)].reset_index(drop=True)
    if len(ext_long) == 0:
        raise ValueError(f"External corpus at {ext_wide_path} produced 0 valid rows after cleaning.")

    label_to_id = {m: i for i, m in enumerate(labels)}
    ext_long["label"] = ext_long["model"].map(label_to_id)

    comp_labels = comp_train_df["label"].values if "label" in comp_train_df.columns else comp_train_df["model"].map(label_to_id).values
    w_comp, w_ext = compute_fold_variance_capped_weights(
        comp_labels, ext_long["label"].values, ext_long["_n_models"].values,
        n_classes=len(labels), ext_mass_ratio=ext_mass_ratio, max_single_ext_weight=0.35,
    )

    # Standardize column sets
    comp_sub = comp_train_df[["text", "model", "prompt_id"]].copy()
    comp_sub["label"] = comp_labels
    comp_sub["_n_models"] = 18

    ext_sub = ext_long[["text", "model", "prompt_id", "label", "_n_models"]].copy()

    combined_df = pd.concat([comp_sub, ext_sub], ignore_index=True)
    combined_weights = np.concatenate([w_comp, w_ext], axis=0)
    return combined_df, combined_weights


def prep_transformer_text(text: str) -> str:
    """Map newlines to visible [NL] tokens so SentencePiece/WordPiece preserve line/list cadence."""
    if not isinstance(text, str):
        return ""
    s = text.replace("\r\n", "\n").replace("\r", "\n")
    s = s.replace("\u00a0", " ").replace("\u200b", "").replace("\ufeff", "")
    s = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", s)
    s = s.replace("\n\n", " [PAR] ").replace("\n", " [NL] ")
    return s.strip()


def tokenize_with_head_tail(
    tokenizer, texts, max_length=448, head_tokens=None, tail_tokens=None,
    preserve_newlines=True, extra_head_tail=None,
):
    """
    Tokenize sequences with balanced Head+Tail truncation (224 tokens each for 448 context).
    Preserves opening thesis and concluding sign-off signatures where LLM stylistic patterns peak.
    For encoder models (DeBERTa/BGE), preserve_newlines maps newlines to dedicated [NL]/[PAR] tokens.
    For decoder models (Qwen), raw newlines are preserved natively.
    When extra_head_tail=(h2, t2) is supplied, derives a second truncation view from the same
    tokenizer pass (eliminating duplicate BPE/SentencePiece tokenization for TTA).
    Returns unpadded token lists (or tuple of two dicts when extra_head_tail is provided).
    """
    if head_tokens is None:
        head_tokens = max_length // 2
    if tail_tokens is None:
        tail_tokens = max_length - head_tokens
    if preserve_newlines:
        prep_texts = [prep_transformer_text(t) for t in texts]
    else:
        prep_texts = [clean_text(t) for t in texts]
    enc = tokenizer(prep_texts, truncation=False, padding=False)

    has_token_types = "token_type_ids" in enc
    fallback_id = getattr(tokenizer, "pad_token_id", None)
    if fallback_id is None:
        fallback_id = getattr(tokenizer, "eos_token_id", 0) or 0
    fallback_id = int(fallback_id)

    def _slice_view(h_tok, t_tok):
        input_ids_list = []
        attention_mask_list = []
        token_type_ids_list = []
        limit = h_tok + t_tok
        raw_ids = enc["input_ids"]
        raw_masks = enc["attention_mask"]
        raw_ttypes = enc["token_type_ids"] if has_token_types else None

        for i in range(len(prep_texts)):
            ids = raw_ids[i]
            mask = raw_masks[i]
            if not ids:
                ids = [fallback_id]
                mask = [1]
            elif len(ids) > limit:
                ids = ids[:h_tok] + ids[-t_tok:]
                mask = mask[:h_tok] + mask[-t_tok:]
            input_ids_list.append(ids)
            attention_mask_list.append(mask)
            if has_token_types:
                ttypes = raw_ttypes[i]
                if not ttypes:
                    ttypes = [0]
                elif len(ttypes) > limit:
                    ttypes = ttypes[:h_tok] + ttypes[-t_tok:]
                token_type_ids_list.append(ttypes)

        out = {"input_ids": input_ids_list, "attention_mask": attention_mask_list}
        if has_token_types:
            out["token_type_ids"] = token_type_ids_list
        return out

    primary = _slice_view(head_tokens, tail_tokens)
    if extra_head_tail is not None:
        return primary, _slice_view(int(extra_head_tail[0]), int(extra_head_tail[1]))
    return primary


def find_topic_token_ids(token_id_sequences, groups, min_prompts: int = 3, special_ids=None) -> set:
    """
    Identifies token IDs that appear in fewer than min_prompts distinct prompt groups.
    Special tokens, punctuation, whitespace, and structural markers are protected from masking.
    """
    from collections import defaultdict
    if groups is None or len(token_id_sequences) != len(groups):
        return set()

    token_groups = defaultdict(set)
    for seq, g in zip(token_id_sequences, groups):
        for tok in set(seq):
            token_groups[tok].add(g)

    special_set = set(special_ids) if special_ids else set()
    topic_tokens = {
        tok for tok, g_set in token_groups.items()
        if len(g_set) < min_prompts and tok not in special_set
    }
    return topic_tokens


def compute_same_prompt_sibling_repulsion(
    representations: torch.Tensor,
    batch_groups: torch.Tensor | None,
    batch_labels: torch.Tensor,
    margin: float = 0.15,
) -> torch.Tensor:
    """
    Penalizes representations of responses to the SAME prompt that collapse together
    despite originating from DIFFERENT model generators (author style separation).
    Runs 100% vectorized in PyTorch without CPU-GPU synchronization stalls.
    """
    B = representations.size(0)
    if B <= 1 or batch_groups is None:
        return torch.zeros((), device=representations.device, dtype=representations.dtype)

    same_prompt = (batch_groups.unsqueeze(1) == batch_groups.unsqueeze(0))
    diff_model = (batch_labels.unsqueeze(1) != batch_labels.unsqueeze(0))
    pair_mask = torch.triu(same_prompt & diff_model, diagonal=1)

    n_pairs = pair_mask.sum()
    f_norm = F.normalize(representations.float(), p=2, dim=-1)
    cos_sim = torch.mm(f_norm, f_norm.t())
    viol = torch.clamp(cos_sim - margin, min=0.0)
    repulsion = (viol.pow(2) * pair_mask.float()).sum() / n_pairs.clamp(min=1.0)
    return repulsion


def make_prompt_sibling_paired_indices(
    group_ids: np.ndarray,
    labels: np.ndarray,
    lengths: np.ndarray,
    batch_size: int,
    rng: np.random.RandomState,
    mega_factor: int = 8,
) -> np.ndarray:
    """
    Constructs epoch batch indices that co-locate responses to the SAME prompt in the same micro-batch,
    enabling in-batch sibling contrastive separation while sorting by length to minimize padding.
    """
    n_samples = len(group_ids)
    if n_samples <= batch_size:
        return np.arange(n_samples, dtype=np.int64)

    from collections import defaultdict
    group_to_idx = defaultdict(list)
    for idx, gid in enumerate(group_ids):
        group_to_idx[gid].append(idx)

    unique_groups = list(group_to_idx.keys())
    rng.shuffle(unique_groups)

    mega_size = batch_size * mega_factor
    mega_chunks = []
    current_chunk = []
    for gid in unique_groups:
        current_chunk.extend(group_to_idx[gid])
        if len(current_chunk) >= mega_size:
            mega_chunks.append(np.array(current_chunk, dtype=np.int64))
            current_chunk = []
    if current_chunk:
        mega_chunks.append(np.array(current_chunk, dtype=np.int64))

    sorted_epoch_indices = []
    for chunk in mega_chunks:
        chunk_lens = lengths[chunk]
        sorted_epoch_indices.extend(chunk[np.argsort(chunk_lens)])

    return np.array(sorted_epoch_indices, dtype=np.int64)


class TopicBlindAndSiFTPrep:
    """
    1. Stochastically masks rare topic tokens (PromptDF < 3) during training
       so DeBERTa/BGE cannot memorize prompt subjects.
    2. Applies Scale-Invariant Fine-Tuning (SiFT) noise to word embeddings
       to flatten the loss basin and improve out-of-fold generalization.
    Uses O(1) Boolean Lookup Table to eliminate CPU search overhead during prefetching.
    """
    def __init__(self, topic_token_ids, mask_token_id: int, mask_prob: float = 0.30, sift_eps: float = 0.015):
        self.raw_topic_ids = [int(x) for x in topic_token_ids] if topic_token_ids else []
        self.mask_token_id = mask_token_id
        self.mask_prob = mask_prob if mask_token_id is not None else 0.0
        self.sift_eps = sift_eps
        if self.raw_topic_ids:
            max_id = max(self.raw_topic_ids) + 1
            lut = torch.zeros(max_id, dtype=torch.bool)
            lut[self.raw_topic_ids] = True
            self.is_topic_lut = lut
        else:
            self.is_topic_lut = None

    def mask_topic_tokens(self, input_ids: torch.Tensor) -> torch.Tensor:
        if self.is_topic_lut is None or self.mask_prob <= 0.0:
            return input_ids
        if self.is_topic_lut.device != input_ids.device:
            self.is_topic_lut = self.is_topic_lut.to(input_ids.device)
        max_lut = self.is_topic_lut.shape[0]
        valid_range = (input_ids >= 0) & (input_ids < max_lut)
        clamped = input_ids.clamp(0, max_lut - 1)
        is_topic = valid_range & self.is_topic_lut[clamped]
        rand_mask = (torch.rand(input_ids.shape, device=input_ids.device) < self.mask_prob) & is_topic
        return input_ids.clone().masked_fill_(rand_mask, self.mask_token_id)

    def apply_sift_embedding_noise(self, inputs_embeds: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        if self.sift_eps <= 0.0:
            return inputs_embeds
        mask = attention_mask.unsqueeze(-1).type_as(inputs_embeds)
        norm = torch.norm(inputs_embeds * mask, p=2, dim=(1, 2), keepdim=True).clamp(min=1e-6).detach()
        noise = torch.randn_like(inputs_embeds) * mask
        noise = noise / torch.norm(noise, p=2, dim=(1, 2), keepdim=True).clamp(min=1e-6)
        return inputs_embeds + self.sift_eps * norm * noise


class FastBatchPrefetcher:
    """
    Background worker thread that pre-pads and pins micro-batches in host RAM ahead of the GPU.
    Eliminates CPU-GPU synchronization bubbles, keeping Tensor Cores at 100% duty cycle.
    Supports asynchronous CPU-side PromptDF stochastic topic-masking.
    """
    def __init__(self, tokenizer, enc_data, y_labels, indices, batch_size, dev, label_dev=None, cuda_ok=True, maxsize=6, topic_prep=None, group_ids=None):
        self.tokenizer = tokenizer
        self.enc_data = enc_data
        self.y_labels = np.asarray(y_labels, dtype=np.int64)
        self.indices = indices
        self.batch_size = batch_size
        self.dev = dev
        self.label_dev = label_dev if label_dev is not None else dev
        self.cuda_ok = cuda_ok
        self.topic_prep = topic_prep
        if group_ids is not None:
            self.group_ids = pd.factorize(np.asarray(group_ids))[0].astype(np.int64)
        else:
            self.group_ids = None
        self.current_groups = None
        self.queue = queue.Queue(maxsize=maxsize)
        self.stop_event = threading.Event()
        self.exc = None
        self._last_pinned = None
        self.worker = threading.Thread(target=self._produce, daemon=True)
        self.worker.start()

    def _produce(self):
        try:
            n = len(self.indices)
            has_types = "token_type_ids" in self.enc_data
            for start in range(0, n, self.batch_size):
                if self.stop_event.is_set():
                    break
                idx = self.indices[start:start + self.batch_size]
                batch_slice = {
                    "input_ids": [self.enc_data["input_ids"][j] for j in idx],
                    "attention_mask": [self.enc_data["attention_mask"][j] for j in idx],
                }
                if has_types:
                    batch_slice["token_type_ids"] = [self.enc_data["token_type_ids"][j] for j in idx]

                padded = self.tokenizer.pad(batch_slice, padding=True, return_tensors="pt")
                if self.topic_prep is not None:
                    padded["input_ids"] = self.topic_prep.mask_topic_tokens(padded["input_ids"])
                kw = {}
                for k, v in padded.items():
                    kw[k] = v.pin_memory() if self.cuda_ok else v
                labels = torch.from_numpy(self.y_labels[idx])
                if self.cuda_ok:
                    labels = labels.pin_memory()

                groups = None
                if self.group_ids is not None:
                    groups = torch.from_numpy(self.group_ids[idx])
                    if self.cuda_ok:
                        groups = groups.pin_memory()

                while not self.stop_event.is_set():
                    try:
                        self.queue.put((kw, labels, groups), timeout=0.1)
                        break
                    except queue.Full:
                        continue
        except Exception as e:
            self.exc = e
        finally:
            self.queue.put(None)

    def __iter__(self):
        while True:
            item = self.queue.get()
            if item is None:
                if self.exc is not None:
                    raise self.exc
                break
            if len(item) == 3:
                kw, labels, groups = item
            else:
                kw, labels = item
                groups = None
            self._last_pinned = (kw, labels, groups)
            if self.cuda_ok:
                gpu_kw = {k: v.to(self.dev, non_blocking=True) for k, v in kw.items()}
                gpu_labels = labels.to(self.label_dev, non_blocking=True)
                self.current_groups = groups.to(self.label_dev, non_blocking=True) if groups is not None else None
                gpu_kw["_batch_groups"] = self.current_groups
                yield gpu_kw, gpu_labels
            else:
                cpu_kw = {k: v.to(self.dev) for k, v in kw.items()}
                cpu_labels = labels.to(self.label_dev)
                self.current_groups = groups.to(self.label_dev) if groups is not None else None
                cpu_kw["_batch_groups"] = self.current_groups
                yield cpu_kw, cpu_labels

    def close(self):
        self.stop_event.set()
        if self.cuda_ok and str(self.dev).startswith("cuda"):
            try:
                dev_idx = torch.device(self.dev).index if torch.device(self.dev).index is not None else 0
                with torch.cuda.device(dev_idx):
                    torch.cuda.synchronize(dev_idx)
            except Exception:
                pass
        while not self.queue.empty():
            try:
                self.queue.get_nowait()
            except queue.Empty:
                break
        if self.worker.is_alive():
            self.worker.join(timeout=0.5)
        self._last_pinned = None


GLOBAL_RAM_MODEL_CACHE: dict = {}
RAM_CACHE_LOCK = threading.Lock()


def precompute_token_caches(raw_texts_tr, raw_texts_te, deberta_dir=None, bge_dir=None, qwen_dir=None, raw_texts_ext=None, max_length=448):
    """
    Pre-tokenizes entire dataset (train + test + external coreset) once upfront across all models.
    Caches token IDs and attention masks in host RAM (~600 MB total with external coreset),
    eliminating redundant tokenizations across 5 folds and 3 models (over 100k tokenizations saved).
    Executes in parallel across vCPUs via ThreadPoolExecutor.
    """
    from transformers import AutoTokenizer

    caches = {}

    def _tokenize_model(model_tag, model_dir, preserve_newlines, left_pad=False, add_special=True):
        if model_dir is None:
            return None
        t0 = time.perf_counter()
        staged_dir = prepare_model_dir(model_dir, model_tag=model_tag)
        try:
            tokenizer = AutoTokenizer.from_pretrained(
                staged_dir, local_files_only=True, trust_remote_code=True, fix_mistral_regex=True
            )
        except TypeError:
            tokenizer = AutoTokenizer.from_pretrained(
                staged_dir, local_files_only=True, trust_remote_code=True
            )
        if left_pad:
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            tokenizer.padding_side = "left"

        if add_special:
            tokenizer.add_tokens(["[NL]", "[PAR]"], special_tokens=True)

        enc_tr = tokenize_with_head_tail(tokenizer, raw_texts_tr, max_length=max_length, preserve_newlines=preserve_newlines)
        enc_te, enc_te_tta = tokenize_with_head_tail(
            tokenizer, raw_texts_te, max_length=max_length,
            preserve_newlines=preserve_newlines, extra_head_tail=(128, 320),
        )
        enc_ext = tokenize_with_head_tail(tokenizer, raw_texts_ext, max_length=max_length, preserve_newlines=preserve_newlines) if raw_texts_ext else None
        dur = time.perf_counter() - t0
        n_ext_str = f" + {len(raw_texts_ext)} ext" if raw_texts_ext else ""
        print(f"  [{model_tag.upper()} Cache] Pre-tokenized {len(raw_texts_tr)} train + {len(raw_texts_te)} test (with TTA){n_ext_str} texts in {dur:.2f}s (RAM resident).", flush=True)
        return {
            "tr": enc_tr,
            "te": enc_te,
            "te_tta": enc_te_tta,
            "ext": enc_ext,
            "tokenizer": tokenizer,
            "staged_dir": staged_dir,
        }

    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        f_deb = executor.submit(_tokenize_model, "deberta", deberta_dir, True, False, True) if deberta_dir else None
        f_bge = executor.submit(_tokenize_model, "bge", bge_dir, True, False, True) if bge_dir else None
        f_qwen = executor.submit(_tokenize_model, "qwen", qwen_dir, False, True, False) if qwen_dir else None

        if f_deb:
            res = f_deb.result()
            if res:
                caches["deberta"] = res
        if f_bge:
            res = f_bge.result()
            if res:
                caches["bge"] = res
        if f_qwen:
            res = f_qwen.result()
            if res:
                caches["qwen"] = res

    return caches

# --- 3. 145-DIMENSIONAL RICH STYLOMETRIC EXTRACTOR ---

class RichStyleFeatures(BaseEstimator, TransformerMixin):
    """Linguistic and typographical features (145 features) capturing authorial habits."""
    PUNCTUATION = [
        "\u2014", "\u2013", "-", ",", ".", ";", ":", "!", "?", "\u2026", "...",
        '"', "'", "\u2018", "\u2019", "\u201c", "\u201d", "(", ")", "[", "]",
        "*", "#", "\n", "\t", "/", "%", "`",
    ]
    FUNCTION_WORDS = (
        "the a an and or but if then so because although however therefore thus "
        "moreover furthermore indeed perhaps might could would should may must can will "
        "we you i it they this that these those not no very more most such"
    ).split()
    CONTRACTION_RE = re.compile(r"\b\w+['\u2019\u2018]\w+\b")
    WORD_RE = re.compile(r"\b\w+(?:['\u2019\u2018]\w+)?\b")
    SENT_RE = re.compile(r"[.!?]+\s*")
    H_RE = re.compile(r"^\s*#{1,6}\s")
    LIST_RE = re.compile(r"^\s*[-*\u2022]\s|^\s*\d+\.\s")
    BOLD_RE = re.compile(r"^\s*\*\*")
    OPENING_RE = re.compile(r"^(certainly|sure|here is|as an|to answer|i would)", re.I)
    CLOSING_RE = re.compile(r"(overall|in conclusion|in summary|ultimately|hope this helps|to summarize)", re.I)
    ASCII_UPPER_RE = re.compile(r"[A-Z]")
    ASCII_DIGIT_RE = re.compile(r"[0-9]")

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rows = []
        word_re = self.WORD_RE
        sent_re = self.SENT_RE
        contr_re = self.CONTRACTION_RE
        h_re = self.H_RE
        list_re = self.LIST_RE
        bold_re = self.BOLD_RE
        open_re = self.OPENING_RE
        close_re = self.CLOSING_RE
        punct_syms = self.PUNCTUATION
        func_words = self.FUNCTION_WORDS
        upper_re = self.ASCII_UPPER_RE
        digit_re = self.ASCII_DIGIT_RE
        sqrt = math.sqrt
        log1p = math.log1p

        def _mean_std(vals):
            n = len(vals)
            if n == 0:
                return 0.0, 0.0
            mu = sum(vals) / n
            if n == 1:
                return mu, 0.0
            var = sum((v - mu) * (v - mu) for v in vals) / n
            return mu, sqrt(var) if var > 0.0 else 0.0

        def _median(vals):
            n = len(vals)
            if n == 0:
                return 0.0
            sv = sorted(vals)
            mid = n >> 1
            if n & 1:
                return float(sv[mid])
            return 0.5 * (sv[mid - 1] + sv[mid])

        for text in X:
            t = clean_text(text)
            nc = max(len(t), 1)
            words = word_re.findall(t)
            nw = max(len(words), 1)
            lower = [w.lower() for w in words]
            w_lengths = [len(w) for w in words] if words else [0]
            sents = [s for s in sent_re.split(t) if s.strip()]
            s_lengths = [len(s.split()) for s in sents] if sents else [0]

            w_mean, w_std = _mean_std(w_lengths)
            w_med = _median(w_lengths)
            s_mean, s_std = _mean_std(s_lengths)

            short_cnt = 0
            long_cnt = 0
            multi_cnt = 0
            if words:
                for wl in w_lengths:
                    if wl <= 3:
                        short_cnt += 1
                    elif wl >= 8:
                        long_cnt += 1
                        if wl >= 12:
                            multi_cnt += 1

            if t.isascii():
                upper_cnt = len(upper_re.findall(t))
                digit_cnt = len(digit_re.findall(t))
                non_ascii_cnt = 0
            else:
                upper_cnt = sum(c.isupper() for c in t)
                digit_cnt = sum(c.isdigit() for c in t)
                non_ascii_cnt = sum(ord(c) > 127 for c in t)

            feat = [
                log1p(nc), log1p(nw), log1p(len(sents)),
                w_mean, w_std, w_med,
                s_mean, s_std,
                len(set(lower)) / nw,
                short_cnt / nw,
                long_cnt / nw,
                upper_cnt / nc,
                digit_cnt / nc,
                non_ascii_cnt / nc,
                len(contr_re.findall(t)) / nw,
            ]
            p_counts = {}
            for p in punct_syms:
                cnt = t.count(p)
                p_counts[p] = cnt
                feat.append(log1p(cnt))
                feat.append(cnt / nw)
            w_counts = Counter(lower)
            for fw in func_words:
                feat.append(w_counts.get(fw, 0) / nw)

            # Positional Tri-Partition Stylometrics (Head 25%, Body 50%, Tail 25%)
            n_quarter = max(nw // 4, 1)
            if words:
                head_words = words[:n_quarter]
                tail_words = words[max(nw - n_quarter, 0):]
                head_w_lens = w_lengths[:n_quarter] or [0]
                body_w_lens = w_lengths[n_quarter:max(nw - n_quarter, n_quarter + 1)] or [0]
                tail_w_lens = w_lengths[max(nw - n_quarter, 0):] or [0]
            else:
                head_words = []
                tail_words = []
                head_w_lens = [0]
                body_w_lens = [0]
                tail_w_lens = [0]

            head_text = " ".join(head_words)
            tail_text = " ".join(tail_words)

            hw_mean, hw_std = _mean_std(head_w_lens)
            tw_mean, tw_std = _mean_std(tail_w_lens)
            bw_mean, bw_std = _mean_std(body_w_lens)

            h_up = len(upper_re.findall(head_text)) if head_text.isascii() else sum(c.isupper() for c in head_text)
            t_up = len(upper_re.findall(tail_text)) if tail_text.isascii() else sum(c.isupper() for c in tail_text)

            feat.extend([
                float(bool(h_re.search(t))),
                float(bool(list_re.search(t))),
                float(bool(bold_re.search(t))),
                float(bool(open_re.search(head_text))),
                float(bool(close_re.search(tail_text))),
                float(t.endswith("?")),
                float(t.endswith("!")),
                hw_mean, hw_std,
                h_up / max(len(head_text), 1),
                tw_mean, tw_std,
                t_up / max(len(tail_text), 1),
                tail_text.count(".") / max(len(tail_words), 1),
                tail_text.count("!") / max(len(tail_words), 1),
                tail_text.count("?") / max(len(tail_words), 1),
                bw_mean, bw_std,
            ])

            # Paragraph and Line Cadence
            paragraphs = [p for p in t.split("\n\n") if p.strip()]
            n_paras = max(len(paragraphs), 1)
            para_lengths = [len(p.split()) for p in paragraphs] if paragraphs else [0]
            lines = [l for l in t.split("\n") if l.strip()]
            line_lengths = [len(l.split()) for l in lines] if lines else [0]
            p_mean, p_std = _mean_std(para_lengths)
            l_mean, l_std = _mean_std(line_lengths)

            feat.extend([
                log1p(n_paras), p_mean, p_std,
                log1p(max(len(lines), 1)), l_mean, l_std,
                t.count("\n\n") / max(p_counts["\n"], 1),
                len(set(lower)) / max(sqrt(nw), 1e-6),    # Root Type-Token Ratio
                multi_cnt / nw,                            # Multisyllabic density
                t.count("```") / nw,                       # Code block density
                p_counts["`"] / nw,                        # Inline code density
                t.count("$") / nw,                         # LaTeX math density
                t.count("\\") / nw,                        # TeX / escape density
                p_counts[":"] / max(p_counts[";"] + 1, 1),  # Colon-to-semicolon ratio
                (p_counts['"'] + p_counts["\u201c"] + p_counts["\u201d"]) / nw,  # Quotation density
            ])
            rows.append(feat)
        return np.asarray(rows, dtype=np.float64)

# --- 4. CPU MULTIVIEW DUAL-LINEAR PIPELINE (~850k TF-IDF + 145 Stylometrics) ---

def align_margins(margins: np.ndarray, model_classes: list, canonical_labels: list, fill_value: float = -999.0) -> np.ndarray:
    """Reorder margin or probability columns to match canonical LABELS."""
    c_to_idx = {c: i for i, c in enumerate(model_classes)}
    aligned = np.full((margins.shape[0], len(canonical_labels)), fill_value, dtype=np.float64)
    for col, lab in enumerate(canonical_labels):
        if lab in c_to_idx:
            aligned[:, col] = margins[:, c_to_idx[lab]]
    return aligned


def compute_prompt_df_mask(X, groups, min_prompts=3):
    """
    Computes boolean feature mask based on Prompt Document Frequency.
    Prunes rare features appearing in fewer than min_prompts distinct prompts.
    Vectorized sparse matrix multiplication: O(prompts * nonzeros), executes in <0.5s.
    """
    if min_prompts <= 1 or groups is None:
        return np.ones(X.shape[1], dtype=np.float32)
    B = (X > 0).astype(np.float32).tocsr()
    codes = pd.factorize(groups)[0]
    G = sp.csr_matrix((np.ones(len(codes), np.float32), (codes, np.arange(len(codes)))))
    pdf = np.asarray(((G @ B) > 0).sum(axis=0)).ravel()
    mask = (pdf >= min_prompts).astype(np.float32)
    if not np.any(mask > 0):
        return np.ones(X.shape[1], dtype=np.float32)
    return mask


class NonLinearStylometricKernelExpander(BaseEstimator, TransformerMixin):
    """
    Expands dense continuous stylometric features into a high-capacity non-linear Hilbert space using:
      1. Internal unit-variance scaling clipped to [-6.0, 6.0] to prevent quadratic blowout.
      2. Degree-2 polynomial interaction cross-products among top-36 variance features (666 quad terms).
      3. Nyström RBF kernel approximation mapping to 1,024 non-linear components (gamma=1.0/n_feats).
    Total expansion: 145 (raw scaled) + 666 (quad cross-products) + 1,024 (RBF components) = 1,835 features.
    Utilizes ~3.2 GB of System RAM to grant CPU linear models (LogisticRegression / Ridge)
    expressive curved decision boundaries between LLM stylistic signatures in <3.0s per fold.
    """
    def __init__(self, n_top_poly=36, n_nystroem=1024, seed=42):
        self.n_top_poly = n_top_poly
        self.n_nystroem = n_nystroem
        self.seed = seed

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=np.float64)
        self.in_scaler_ = StandardScaler()
        X_s = np.clip(self.in_scaler_.fit_transform(X), -6.0, 6.0)

        n_samples, n_feats = X_s.shape
        iqr = np.percentile(X_s, 75, axis=0) - np.percentile(X_s, 25, axis=0)
        k_poly = min(self.n_top_poly, n_feats)
        self.top_poly_idx_ = np.argsort(-iqr)[:k_poly]

        k_nys = min(self.n_nystroem, max(1, n_samples - 1))
        self.nystroem_ = Nystroem(
            kernel="rbf", n_components=k_nys, random_state=self.seed, gamma=1.0 / max(n_feats, 1)
        )
        self.nystroem_.fit(X_s)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        X_s = np.clip(self.in_scaler_.transform(X), -6.0, 6.0)
        parts = [X_s]

        if hasattr(self, "top_poly_idx_") and len(self.top_poly_idx_) > 1:
            X_sub = X_s[:, self.top_poly_idx_]
            k = X_sub.shape[1]
            quad_cols = [X_sub[:, i:i+1] * X_sub[:, i:] for i in range(k)]
            parts.append(np.hstack(quad_cols) * 0.35)

        if hasattr(self, "nystroem_"):
            parts.append(self.nystroem_.transform(X_s) * 1.5)

        return np.hstack(parts)


class DenseStylometricClassifier(BaseEstimator):
    """
    Standardized dense stylometrics classifier with optional non-linear kernel expansion.
    Scales continuous stylometric features and fits an L2-regularized
    LogisticRegression classifier with balanced class weights to produce calibrated probabilities.
    Supports sample_weight with automatic class_weight=None override to prevent double-balancing.
    """
    def __init__(self, c=0.2, seed=42, class_weight="balanced", canonical_labels=None, use_kernel_expander=True):
        self.c = c
        self.seed = seed
        self.class_weight = class_weight
        self.canonical_labels = canonical_labels
        self.use_kernel_expander = use_kernel_expander

    def fit(self, V, y, sample_weight=None):
        self.scaler_ = StandardScaler()
        cw = None if sample_weight is not None else self.class_weight
        self.clf_ = LogisticRegression(
            C=self.c, max_iter=300, random_state=self.seed, class_weight=cw, solver="lbfgs"
        )
        if self.use_kernel_expander:
            self.expander_ = NonLinearStylometricKernelExpander(seed=self.seed)
            V_feat = self.expander_.fit_transform(V)
        else:
            self.expander_ = None
            V_feat = V
        V_scaled = self.scaler_.fit_transform(V_feat)
        if sample_weight is not None:
            self.clf_.fit(V_scaled, y, sample_weight=sample_weight)
        else:
            self.clf_.fit(V_scaled, y)
        raw_classes = list(self.clf_.classes_)
        if self.canonical_labels is not None:
            self.classes_ = np.array(self.canonical_labels)
            self._raw_classes = raw_classes
        else:
            self.classes_ = self.clf_.classes_
            self._raw_classes = raw_classes
        return self

    def predict_proba(self, V):
        if getattr(self, "expander_", None) is not None:
            V_feat = self.expander_.transform(V)
        else:
            V_feat = V
        V_scaled = self.scaler_.transform(V_feat)
        p = self.clf_.predict_proba(V_scaled)
        if self.canonical_labels is not None:
            p_aligned = align_margins(p, self._raw_classes, self.canonical_labels, fill_value=0.0)
            return p_aligned / np.maximum(p_aligned.sum(axis=1, keepdims=True), 1e-12)
        return p


class DualLinearClassifier(BaseEstimator):
    """
    Ensemble of LinearSVC (hinge loss margin maximizer) and RidgeClassifier (L2 regularized least squares).
    Fast, memory-safe, and bounded to max_iter=800 with dual="auto" to guarantee rapid convergence.
    Uses class_weight="balanced" to directly align optimization with Macro Balanced Accuracy.
    Regularization tuned (c_svc=1.5, alpha_ridge=1.0, svc_weight=0.70) to control variance under p >> n regime.
    Enforces canonical label ordering on decision_function output to guarantee ensemble alignment.
    Supports sample_weight with automatic class_weight=None override to prevent double-balancing inversion.
    """
    def __init__(self, c_svc=1.5, alpha_ridge=1.0, svc_weight=0.70, seed=42, class_weight="balanced", canonical_labels=None):
        self.c_svc = c_svc
        self.alpha_ridge = alpha_ridge
        self.svc_weight = svc_weight
        self.seed = seed
        self.class_weight = class_weight
        self.canonical_labels = canonical_labels

    def fit(self, X, y, sample_weight=None):
        cw = None if sample_weight is not None else self.class_weight
        self.svc_1_ = LinearSVC(penalty="l2", loss="squared_hinge", C=self.c_svc * 0.75, class_weight=cw, dual="auto", tol=1e-3, max_iter=800, random_state=self.seed)
        self.svc_2_ = LinearSVC(penalty="l2", loss="hinge", C=self.c_svc * 1.25, class_weight=cw, dual="auto", tol=1e-3, max_iter=800, random_state=self.seed + 7)
        self.ridge_1_ = RidgeClassifier(alpha=self.alpha_ridge * 0.6, solver="lsqr", tol=5e-3, class_weight=cw, random_state=self.seed)
        self.ridge_2_ = RidgeClassifier(alpha=self.alpha_ridge * 1.5, solver="lsqr", tol=5e-3, class_weight=cw, random_state=self.seed + 7)

        for clf in (self.svc_1_, self.svc_2_, self.ridge_1_, self.ridge_2_):
            if sample_weight is not None:
                clf.fit(X, y, sample_weight=sample_weight)
            else:
                clf.fit(X, y)

        self.svc_ = self.svc_1_
        self.ridge_ = self.ridge_1_
        raw_classes = list(self.svc_1_.classes_)
        if self.canonical_labels is not None:
            self.classes_ = np.array(self.canonical_labels)
            self._raw_classes = raw_classes
        else:
            self.classes_ = self.svc_1_.classes_
            self._raw_classes = raw_classes
        return self

    def decision_function(self, X):
        def _get_2d(clf):
            d = clf.decision_function(X)
            return np.column_stack([-d, d]) if d.ndim == 1 else d

        d_svc = 0.55 * _get_2d(self.svc_1_) + 0.45 * _get_2d(self.svc_2_)
        d_ridge = 0.50 * _get_2d(self.ridge_1_) + 0.50 * _get_2d(self.ridge_2_)
        m = self.svc_weight * d_svc + (1.0 - self.svc_weight) * d_ridge
        if self.canonical_labels is not None:
            return align_margins(m, self._raw_classes, self.canonical_labels)
        return m


class IntraFamilyCliqueSpecialist(BaseEstimator):
    """
    Trains dedicated specialist linear classifiers within each multi-model family clique
    (Anthropic-5, Google-4, OpenAI-5, Zhipu-2) on the combined competition + external corpus.
    Provides sharp conditional intra-family sibling log-probabilities to break sibling ties.
    Zero-mean row centering ensures family overall mass is unchanged, strictly redistributing
    preference between ambiguous siblings.
    """
    def __init__(self, alpha_ridge: float = 1.0, beta_refine: float = 0.25, seed: int = 42):
        self.alpha_ridge = alpha_ridge
        self.beta_refine = beta_refine
        self.seed = seed
        self.specialists_ = {}

    def fit(self, X, y, sample_weight=None):
        y_arr = np.asarray(y)
        for fam, indices in FAMILY_TO_INDICES.items():
            if len(indices) < 2:
                continue
            fam_labels = [LABELS[i] for i in indices]
            fam_mask = np.isin(y_arr, fam_labels) | np.isin(y_arr, indices)
            if np.sum(fam_mask) < 20:
                continue
            X_fam = X[fam_mask]
            y_fam = y_arr[fam_mask]
            if len(np.unique(y_fam)) < 2:
                continue
            w_fam = sample_weight[fam_mask] if sample_weight is not None else None
            if w_fam is not None and w_fam.sum() > 0:
                w_fam = w_fam * (len(w_fam) / w_fam.sum())

            cw_fam = None if w_fam is not None else "balanced"
            clf = RidgeClassifier(alpha=self.alpha_ridge, solver="lsqr", tol=5e-3, class_weight=cw_fam, random_state=self.seed)
            clf.fit(X_fam, y_fam, sample_weight=w_fam)
            self.specialists_[fam] = (clf, list(clf.classes_))
        return self

    def _compute_sibling_deltas(self, X_eval, n_rows: int, n_cols: int) -> np.ndarray:
        """Compute unscaled intra-family sibling log-probability deltas once per split."""
        deltas = np.zeros((n_rows, n_cols), dtype=np.float64)
        for fam, (clf, classes) in self.specialists_.items():
            d = clf.decision_function(X_eval)
            if d.ndim == 1:
                d = np.column_stack([-d, d])
            max_d = np.max(d, axis=1, keepdims=True)
            exp_d = np.exp(d - max_d)
            p_cond = exp_d / np.maximum(np.sum(exp_d, axis=1, keepdims=True), 1e-12)
            log_p = np.log(p_cond + 1e-12)
            log_p_centered = log_p - log_p.mean(axis=1, keepdims=True)

            for col_idx, c_name in enumerate(classes):
                if c_name in LABEL_TO_IDX:
                    glob_idx = LABEL_TO_IDX[c_name]
                elif isinstance(c_name, (int, np.integer)) and 0 <= c_name < len(LABELS):
                    glob_idx = int(c_name)
                else:
                    continue
                deltas[:, glob_idx] += log_p_centered[:, col_idx]
        return deltas

    def refine_margins(self, global_margins: np.ndarray, X_eval, beta: float = None, precomputed_deltas: np.ndarray = None) -> np.ndarray:
        """
        Refines global margins with conditional intra-family specialist scores:
        z_m <- z_m + beta * sibling_score_m
        """
        if beta is None:
            beta = self.beta_refine
        if not self.specialists_ or len(global_margins) == 0 or beta <= 0.0:
            return global_margins
        deltas = precomputed_deltas if precomputed_deltas is not None else self._compute_sibling_deltas(
            X_eval, global_margins.shape[0], global_margins.shape[1]
        )
        return global_margins + beta * deltas

    def optimize_beta(self, base_val_margins: np.ndarray, X_va, y_va_int: np.ndarray, candidates=(0.0, 0.10, 0.20, 0.30, 0.40), precomputed_deltas: np.ndarray = None) -> float:
        """
        Tunes beta on validation fold to ensure monotonic accuracy improvement.
        Candidate 0.0 guarantees refinement can never degrade validation score.
        Computes specialist deltas once instead of re-scoring across all candidate betas.
        """
        if not self.specialists_ or len(base_val_margins) == 0:
            return self.beta_refine
        deltas = precomputed_deltas if precomputed_deltas is not None else self._compute_sibling_deltas(
            X_va, base_val_margins.shape[0], base_val_margins.shape[1]
        )
        best_beta = self.beta_refine
        best_acc = -1.0
        for b in candidates:
            m_cand = base_val_margins + b * deltas if b > 0.0 else base_val_margins
            acc = balanced_accuracy_score(y_va_int, np.argmax(m_cand, axis=1))
            if acc > best_acc:
                best_acc = acc
                best_beta = b
        self.beta_refine = best_beta
        return best_beta


class StyleInvariantSubspaceProjector(BaseEstimator):
    """
    Discovers the Prompt-Topic Subspace vs. Author-Style Subspace using multi-model prompts,
    and attenuates topic-correlated directions so unseen test prompts cannot shift predictions.
    Operates strictly on dense representations (e.g. 145-dim stylometrics, 18-dim OOF probabilities,
    or neural latent embeddings) to prevent sparse matrix densification and memory explosion.
    Class-demeaned within-prompt SVD prevents partial prompts from leaking author style into the topic subspace.
    """
    def __init__(self, n_topic_components: int = 16, attenuation: float = 0.75):
        self.n_topic_components = n_topic_components
        self.attenuation = attenuation
        self.U_topic_ = None
        self.mean_ = None

    def fit(self, X: np.ndarray, prompt_ids: np.ndarray, y_labels: np.ndarray = None):
        """
        X: [N, D] dense feature representations
        prompt_ids: [N] prompt group identifiers
        y_labels: [N] optional class labels to demean author style before computing prompt centroids
        """
        if sp.issparse(X):
            raise TypeError("StyleInvariantSubspaceProjector must be applied strictly to dense matrices to prevent RAM densification.")
        X_arr = np.asarray(X, dtype=np.float32)
        if not np.all(np.isfinite(X_arr)):
            X_arr = np.nan_to_num(X_arr, nan=0.0, posinf=0.0, neginf=0.0)
        if X_arr.shape[0] < 2:
            return self
        self.mean_ = X_arr.mean(axis=0, keepdims=True)
        X_c = X_arr - self.mean_

        # CRITICAL FIX: Subtract per-class style centroid BEFORE computing prompt centroids
        # Otherwise, external prompts answered by only 2-3 models leak author style into the topic SVD!
        if y_labels is not None:
            X_res = X_c.copy()
            for c in np.unique(y_labels):
                c_mask = (y_labels == c)
                if np.any(c_mask):
                    X_res[c_mask] -= X_c[c_mask].mean(axis=0, keepdims=True)
        else:
            X_res = X_c

        codes, uniques = pd.factorize(prompt_ids)
        valid_c = codes >= 0
        if not np.all(valid_c):
            codes = codes[valid_c]
            X_res_v = X_res[valid_c]
        else:
            X_res_v = X_res
        counts = np.bincount(codes)
        multi_codes = np.where(counts >= 2)[0]
        k = min(self.n_topic_components, len(multi_codes) - 1, X.shape[1] - 1)
        if k <= 0:
            return self

        sums = np.zeros((len(counts), X_res_v.shape[1]), dtype=np.float32)
        np.add.at(sums, codes, X_res_v)
        centroids = (sums[multi_codes] / counts[multi_codes, None]).astype(np.float32)
        try:
            _, _, Vt = np.linalg.svd(centroids, full_matrices=False)
            k_actual = min(k, Vt.shape[0])
            self.U_topic_ = Vt[:k_actual].T  # [D, K]
        except np.linalg.LinAlgError:
            self.U_topic_ = None
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X_arr = np.asarray(X, dtype=np.float32)
        if not np.all(np.isfinite(X_arr)):
            X_arr = np.nan_to_num(X_arr, nan=0.0, posinf=0.0, neginf=0.0)
        if self.U_topic_ is None or self.mean_ is None:
            return X_arr
        X_c = X_arr - self.mean_
        topic_proj = (X_c @ self.U_topic_) @ self.U_topic_.T
        return (X_c - self.attenuation * topic_proj) + self.mean_


def build_cpu_feature_extractor(style_weight=0.2, include_style=True):
    """
    High-throughput 5-branch feature union (~850k features):
    char n-grams (2-5), word n-grams (1-3), char_wb (2-5), punct (1-3), and stylometrics.
    Fits once upfront on all texts to eliminate redundant vocabulary building across folds.
    Uses float32 CSR matrices and parallel branch extraction across available CPU cores.
    """
    char_branch = TfidfVectorizer(
        analyzer="char", ngram_range=(2, 5), lowercase=False,
        sublinear_tf=True, min_df=3, max_features=250_000, dtype=np.float32,
    )
    word_branch = TfidfVectorizer(
        ngram_range=(1, 3), lowercase=False, token_pattern=r"\S+",
        sublinear_tf=True, min_df=2, max_features=250_000, dtype=np.float32,
    )
    boundary_branch = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 5),
        sublinear_tf=True, min_df=3, max_features=250_000, dtype=np.float32,
    )
    punct_branch = TfidfVectorizer(
        token_pattern=r"(?u)\b\w+\b|[^\w\s]", ngram_range=(1, 3),
        lowercase=False, sublinear_tf=True, min_df=2, max_features=100_000, dtype=np.float32,
    )
    transformers = [
        ("char", char_branch),
        ("word", word_branch),
        ("boundary", boundary_branch),
        ("punct", punct_branch),
    ]
    weights = None
    if include_style:
        style_branch = Pipeline([
            ("extract", RichStyleFeatures()),
            ("scale", StandardScaler()),
        ])
        transformers.append(("style", style_branch))
        weights = {"style": style_weight}

    return FeatureUnion(
        transformer_list=transformers,
        transformer_weights=weights,
        n_jobs=min(4, os.cpu_count() or 1),
    )

# --- 5. INPUT DISCOVERY ---

def discover_paths():
    """
    Discover paths for competition data, Qwen, DeBERTa, and BGE model weights.
    Returns (data_dir, qwen_dir, deberta_dir, bge_dir, ext_path, working_dir).
    Supports Kaggle environment (/kaggle/input) and local offline dev seamlessly.
    """
    local_data_dir = os.environ.get("NTL_DATA_DIR")
    input_root = Path(local_data_dir) if local_data_dir else (Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("."))
    working_dir = Path(os.environ.get("NTL_WORKING", "/kaggle/working" if Path("/kaggle").exists() else "data/sample/working_verifier"))
    working_dir.mkdir(parents=True, exist_ok=True)

    data_dir = None
    qwen_dir = None
    bge_dir = None
    qwen_candidates = []
    deberta_candidates = []

    for candidate in [
        input_root / "competitions/name-that-llm",
        input_root / "name-that-llm",
        input_root / "sample",
        input_root / "data/sample",
        Path("data/sample"),
        Path("data"),
        input_root,
    ]:
        if (candidate / "train.csv").is_file() and (candidate / "test.csv").is_file():
            data_dir = candidate
            break

    if data_dir is None and input_root.exists():
        for p in input_root.rglob("train.csv"):
            if (p.parent / "test.csv").is_file():
                data_dir = p.parent
                break

    if data_dir is None:
        for p in Path(".").rglob("train.csv"):
            if (p.parent / "test.csv").is_file():
                data_dir = p.parent
                break

    if data_dir is None:
        raise FileNotFoundError(f"Could not locate train.csv + test.csv under {input_root} or repository root")

    configs = []
    parquets = []
    ext_tables = []
    if input_root.exists():
        skip_dirs = {".git", "__pycache__", ".ipynb_checkpoints", ".mypy_cache", ".ruff_cache"}
        seen_realpaths = set()
        for root_dir, dirnames, filenames in os.walk(input_root, followlinks=True):
            real_dir = os.path.realpath(root_dir)
            if real_dir in seen_realpaths:
                dirnames[:] = []
                continue
            seen_realpaths.add(real_dir)
            dirnames[:] = [d for d in dirnames if d not in skip_dirs]
            for fname in filenames:
                fpath = Path(root_dir) / fname
                if fname == "config.json":
                    configs.append(fpath)
                elif fname.endswith(".parquet"):
                    parquets.append(fpath)
                    ext_tables.append(fpath)
                elif fname.endswith((".csv", ".jsonl", ".ndjson")) and Path(root_dir) != data_dir:
                    ext_tables.append(fpath)
        configs.sort(key=lambda p: len(p.parts))
        parquets.sort(key=lambda p: len(p.parts))
        ext_tables.sort(key=lambda p: len(p.parts))

    for p in configs:
        p_str = str(p).lower()
        if any(x in p_str for x in ("keras", "tensorflow", "jax")):
            continue
        try:
            cfg = json.loads(p.read_text(encoding="utf-8", errors="ignore"))
            mtype = cfg.get("model_type", "").lower()
            arch = str(cfg.get("architectures", "")).lower()

            is_qwen = "qwen" in mtype or "qwen" in p_str or "qwen" in arch
            if is_qwen:
                # Priority: Qwen 3 (tier 0) > Qwen 2.5 (tier 1) > other (tier 2)
                # Size: 8b/7b (size 0) > other (size 1)
                tier = 0 if ("qwen-3" in p_str or "qwen3" in p_str) else (1 if ("qwen2.5" in p_str or "qwen-2.5" in p_str) else 2)
                size = 0 if ("8b" in p_str or "7b" in p_str) else 1
                qwen_candidates.append(((tier, size, len(p.parts)), p.parent))

            is_deberta = "deberta" in mtype or "deberta" in p_str or "deberta" in arch
            if is_deberta:
                # Prioritize clean base/large over fine-tuned task variants (-offensive, ontonotes, ner, squad, mnli, etc.)
                parent_name = p.parent.name.lower()
                task_tokens = (
                    "ontonotes", "ner", "squad", "mnli", "conll", "khalidalt",
                    "offensive", "toxic", "hate", "sentiment", "emotion",
                    "nli", "xnli", "absa", "rerank", "sarcasm", "irony", "stance",
                )
                is_task_specific = any(x in p_str for x in task_tokens)
                is_exact_clean_large = parent_name in ("deberta-v3-large", "microsoft-deberta-v3-large", "khalidalt-deberta-v3-large")
                tier = 0 if (is_exact_clean_large and not is_task_specific) else (1 if ("deberta-v3-large" in p_str and not is_task_specific) else (2 if not is_task_specific else 3))
                size_score = 0 if "large" in p_str else 1
                deberta_candidates.append(((tier, size_score, len(p.parts)), p.parent))

            is_bge = "bge" in p_str or "bge" in mtype
            if is_bge and bge_dir is None:
                bge_dir = p.parent
        except Exception:
            continue

    if qwen_candidates:
        qwen_candidates.sort(key=lambda x: x[0])
        qwen_dir = qwen_candidates[0][1]

    deberta_dir = None
    if deberta_candidates:
        deberta_candidates.sort(key=lambda x: x[0])
        deberta_dir = deberta_candidates[0][1]

    def _inspect_external_table(p_cand: Path) -> int:
        """Returns format tier: 2=wide schema (>=2 label cols), 1=long schema ('model'+'text'), 0=incompatible."""
        try:
            suf = p_cand.suffix.lower()
            if suf == ".csv":
                cols = set(pd.read_csv(p_cand, nrows=2).columns)
            elif suf in (".jsonl", ".ndjson"):
                cols = set(pd.read_json(p_cand, lines=True, nrows=2).columns)
            else:
                try:
                    import pyarrow.parquet as pq
                    cols = set(pq.read_schema(p_cand).names)
                except Exception:
                    cols = set(pd.read_parquet(p_cand).columns)
            if sum(1 for lbl in LABELS if lbl in cols) >= 2:
                return 2
            if "model" in cols and "text" in cols:
                return 1
        except Exception:
            pass
        return 0

    ext_path = None
    env_ext = os.environ.get("NTL_EXT_PATH")
    if env_ext and Path(env_ext).is_file() and _inspect_external_table(Path(env_ext)) > 0:
        ext_path = Path(env_ext)

    if ext_path is None:
        # 1. Prioritize exact external_short_wide.* inside input_root first
        exact_matches = [p for p in ext_tables if p.name in ("external_short_wide.parquet", "external_short_wide.csv") and p.is_file()]
        for cand in exact_matches + [
            input_root / "datasets/external_short_wide.parquet",
            input_root / "external_short_wide.parquet",
            Path("/kaggle/working/external_short_wide.parquet"),
        ]:
            if cand.is_file() and _inspect_external_table(cand) > 0:
                ext_path = cand
                break

    if ext_path is None:
        # 2. Score all schema-compatible tables discovered inside input_root BEFORE repo CWD fallbacks
        ext_candidates = []
        for p in ext_tables:
            p_path_str = str(p).lower()
            if any(bad in p_path_str for bad in ("submission", "working", "test_holdout", "sample")):
                continue
            tier = _inspect_external_table(p)
            if tier == 0:
                continue
            short_bonus = 4 if "short" in p_path_str else 0
            name_bonus = 2 if any(k in p_path_str for k in ("external", "wide", "corpus", "response", "pool")) else 0
            ext_bonus = 3 if p.suffix.lower() == ".parquet" else 0
            ext_candidates.append((-(tier * 20 + short_bonus + name_bonus + ext_bonus), len(p.parts), p))
        if ext_candidates:
            ext_candidates.sort(key=lambda x: (x[0], x[1]))
            ext_path = ext_candidates[0][2]

    if ext_path is None and not local_data_dir and not Path("/kaggle/input").exists():
        # 3. Local repo defaults when running without NTL_DATA_DIR override outside Kaggle
        fallback_tables = [
            Path("datasets/external_short_wide.parquet"),
            Path("data/external_short_wide.parquet"),
        ]
        for d_fallback in (Path("datasets"), Path("data/processed")):
            if d_fallback.is_dir():
                fallback_tables.extend(sorted(d_fallback.glob("*.parquet")))
                fallback_tables.extend(sorted(d_fallback.glob("*.csv")))
        for cand in fallback_tables:
            if cand.is_file() and not any(bad in str(cand).lower() for bad in ("submission", "working", "test_holdout", "sample")):
                if _inspect_external_table(cand) > 0:
                    ext_path = cand
                    break

    deb_variant = "large" if deberta_dir and "large" in str(deberta_dir).lower() else "base/unknown"
    qwen_variant = "qwen-3-8b" if qwen_dir and ("qwen-3" in str(qwen_dir).lower() or "qwen3" in str(qwen_dir).lower()) else ("qwen-2.5-7b" if qwen_dir and "2.5" in str(qwen_dir).lower() else "custom/unknown")
    print(f"  [Paths] data={data_dir}")
    print(f"  [Paths] qwen={qwen_dir}  ({qwen_variant})")
    print(f"  [Paths] deberta={deberta_dir}  ({deb_variant})")
    print(f"  [Paths] bge={bge_dir}")
    print(f"  [Paths] external={ext_path}")
    if ext_path is None and input_root.exists():
        sample_tables = [str(p) for p in ext_tables[:8]]
        top_mounts = [p.name for p in sorted(input_root.iterdir())] if input_root.is_dir() else []
        print(f"  [Paths] Note: No external corpus matched under {input_root} (mounts={top_mounts}, tabular_files={sample_tables}).")
        print("  [Paths] Hint: An attached notebook like 'Name That LLM Wide Data Overview' only provides external data if its output exports a .parquet/.csv file; otherwise attach Dataset 'olegzholobov/name-that-llm-response-corpus' ('external_short_wide.parquet').")
    if deberta_dir is not None:
        deberta_dir = prepare_model_dir(deberta_dir, "deberta")
    if bge_dir is not None:
        bge_dir = prepare_model_dir(bge_dir, "bge")
    return data_dir, qwen_dir, deberta_dir, bge_dir, ext_path, working_dir

def prepare_model_dir(model_dir, model_tag="model"):
    """
    Optimizes model loading from Kaggle network FUSE storage (/kaggle/input).
    If the model directory is on network storage (/kaggle/input) and is under 6GB,
    stream-copies it sequentially to local /tmp NVMe storage (12s vs 500s of FUSE random seeks).
    """
    if model_dir is None:
        return None
    model_path = Path(model_dir)
    if not str(model_path).startswith("/kaggle/input"):
        return str(model_dir)

    # Qwen 7B (15GB) uses safetensors and mmap across dual GPUs; avoid copying 15GB to /tmp
    if "qwen" in str(model_path).lower() or model_tag.lower() == "qwen":
        return str(model_dir)

    local_cache = Path(f"/tmp/model_cache_{model_tag}")
    if (local_cache / "config.json").is_file():
        return str(local_cache)

    print(f"  [{model_tag}] Staging weights from network FUSE to local NVMe (/tmp)...", flush=True)
    t0 = time.perf_counter()
    local_cache.mkdir(parents=True, exist_ok=True)

    try:
        for root, dirs, files in os.walk(model_path):
            rel = Path(root).relative_to(model_path)
            target_dir = local_cache / rel
            target_dir.mkdir(parents=True, exist_ok=True)
            for fname in files:
                src_file = Path(root) / fname
                dst_file = target_dir / fname
                if not dst_file.exists() or dst_file.stat().st_size != src_file.stat().st_size:
                    with open(src_file, "rb") as fsrc, open(dst_file, "wb") as fdst:
                        shutil.copyfileobj(fsrc, fdst, length=16 * 1024 * 1024)
        elapsed = time.perf_counter() - t0
        print(f"  [{model_tag}] Staged locally in {elapsed:.1f}s (eliminates FUSE seek latency).", flush=True)
        return str(local_cache)
    except Exception as e:
        print(f"  [{model_tag}] Warning: Local staging error: {e}. Falling back to direct path.", flush=True)
        return str(model_dir)


class MultiSampleDropoutHead(nn.Module):
    """
    5-way Multi-Sample Dropout classification head.
    Computes 5 distinct logit branches and exposes branch variance for R-Drop / consistency regularization.
    """
    def __init__(self, in_features, num_classes, dropouts=(0.1, 0.2, 0.3, 0.4, 0.5)):
        super().__init__()
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in dropouts])
        self.linear = nn.Linear(in_features, num_classes)
        nn.init.xavier_uniform_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)
        self.last_branch_var = None

    def forward(self, x):
        if not self.training:
            self.last_branch_var = None
            return self.linear(x)
        branch_logits = torch.stack([self.linear(drop(x)) for drop in self.dropouts], dim=0)  # [5, B, C]
        # Store inter-branch logit variance for self-consistency regularization (R-Drop style)
        self.last_branch_var = torch.mean(torch.var(branch_logits.float(), dim=0))
        return torch.mean(branch_logits, dim=0)


class SpikeAndGapHierarchicalHead(nn.Module):
    """
    Captures token-level anomalous spikes, hidden-state transition jerk,
    and multi-subcenter class geometry for fine-grained sibling LLM attribution.
    Extracts multi-moments (Mean + Learned Spike Attention + GeM Max Spike + Transition Jerk Stats)
    and projects onto a unit hypersphere with K=3 sub-centers per class.
    Supports Multi-Layer Hidden State Fusion across discourse (-1), syntax (-5), and BPE cadence (-9).
    """
    def __init__(self, hidden_size: int = 4096, num_classes: int = 18, sub_centers: int = 3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_classes = num_classes
        self.sub_centers = sub_centers
        self.layer_mix_weights = nn.Parameter(torch.zeros(3))

        self.spike_gate = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        fused_dim = hidden_size * 3 + 4
        self.proj = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(0.15),
            nn.Linear(fused_dim, 1024),
            nn.GELU(),
            nn.LayerNorm(1024)
        )

        self.weight = nn.Parameter(torch.empty(num_classes * sub_centers, 1024))
        nn.init.xavier_uniform_(self.weight)
        self.scale = nn.Parameter(torch.tensor(14.0))  # Learnable cosine temperature scale

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        """
        hidden_states: [B, T, D] tensor OR tuple/list of hidden states from causal LLM backbone
        attention_mask: [B, T] (1 for valid tokens, 0 for pad)
        """
        target_dev = self.weight.device
        target_dtype = self.weight.dtype

        # Multi-GPU safe layer mixing: move each strided layer to target_dev BEFORE weighted sum
        if isinstance(hidden_states, (tuple, list)):
            w = F.softmax(self.layer_mix_weights, dim=0)
            mixed = None
            if len(hidden_states) == 3:
                target_layers = hidden_states
            else:
                target_layers = [hidden_states[offset if abs(offset) <= len(hidden_states) else -1] for offset in (-1, -5, -9)]
            for i, h_l_raw in enumerate(target_layers):
                h_l = h_l_raw.to(device=target_dev)
                if not torch.is_autocast_enabled() and h_l.dtype != target_dtype:
                    h_l = h_l.to(dtype=target_dtype)
                mixed = (w[i] * h_l) if mixed is None else (mixed + w[i] * h_l)
            hidden_states = mixed
        else:
            if hidden_states.device != target_dev:
                hidden_states = hidden_states.to(target_dev)
            if not torch.is_autocast_enabled() and hidden_states.dtype != target_dtype:
                hidden_states = hidden_states.to(dtype=target_dtype)

        if attention_mask is None:
            attention_mask = torch.ones(hidden_states.shape[:2], device=target_dev, dtype=torch.long)
        elif attention_mask.device != target_dev:
            attention_mask = attention_mask.to(target_dev)

        mask = attention_mask.unsqueeze(-1).to(dtype=hidden_states.dtype)
        lengths = torch.clamp(mask.sum(dim=1), min=1.0)

        # 1. Mean Pooling (Global stylistic baseline)
        h_mean = (hidden_states * mask).sum(dim=1) / lengths

        # 2. Learned Anomaly-Spike Attention Pooling (isolates signature formatting & refusal spikes)
        gate_logits = self.spike_gate(hidden_states)
        gate_logits = gate_logits.masked_fill(mask == 0, -1e4)
        gate_weights = F.softmax(gate_logits, dim=1)
        h_attn_spike = (hidden_states * gate_weights).sum(dim=1)

        # 3. Masked Max Activation Spike Pooling across tokens
        masked_h = hidden_states.masked_fill(mask == 0, -1e4)
        h_max_spike, _ = masked_h.max(dim=1)

        # 4. Token-to-Token Transition Jerk (||h_t - h_{t-1}||_2 syntactic jump)
        B, T, D = hidden_states.shape
        if T > 1:
            diffs = torch.norm(hidden_states[:, 1:, :] - hidden_states[:, :-1, :], p=2, dim=-1)
            valid_pairs = (attention_mask[:, 1:] * attention_mask[:, :-1]).to(dtype=hidden_states.dtype)
            pair_lens = torch.clamp(valid_pairs.sum(dim=1, keepdim=True), min=1.0)

            gap_mean = (diffs * valid_pairs).sum(dim=1, keepdim=True) / pair_lens
            masked_diffs = diffs.masked_fill(valid_pairs == 0, -1e4)
            gap_max, _ = masked_diffs.max(dim=1, keepdim=True)
            gap_max = torch.clamp(gap_max, min=0.0)
            has_pairs = (valid_pairs.sum(dim=1, keepdim=True) > 0)
            gap_std = torch.sqrt(torch.clamp(((diffs - gap_mean) ** 2 * valid_pairs).sum(dim=1, keepdim=True) / pair_lens, min=1e-6))
            gap_std = torch.where(has_pairs, gap_std, torch.zeros_like(gap_std))
        else:
            gap_mean = torch.zeros((B, 1), device=hidden_states.device, dtype=hidden_states.dtype)
            gap_max = torch.zeros((B, 1), device=hidden_states.device, dtype=hidden_states.dtype)
            gap_std = torch.zeros((B, 1), device=hidden_states.device, dtype=hidden_states.dtype)

        seq_len_norm = (lengths / 512.0).to(dtype=hidden_states.dtype)
        gap_stats = torch.cat([gap_mean, gap_max, gap_std, seq_len_norm], dim=-1)

        fused = torch.cat([h_mean, h_attn_spike, h_max_spike, gap_stats], dim=-1)
        emb = F.normalize(self.proj(fused), p=2, dim=-1)

        w_norm = F.normalize(self.weight, p=2, dim=-1)
        cos_sim = F.linear(emb, w_norm)
        cos_sim = cos_sim.view(-1, self.num_classes, self.sub_centers)

        # Max-cosine over sub-centers per class with bounded scale
        logits, _ = cos_sim.max(dim=-1)
        return logits * torch.clamp(self.scale, 5.0, 30.0)


class StylometricMultiMomentPooler(nn.Module):
    """
    Multi-Moment (CLS + Mean + Max-Spike + Std-Dispersion) Pooler for Transformer Encoders (BGE / DeBERTa).
    Normalizes each statistical moment independently before projection so extreme h_max spikes
    never drown out h_mean or syntactic dispersion h_std.
    Supports Multi-Layer Hidden State Fusion across discourse (-1), syntax (-3), and surface structure (-6).
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.layer_mix_weights = nn.Parameter(torch.zeros(3))
        self.norm_cls = nn.LayerNorm(hidden_size)
        self.norm_mean = nn.LayerNorm(hidden_size)
        self.norm_max = nn.LayerNorm(hidden_size)
        self.norm_std = nn.LayerNorm(hidden_size)
        self.dense = nn.Linear(hidden_size * 4, hidden_size)
        self.activation = nn.GELU()
        self.norm_out = nn.LayerNorm(hidden_size)
        self._current_mask = None
        self._extra_layers = None
        nn.init.xavier_uniform_(self.dense.weight, gain=0.5)
        nn.init.zeros_(self.dense.bias)

    @property
    def output_dim(self) -> int:
        return self.dense.out_features

    def set_mask(self, mask: torch.Tensor | None):
        self._current_mask = mask

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor | None = None) -> torch.Tensor:
        mask = attention_mask if attention_mask is not None else self._current_mask
        if self._extra_layers is not None and len(self._extra_layers) == 3:
            w = F.softmax(self.layer_mix_weights, dim=0)
            h_fused = None
            for idx, h_l in enumerate(self._extra_layers):
                h_l_dev = h_l.to(device=hidden_states.device, dtype=hidden_states.dtype)
                h_fused = (w[idx] * h_l_dev) if h_fused is None else (h_fused + w[idx] * h_l_dev)
            hidden_states = h_fused
            self._extra_layers = None
        h_cls = self.norm_cls(hidden_states[:, 0])
        if mask is not None:
            m = mask.to(device=hidden_states.device, dtype=hidden_states.dtype).unsqueeze(-1)
            lengths = torch.clamp(m.sum(dim=1), min=1.0)
            mean_raw = (hidden_states * m).sum(dim=1) / lengths
            h_mean = self.norm_mean(mean_raw)
            masked_h = hidden_states.masked_fill(m == 0, -10000.0)
            h_max_raw, _ = masked_h.max(dim=1)
            h_max = self.norm_max(h_max_raw)
            var_raw = ((hidden_states - mean_raw.unsqueeze(1)) ** 2 * m).sum(dim=1) / lengths
            h_std = self.norm_std(torch.sqrt(torch.clamp(var_raw, min=1e-6)))
        else:
            mean_raw = hidden_states.mean(dim=1)
            h_mean = self.norm_mean(mean_raw)
            h_max_raw, _ = hidden_states.max(dim=1)
            h_max = self.norm_max(h_max_raw)
            h_std = self.norm_std(torch.sqrt(torch.clamp(hidden_states.var(dim=1, unbiased=False), min=1e-6)))
        fused = torch.cat([h_cls, h_mean, h_max, h_std], dim=-1)
        return self.norm_out(self.activation(self.dense(fused)))


def get_model_pooler(model):
    """Retrieves StylometricMultiMomentPooler from DeBERTa, BERT/BGE, or generic model."""
    candidates = [
        getattr(model, "pooler", None),
        getattr(getattr(model, "bert", None), "pooler", None),
        getattr(getattr(model, "deberta", None), "pooler", None),
        getattr(getattr(model, "model", None), "pooler", None),
    ]
    for p in candidates:
        if p is not None and hasattr(p, "layer_mix_weights"):
            return p
    for p in candidates:
        if p is not None and hasattr(p, "dense"):
            return p
    return None


def set_model_pooler_mask(model, mask):
    """Passes current batch attention mask to StylometricMultiMomentPooler if present."""
    pooler = get_model_pooler(model)
    if pooler is not None and hasattr(pooler, "set_mask"):
        pooler.set_mask(mask)


class FocalLoss(nn.Module):
    """
    Focal Loss with per-class weighting and sibling difficulty angular margins.
    Modulates cross entropy by (1 - p_t)^gamma using the predicted probability of the true class.
    Down-weights well-classified easy examples to focus gradients on hard minority classes.
    alpha (class weights) additionally up-weights rare classes.
    Sibling difficulty margins Delta_c force larger angular separation for ambiguous sibling cliques.
    """
    def __init__(self, weight=None, gamma=1.0, label_smoothing=0.01, reduction="mean", use_sibling_margins: bool = True, sibling_margins=None):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction
        self.use_sibling_margins = use_sibling_margins
        if weight is not None:
            self.register_buffer("weight", weight)
        else:
            self.weight = None

        if sibling_margins is not None:
            sm = torch.tensor(sibling_margins, dtype=torch.float32) if not isinstance(sibling_margins, torch.Tensor) else sibling_margins.float()
            self.register_buffer("sibling_margins", sm)
        elif use_sibling_margins:
            self.register_buffer("sibling_margins", torch.tensor(SIBLING_DIFFICULTY_MARGINS, dtype=torch.float32))
        else:
            self.sibling_margins = None

    def forward(self, logits, targets):
        num_classes = logits.size(-1)
        targets = targets.to(device=logits.device, dtype=torch.long).clamp(0, num_classes - 1)
        logits_f = logits.float()
        if self.use_sibling_margins and self.sibling_margins is not None and self.sibling_margins.size(0) == num_classes:
            sm = self.sibling_margins.to(logits.device)
            target_margins = sm[targets].unsqueeze(1)
            logits_f = logits_f.scatter_add(1, targets.unsqueeze(1), -target_margins)
        log_prob = torch.nn.functional.log_softmax(logits_f, dim=-1)
        log_p_t = log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        p_t = torch.exp(log_p_t)
        focal_weight = (1.0 - p_t).pow(self.gamma)
        if self.label_smoothing > 0:
            eps = self.label_smoothing
            sum_log_prob = log_prob.sum(dim=-1)
            # Analytical label smoothed cross entropy: -(1 - eps) * log(p_t) - (eps / (K - 1)) * sum_{c != t} log(p_c)
            # where sum_{c != t} log(p_c) = sum_log_prob - log_p_t
            ce_loss = -(1.0 - eps) * log_p_t - (eps / (num_classes - 1)) * (sum_log_prob - log_p_t)
        else:
            ce_loss = -log_p_t
        loss = focal_weight * ce_loss
        if self.weight is not None:
            w = self.weight.to(logits.device)[targets]
            loss = loss * w
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


class BalancedLogitAdjustedLDAMLoss(nn.Module):
    """
    SOTA Long-Tail Loss combining:
      1. Menon et al. (ICLR 2021) Logit Adjustment to remove empirical class prior skew
      2. Cao et al. (NeurIPS 2019) LDAM Margin (Delta_c ~ N_c^{-1/4}) to widen minority sibling boundaries
      3. Focal modulation (1 - p_t)^gamma to focus on hard ambiguous passages
    """
    def __init__(self, class_counts: list[int] | np.ndarray | torch.Tensor, tau: float = 1.0, max_margin: float = 0.35, gamma: float = 1.0):
        super().__init__()
        if not isinstance(class_counts, torch.Tensor):
            counts = torch.tensor(class_counts, dtype=torch.float32)
        else:
            counts = class_counts.float()
        counts = counts.clamp(min=1.0)
        priors = counts / counts.sum()
        self.register_buffer("log_priors", tau * torch.log(priors))

        # LDAM margin: inversely proportional to N_c^(1/4) so starved classes get widest margin
        margins = 1.0 / torch.pow(counts, 0.25)
        margins = margins * (max_margin / margins.max())
        self.register_buffer("margins", margins)
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.to(device=logits.device, dtype=torch.long)
        target_margins = self.margins.to(logits.device)[targets].unsqueeze(1)
        margin_logits = logits.scatter_add(1, targets.unsqueeze(1), -target_margins)
        adjusted_logits = margin_logits + self.log_priors.unsqueeze(0).to(logits.device)

        log_probs = F.log_softmax(adjusted_logits.float(), dim=-1)
        target_log_probs = log_probs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
        probs = torch.exp(target_log_probs)
        focal_weight = (1.0 - probs).pow(self.gamma)
        return -(focal_weight * target_log_probs).mean()


def get_optimizer_grouped_parameters(model, lr_head, lr_backbone, layer_decay=0.95, weight_decay=0.01):
    """
    Layer-Wise Learning Rate Decay (LLRD) for 24-layer transformers (DeBERTa / BGE).
    Decays learning rate geometrically from top layer down to embeddings (ξ = 0.95).
    Explicitly assigns rel_embeddings to embeddings LR and LayerNorm to top layer LR.
    Guarantees 100% parameter coverage without generic frozen buckets.
    """
    def is_no_decay(name, param):
        return param.ndim < 2 or any(nd in name.lower() for nd in ["bias", "norm"])

    base_model = getattr(model, "deberta", getattr(model, "bert", getattr(model, "model", model)))
    encoder = getattr(base_model, "encoder", None)
    layers = getattr(encoder, "layer", None)
    embeddings = getattr(base_model, "embeddings", None)

    assigned_param_ids = set()
    params_group = []

    # Head and pooler modules (receive lr_head)
    head_modules = [
        getattr(model, "classifier", None),
        getattr(model, "pooler", None),
        getattr(base_model, "pooler", None),
    ]
    head_params = []
    for mod in head_modules:
        if mod is not None:
            head_params.extend(list(mod.named_parameters()))

    if head_params:
        p_decay = [p for n, p in head_params if not is_no_decay(n, p) and p.requires_grad]
        p_nodecay = [p for n, p in head_params if is_no_decay(n, p) and p.requires_grad]
        if p_decay:
            params_group.append({"params": p_decay, "lr": lr_head, "weight_decay": weight_decay})
        if p_nodecay:
            params_group.append({"params": p_nodecay, "lr": lr_head, "weight_decay": 0.0})
        for n, p in head_params:
            if p.requires_grad:
                assigned_param_ids.add(id(p))

    # Encoder layers with geometric decay (ξ = 0.95)
    if layers is not None:
        n_layers = len(layers)
        for i in reversed(range(n_layers)):
            layer = layers[i]
            decay_factor = layer_decay ** (n_layers - 1 - i)
            layer_lr = lr_backbone * decay_factor
            l_params = list(layer.named_parameters())
            p_decay = [p for n, p in l_params if not is_no_decay(n, p) and p.requires_grad]
            p_nodecay = [p for n, p in l_params if is_no_decay(n, p) and p.requires_grad]
            if p_decay:
                params_group.append({"params": p_decay, "lr": layer_lr, "weight_decay": weight_decay})
            if p_nodecay:
                params_group.append({"params": p_nodecay, "lr": layer_lr, "weight_decay": 0.0})
            for n, p in l_params:
                if p.requires_grad:
                    assigned_param_ids.add(id(p))

    # Embeddings layer: decayed geometrically below bottom encoder layer to prevent memorizing rare prompt nouns
    n_layers = len(layers) if layers is not None else 24
    emb_lr = lr_backbone * (layer_decay ** n_layers)
    emb_params = []
    if embeddings is not None:
        emb_params.extend(list(embeddings.named_parameters()))
    if encoder is not None and hasattr(encoder, "rel_embeddings"):
        emb_params.extend(list(encoder.rel_embeddings.named_parameters()))

    if emb_params:
        p_decay = [p for n, p in emb_params if not is_no_decay(n, p) and p.requires_grad]
        p_nodecay = [p for n, p in emb_params if is_no_decay(n, p) and p.requires_grad]
        if p_decay:
            params_group.append({"params": p_decay, "lr": emb_lr, "weight_decay": weight_decay})
        if p_nodecay:
            params_group.append({"params": p_nodecay, "lr": emb_lr, "weight_decay": 0.0})
        for n, p in emb_params:
            if p.requires_grad:
                assigned_param_ids.add(id(p))

    # Encoder final LayerNorm assigned to top layer LR
    if encoder is not None and hasattr(encoder, "LayerNorm"):
        ln_params = list(encoder.LayerNorm.named_parameters())
        p_ln = [p for n, p in ln_params if p.requires_grad]
        if p_ln:
            params_group.append({"params": p_ln, "lr": lr_backbone, "weight_decay": 0.0})
            for n, p in ln_params:
                if p.requires_grad:
                    assigned_param_ids.add(id(p))

    # Fallback for any leftover parameters (ensures zero frozen parameters accidentally included)
    leftover = [(n, p) for n, p in model.named_parameters() if id(p) not in assigned_param_ids and p.requires_grad]
    if leftover:
        p_decay = [p for n, p in leftover if not is_no_decay(n, p)]
        p_nodecay = [p for n, p in leftover if is_no_decay(n, p)]
        if p_decay:
            params_group.append({"params": p_decay, "lr": lr_backbone, "weight_decay": weight_decay})
        if p_nodecay:
            params_group.append({"params": p_nodecay, "lr": lr_backbone, "weight_decay": 0.0})

    return params_group

# --- 6. UNIFIED HIGH-THROUGHPUT TRANSFORMER TRAINING (GPU 0 & GPU 1) ---

def _seq_equal(a, b):
    if isinstance(a, list) and isinstance(b, list):
        return a == b
    return np.array_equal(a, b)


def _blend_dedup_tta(eval_fn, enc_data, enc_data_tta):
    """Evaluate enc_data once and only re-evaluate enc_data_tta indices whose tokens actually differ."""
    base_logits = eval_fn(enc_data)
    if enc_data_tta is None:
        return base_logits
    n = len(enc_data.get("input_ids", []))
    ids_base = enc_data.get("input_ids", [])
    ids_tta = enc_data_tta.get("input_ids", [])
    mask_base = enc_data.get("attention_mask", [])
    mask_tta = enc_data_tta.get("attention_mask", [])
    tti_base = enc_data.get("token_type_ids")
    tti_tta = enc_data_tta.get("token_type_ids")
    if len(ids_tta) != n or (tti_base is None) != (tti_tta is None):
        diff_idx = list(range(n))
    else:
        has_tti = tti_base is not None
        diff_idx = [
            i for i in range(n)
            if not (
                _seq_equal(ids_base[i], ids_tta[i])
                and _seq_equal(mask_base[i], mask_tta[i])
                and (not has_tti or _seq_equal(tti_base[i], tti_tta[i]))
            )
        ]
    if not diff_idx:
        return base_logits
    sub_tta = {k: [v[i] for i in diff_idx] for k, v in enc_data_tta.items() if isinstance(v, list)}
    tta_sub_logits = eval_fn(sub_tta)
    blended = base_logits.copy()
    blended[diff_idx] = 0.5 * base_logits[diff_idx] + 0.5 * tta_sub_logits
    return blended


def evaluate_transformer_split(model, enc_data, tokenizer, dev, use_amp, num_classes, enc_data_tta=None):
    """
    Length-sorted batched inference for encoder models with eval_bs=48 and Multi-Layer Fusion.
    Groups evaluation passages by length to eliminate redundant padding tokens,
    then inverts the sort order back to the original index sequence.
    Optionally evaluates only diverging enc_data_tta sequences (Deduplicated TTA) and averages logits.
    """
    def _eval_single(enc):
        n_samples = len(enc["input_ids"]) if "input_ids" in enc else 0
        if n_samples == 0:
            return np.zeros((0, num_classes), dtype=np.float64)
        lengths = [len(x) for x in enc["input_ids"]]
        sort_idx = np.argsort(lengths)
        cuda_ok = torch.cuda.is_available() and getattr(dev, "type", str(dev)) != "cpu"
        pooler = get_model_pooler(model)
        eval_bs = 48  # Fast inference under inference_mode() on Tesla T4 (with OOM halving backoff)
        while True:
            all_logits = []
            try:
                with inference_context():
                    for start in range(0, n_samples, eval_bs):
                        end = min(start + eval_bs, n_samples)
                        batch_indices = sort_idx[start:end]
                        sl = {
                            "input_ids": [enc["input_ids"][j] for j in batch_indices],
                            "attention_mask": [enc["attention_mask"][j] for j in batch_indices],
                        }
                        if "token_type_ids" in enc:
                            sl["token_type_ids"] = [enc["token_type_ids"][j] for j in batch_indices]
                        padded = tokenizer.pad(sl, padding=True, return_tensors="pt")
                        kw = {
                            "input_ids": padded["input_ids"].to(dev),
                            "attention_mask": padded["attention_mask"].to(dev),
                        }
                        if "token_type_ids" in padded:
                            kw["token_type_ids"] = padded["token_type_ids"].to(dev)
                        with torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.float16):
                            set_model_pooler_mask(model, kw["attention_mask"])
                            logits = model(**kw).logits
                        all_logits.append(logits.float().cpu().numpy())
                break
            except torch.cuda.OutOfMemoryError:
                set_model_pooler_mask(model, None)
                if pooler is not None:
                    pooler._extra_layers = None
                logits = kw = padded = None
                all_logits.clear()
                gc.collect()
                if cuda_ok:
                    torch.cuda.empty_cache()
                if eval_bs <= 1:
                    raise
                eval_bs = max(eval_bs // 2, 1)
                print(f"  [Eval] CUDA OOM at eval_bs={eval_bs * 2}; retrying with eval_bs={eval_bs}...", flush=True)
        set_model_pooler_mask(model, None)
        sorted_logits = np.concatenate(all_logits, axis=0)
        inv_idx = np.empty_like(sort_idx)
        inv_idx[sort_idx] = np.arange(len(sort_idx))
        return sorted_logits[inv_idx]

    model.eval()
    return _blend_dedup_tta(_eval_single, enc_data, enc_data_tta)


def check_finite_logits(logits: np.ndarray, name: str) -> np.ndarray:
    """Validates that a logits matrix contains strictly finite values."""
    if logits is None:
        return None
    if not np.all(np.isfinite(logits)):
        n_nan = int(np.sum(np.isnan(logits)))
        n_inf = int(np.sum(np.isinf(logits)))
        raise FloatingPointError(f"[{name}] Non-finite logits detected: {n_nan} NaNs, {n_inf} Infs in shape {logits.shape}")
    return logits


@contextlib.contextmanager
def capture_intermediate_layers(model, layer_indices=(-1, -5, -9), target_pooler=None):
    """
    Lightweight forward-hook capture of intermediate decoder/encoder layers.
    If target_pooler is provided, injects the captured layers directly into the pooler
    before it executes. Resets captured state at the earliest layer to support lifecycle
    across multiple micro-batches without hook re-registration overhead.
    Uses recursive layer discovery to navigate arbitrary HF/PEFT wrapper nesting.
    """
    def _find_layers(m, depth=0):
        if depth > 6 or m is None:
            return None
        if hasattr(m, "layers") and isinstance(m.layers, (nn.ModuleList, tuple, list)):
            return m.layers
        if hasattr(m, "layer") and isinstance(m.layer, (nn.ModuleList, tuple, list)):
            return m.layer
        for child_name in ["base_model", "model", "encoder", "transformer", "bert", "deberta", "roberta"]:
            if hasattr(m, child_name):
                res = _find_layers(getattr(m, child_name), depth + 1)
                if res is not None:
                    return res
        for child in m.children():
            if hasattr(child, "layers") and isinstance(child.layers, (nn.ModuleList, tuple, list)):
                return child.layers
            if hasattr(child, "layer") and isinstance(child.layer, (nn.ModuleList, tuple, list)):
                return child.layer
        return None

    dec_layers = _find_layers(model)
    captured = {}
    handles = []
    if dec_layers is not None:
        n_dec = len(dec_layers)
        actual_indices = [idx if idx >= 0 else n_dec + idx for idx in layer_indices]
        valid_indices = [a for a in actual_indices if 0 <= a < n_dec]
        min_idx = min(valid_indices) if valid_indices else None
        for orig_idx, actual_idx in zip(layer_indices, actual_indices):
            if 0 <= actual_idx < n_dec:
                def _make_hook(orig_i, act_i):
                    def _hook(m, inp, out):
                        if act_i == min_idx:
                            captured.clear()
                        captured[orig_i] = out[0] if isinstance(out, tuple) else out
                        if target_pooler is not None and len(captured) == len(layer_indices):
                            target_pooler._extra_layers = [captured[x] for x in layer_indices]
                            captured.clear()
                    return _hook
                handles.append(dec_layers[actual_idx].register_forward_hook(_make_hook(orig_idx, actual_idx)))
    try:
        yield captured
    finally:
        for h in handles:
            h.remove()
        if target_pooler is not None:
            target_pooler._extra_layers = None
        captured.clear()


def evaluate_qwen_split(backbone, head, enc_data, tokenizer, first_dev, head_dev, cuda_ok, num_classes, enc_data_tta=None):
    """
    Length-sorted batched inference for Qwen 3 8B / 2.5 7B LoRA decoder.
    Passes representations through backbone to SpikeAndGapHierarchicalHead or classification head.
    Optionally evaluates enc_data_tta and averages test predictions (Test-Time Augmentation).
    """
    def _eval_single(enc):
        n_samples = len(enc["input_ids"]) if "input_ids" in enc else 0
        if n_samples == 0:
            return np.zeros((0, num_classes), dtype=np.float64)
        lengths = [len(x) for x in enc["input_ids"]]
        sort_idx = np.argsort(lengths)
        inf_bs = 24  # Fast sharded inference under inference_mode() (with OOM halving backoff)
        autocast_ctx = amp_autocast() if cuda_ok else contextlib.nullcontext()
        while True:
            all_logits = []
            try:
                with inference_context():
                    for start in range(0, n_samples, inf_bs):
                        end = min(start + inf_bs, n_samples)
                        batch_indices = sort_idx[start:end]
                        sl = {
                            "input_ids": [enc["input_ids"][j] for j in batch_indices],
                            "attention_mask": [enc["attention_mask"][j] for j in batch_indices],
                        }
                        padded = tokenizer.pad(sl, padding=True, return_tensors="pt")
                        batch_kwargs = {
                            "input_ids": padded["input_ids"].to(first_dev),
                            "attention_mask": padded["attention_mask"].to(first_dev),
                        }
                        with autocast_ctx:
                            out = backbone(**batch_kwargs, output_hidden_states=False)
                            h_input = getattr(out, "last_hidden_state", out[0] if isinstance(out, tuple) else out)
                            del out
                            if isinstance(head, SpikeAndGapHierarchicalHead):
                                logits = head(h_input, batch_kwargs["attention_mask"]).float().cpu().numpy()
                            else:
                                last_h = h_input[0] if isinstance(h_input, (tuple, list)) else h_input
                                if getattr(tokenizer, "padding_side", "left") == "left":
                                    pooled = last_h[:, -1, :].float()
                                else:
                                    attn_mask = batch_kwargs["attention_mask"].to(last_h.device)
                                    seq_lens = torch.clamp(attn_mask.sum(dim=1) - 1, min=0)
                                    batch_idx = torch.arange(last_h.size(0), device=last_h.device)
                                    pooled = last_h[batch_idx, seq_lens].float()
                                logits = head(pooled.to(head_dev)).float().cpu().numpy()
                            del h_input
                        all_logits.append(logits)
                break
            except torch.cuda.OutOfMemoryError:
                out = h_input = logits = batch_kwargs = padded = None
                all_logits.clear()
                gc.collect()
                if cuda_ok:
                    for gi in range(torch.cuda.device_count()):
                        with torch.cuda.device(gi):
                            torch.cuda.synchronize(gi)
                            torch.cuda.empty_cache()
                if inf_bs <= 1:
                    raise
                inf_bs = max(inf_bs // 2, 1)
                print(f"  [Qwen Eval] CUDA OOM at inf_bs={inf_bs * 2}; retrying with inf_bs={inf_bs}...", flush=True)
        sorted_logits = np.concatenate(all_logits, axis=0)
        inv_idx = np.empty_like(sort_idx)
        inv_idx[sort_idx] = np.arange(len(sort_idx))
        return sorted_logits[inv_idx]

    backbone.eval()
    head.eval()
    return _blend_dedup_tta(_eval_single, enc_data, enc_data_tta)


def get_cosine_schedule_with_warmup_and_min_lr(
    optimizer,
    num_warmup_steps: int,
    num_training_steps: int,
    min_lr_ratio: float = 0.10,
    num_cycles: float = 0.5,
    last_epoch: int = -1,
):
    """
    Cosine learning rate schedule with linear warmup and non-zero minimum LR floor.
    Standard cosine schedules decay to 0.0 at the final step, which stalls parameter updates
    during the final 20% of training when SWA checkpoints are accumulated.
    By enforcing a minimum floor (min_lr_ratio=0.10 of initial lr), the optimizer continues to
    explore the loss basin and generates diverse checkpoints for SWA averaging.
    """
    def lr_lambda(current_step: int):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * max(0.0, cosine_decay)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda, last_epoch)


def _train_transformer_fold_impl(
    model_name,
    texts_tr, y_tr_str, texts_va, y_va_str, texts_te,
    model_dir, labels,
    device="cuda:0",
    batch_size=12,
    grad_accum=1,
    max_length=448,
    epochs=3,
    lr_backbone=1.5e-5,
    lr_head=1.2e-4,
    weight_decay=0.01,
    label_smoothing=0.01,
    layer_decay=0.95,
    enable_grad_checkpointing=True,
    deadline=None,
    cached_enc_tr=None,
    cached_enc_va=None,
    cached_enc_te=None,
    cached_enc_te_tta=None,
    cached_tokenizer=None,
    group_ids_tr=None,
    use_topic_masking=True,
    use_sift=True,
    n_frozen_bottom_layers=4,
):
    """
    Unified, high-throughput transformer fine-tuning loop for DeBERTa-v3-large and BGE-large-en-v1.5.
    Features:
    - Stylistic [NL] newline preservation + Head+Tail truncation (224 head + 224 tail = 448 context).
    - FastBatchPrefetcher with background worker thread and pinned RAM transfers (100% GPU saturation).
    - Upfront RAM token cache reuse (eliminates redundant tokenizations across folds).
    - Gradient checkpointing ON by default (reduces activation VRAM by 90%, guaranteed zero OOM).
    - LLRD (layer_decay = 0.95) with explicit relative embeddings and LayerNorm discovery.
    - Synchronized gradient accumulation with non-fatal NaN recovery.
    - Protected scheduler stepping with GradScaler scale checking (no PyTorch UserWarnings).
    - Per-device context cache clearing.
    """
    label2id = {l: i for i, l in enumerate(labels)}
    texts_tr = list(texts_tr)
    texts_va = list(texts_va)
    texts_te = list(texts_te)

    y_tr_int = np.array([label2id[y] for y in y_tr_str], dtype=np.int64)
    y_va_int = np.array([label2id[y] for y in y_va_str], dtype=np.int64)

    cuda_ok = torch.cuda.is_available()
    dev = torch.device(device if cuda_ok else "cpu")
    if cuda_ok:
        try:
            torch.cuda.set_device(dev)
        except Exception:
            pass

    model_tag = "deberta" if "deberta" in model_name.lower() else ("bge" if "bge" in model_name.lower() else "model")
    model_dir = prepare_model_dir(model_dir, model_tag=model_tag)
    print(f"  [{model_name}] Loading from {model_dir} on {dev}...", flush=True)

    if cached_tokenizer is not None:
        tokenizer = cached_tokenizer
    else:
        try:
            tokenizer = AutoTokenizer.from_pretrained(
                model_dir, local_files_only=True, trust_remote_code=True, fix_mistral_regex=True
            )
        except TypeError:
            tokenizer = AutoTokenizer.from_pretrained(
                model_dir, local_files_only=True, trust_remote_code=True
            )
        tokenizer.add_tokens(["[NL]", "[PAR]"], special_tokens=True)

    id2label = {i: l for i, l in enumerate(labels)}

    with RAM_CACHE_LOCK:
        has_cached = model_tag in GLOBAL_RAM_MODEL_CACHE
        if has_cached:
            cached_weights = GLOBAL_RAM_MODEL_CACHE[model_tag]
            cached_config = GLOBAL_RAM_MODEL_CACHE[f"{model_tag}_config"]

    if has_cached:
        print(f"  [{model_name}] Loading base weights from Host RAM cache (0 disk I/O)...", flush=True)
        try:
            model = AutoModelForSequenceClassification.from_config(cached_config)
        except Exception:
            model = AutoModelForSequenceClassification.from_pretrained(
                model_dir, num_labels=len(labels), id2label=id2label, label2id=label2id, local_files_only=True
            )
        if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
            model.resize_token_embeddings(len(tokenizer))
        model.load_state_dict(cached_weights, strict=False)
    else:
        with HeartbeatTimer(f"Reading {model_name} from disk onto {dev}"):
            kw_load = {
                "num_labels": len(labels),
                "id2label": id2label,
                "label2id": label2id,
                "local_files_only": True,
                "trust_remote_code": True,
                "ignore_mismatched_sizes": True,
                "low_cpu_mem_usage": True,
            }
            try:
                model = AutoModelForSequenceClassification.from_pretrained(
                    model_dir,
                    dtype=torch.float32,
                    **kw_load,
                )
            except TypeError:
                try:
                    model = AutoModelForSequenceClassification.from_pretrained(
                        model_dir,
                        torch_dtype=torch.float32,
                        **kw_load,
                    )
                except TypeError:
                    kw_load.pop("low_cpu_mem_usage", None)
                    model = AutoModelForSequenceClassification.from_pretrained(
                        model_dir,
                        torch_dtype=torch.float32,
                        **kw_load,
                    )

        # Ensure model embeddings match tokenizer vocabulary size and initialize newline weights unconditionally
        emb_module = model.get_input_embeddings()
        if len(tokenizer) > emb_module.weight.shape[0]:
            model.resize_token_embeddings(len(tokenizer))
            emb_module = model.get_input_embeddings()
        with torch.no_grad():
            rng_emb = torch.Generator().manual_seed(42)
            emb_std = float(emb_module.weight[:min(len(tokenizer), 30000)].std().clamp(min=1e-4))
            for new_tok, src_tok in (("[NL]", "."), ("[PAR]", "|")):
                n_id = tokenizer.convert_tokens_to_ids(new_tok)
                s_id = tokenizer.convert_tokens_to_ids(src_tok)
                if n_id is not None and 0 <= n_id < emb_module.weight.shape[0]:
                    if s_id is not None and 0 <= s_id < emb_module.weight.shape[0] and s_id != getattr(tokenizer, "unk_token_id", -1):
                        base_vec = emb_module.weight[s_id].clone()
                    else:
                        base_vec = emb_module.weight[:min(1000, emb_module.weight.shape[0])].mean(dim=0).clone()
                    ortho_delta = torch.randn(emb_module.weight.shape[1], generator=rng_emb, dtype=base_vec.dtype) * (0.5 * emb_std)
                    emb_module.weight[n_id].copy_(base_vec + ortho_delta.to(base_vec.device))

        try:
            with RAM_CACHE_LOCK:
                GLOBAL_RAM_MODEL_CACHE[model_tag] = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                GLOBAL_RAM_MODEL_CACHE[f"{model_tag}_config"] = copy.deepcopy(model.config)
            cache_mb = sum(p.numel() * p.element_size() for p in GLOBAL_RAM_MODEL_CACHE[model_tag].values()) / (1024 ** 2)
            print(f"  [{model_name}] Staged base weights to Host RAM cache ({cache_mb:.1f} MB)", flush=True)
        except Exception as e:
            print(f"  [{model_name}] Host RAM caching skipped: {e}", flush=True)

    model.config.use_cache = False
    in_features = model.classifier.in_features if hasattr(model.classifier, "in_features") else model.config.hidden_size
    model.classifier = MultiSampleDropoutHead(in_features, len(labels))
    if hasattr(model, "dropout"):
        model.dropout = nn.Identity()
    if hasattr(model, "pooler") and hasattr(model.pooler, "dropout"):
        model.pooler.dropout = nn.Identity()

    # Stylometric Multi-Moment Pooler for BGE and DeBERTa
    # Replaces single-token [CLS] pooler to break semantic collapse and preserve sequence syntax & token spikes
    pooler_instance = StylometricMultiMomentPooler(in_features)
    if "bge" in model_name.lower():
        if hasattr(model, "bert") and hasattr(model.bert, "pooler"):
            model.bert.pooler = pooler_instance
        elif hasattr(model, "pooler"):
            model.pooler = pooler_instance
    elif "deberta" in model_name.lower():
        if hasattr(model, "pooler"):
            model.pooler = pooler_instance
        elif hasattr(model, "deberta") and hasattr(model.deberta, "pooler"):
            model.deberta.pooler = pooler_instance

    use_reentrant_ckpt = False
    ckpt_active = bool(enable_grad_checkpointing)
    if ckpt_active:
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            use_reentrant_ckpt = True
            model.gradient_checkpointing_enable()

    if n_frozen_bottom_layers > 0:
        base_m = getattr(model, "deberta", getattr(model, "bert", getattr(model, "model", None)))
        if base_m is not None:
            embeddings = getattr(base_m, "embeddings", None)
            if embeddings is not None:
                for p in embeddings.parameters():
                    p.requires_grad = False
        encoder = getattr(base_m, "encoder", None) if base_m is not None else None
        if encoder is not None:
            rel_embeddings = getattr(encoder, "rel_embeddings", None)
            if rel_embeddings is not None:
                for p in rel_embeddings.parameters():
                    p.requires_grad = False
            layers = getattr(encoder, "layer", None)
            if layers is not None:
                n_freeze = min(n_frozen_bottom_layers, len(layers))
                for l_idx in range(n_freeze):
                    for p in layers[l_idx].parameters():
                        p.requires_grad = False
                print(f"  [{model_name}] Frozen bottom {n_freeze} layers + embeddings (true zero-grad skip on layers 0..{n_freeze-1}, saves 1.5GB+ VRAM).", flush=True)

    model = model.to(dev)
    gc.collect()
    if cuda_ok:
        if str(dev).startswith("cuda"):
            dev_idx = torch.device(dev).index if torch.device(dev).index is not None else 0
            with torch.cuda.device(dev_idx):
                torch.cuda.synchronize(dev_idx)
                torch.cuda.empty_cache()
        else:
            torch.cuda.empty_cache()
    ckpt_status = "ON" if enable_grad_checkpointing else "OFF"
    print(f"  [{model_name}] Ready on {dev} (FP16 Tensor Cores ON, grad_ckpt={ckpt_status}, Multi-Sample Dropout ON). {get_mem_status(dev)}", flush=True)

    cw = compute_smoothed_class_weights(y_tr_int, num_classes=len(labels), gamma=0.5)
    weight_tensor = torch.tensor(cw, dtype=torch.float32, device=dev)
    criterion = FocalLoss(weight=weight_tensor, gamma=1.0, label_smoothing=label_smoothing)
    family_criterion = nn.CrossEntropyLoss().to(dev)
    FAMILY_AUX_WEIGHT = 0.25
    label_to_fam_tensor = torch.tensor(LABEL_TO_FAMILY_IDX, dtype=torch.long, device=dev)

    opt_params = get_optimizer_grouped_parameters(
        model, lr_head=lr_head, lr_backbone=lr_backbone, layer_decay=layer_decay, weight_decay=weight_decay
    )
    optimizer = AdamW(opt_params, foreach=cuda_ok)

    n_opt_steps_per_epoch = max((len(texts_tr) + batch_size * grad_accum - 1) // (batch_size * grad_accum), 1)
    total_steps = n_opt_steps_per_epoch * epochs
    scheduler = get_cosine_schedule_with_warmup_and_min_lr(
        optimizer,
        num_warmup_steps=max(int(0.1 * total_steps), 1),
        num_training_steps=total_steps,
        min_lr_ratio=0.10,
    )

    if cached_enc_tr is not None:
        enc_tr = cached_enc_tr
        enc_va = cached_enc_va
        enc_te = cached_enc_te
        print(f"  [{model_name}] Using RAM-cached token encodings ({len(enc_tr['input_ids'])} train / {len(enc_va['input_ids'])} val / {len(enc_te['input_ids'])} test passages).", flush=True)
    else:
        print(f"  [{model_name}] Tokenizing {len(texts_tr)} train / {len(texts_va)} val / {len(texts_te)} test passages (Head+Tail L={max_length}, [NL] structure preserved)...", flush=True)
        enc_tr = tokenize_with_head_tail(tokenizer, texts_tr, max_length=max_length, preserve_newlines=True)
        enc_va = tokenize_with_head_tail(tokenizer, texts_va, max_length=max_length, preserve_newlines=True)
        enc_te = tokenize_with_head_tail(tokenizer, texts_te, max_length=max_length, preserve_newlines=True)

    use_amp = cuda_ok
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp, init_scale=4096.0)

    best_val_score = -1.0
    best_val_logits = None
    best_state = None
    epoch_states = []
    tr_lengths = np.array([len(x) for x in enc_tr["input_ids"]])
    n_total_micros = (len(texts_tr) + batch_size - 1) // batch_size
    mega_size = batch_size * 8
    stop_early = False
    n_skipped_micros = 0
    n_oom_skips = 0
    t_train_start = time.perf_counter()
    pure_train_sec = 0.0
    first_step_overhead = 0.0
    n_total_micros_executed = 0
    steady_samples_executed = 0

    topic_prep = None
    if use_topic_masking and group_ids_tr is not None and getattr(tokenizer, "mask_token_id", None) is not None:
        try:
            special_ids = set(tokenizer.all_special_ids) if hasattr(tokenizer, "all_special_ids") else set()
            topic_token_ids = find_topic_token_ids(enc_tr["input_ids"], group_ids_tr, min_prompts=3, special_ids=special_ids)
            sift_eps = 0.015 if use_sift else 0.0
            topic_prep = TopicBlindAndSiFTPrep(topic_token_ids, mask_token_id=tokenizer.mask_token_id, mask_prob=0.30, sift_eps=sift_eps)
            print(f"  [{model_name}] TopicBlind & SiFT prep active: {len(topic_token_ids)} rare topic tokens identified (mask_prob=0.30, sift_eps={sift_eps}).", flush=True)
        except Exception as e:
            print(f"  [{model_name}] Note: TopicBlind prep skipped ({e}).", flush=True)
            topic_prep = None

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    trainable_names = {n for n, p in model.named_parameters() if p.requires_grad}

    for epoch in range(epochs):
        model.train()
        if group_ids_tr is not None:
            epoch_indices = make_prompt_sibling_paired_indices(group_ids_tr, y_tr_int, tr_lengths, batch_size, np.random.RandomState(42 + epoch))
        else:
            perm = np.random.RandomState(42 + epoch).permutation(len(texts_tr))
            sorted_indices = []
            for m_start in range(0, len(texts_tr), mega_size):
                m_chunk = perm[m_start:m_start + mega_size]
                sorted_indices.extend(m_chunk[np.argsort(tr_lengths[m_chunk])])
            epoch_indices = np.array(sorted_indices, dtype=np.int64)

        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        n_micro = 0
        accumulated_steps = 0
        ep_samples = 0
        ep_steady_samples = 0

        t_batch_start = time.perf_counter()
        t_step1_done = None
        prefetcher = FastBatchPrefetcher(
            tokenizer, enc_tr, y_tr_int, epoch_indices, batch_size, dev, cuda_ok=cuda_ok, topic_prep=topic_prep, group_ids=group_ids_tr
        )
        pooler_inst = get_model_pooler(model)
        try:
            for micro_step, (batch_kwargs, batch_labels) in enumerate(prefetcher):
                    if deadline is not None and (micro_step % 10 == 0) and time.perf_counter() > deadline:
                        print(f"  [{model_name}] [Time Guard] Deadline reached mid-epoch at step {micro_step}/{n_total_micros}. Stopping training.", flush=True)
                        stop_early = True
                        break

                    try:
                        batch_groups = batch_kwargs.pop("_batch_groups", getattr(prefetcher, "current_groups", None))
                        with torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.float16):
                            set_model_pooler_mask(model, batch_kwargs["attention_mask"])
                            if topic_prep is not None and topic_prep.sift_eps > 0:
                                emb = model.get_input_embeddings()(batch_kwargs["input_ids"])
                                emb = topic_prep.apply_sift_embedding_noise(emb, batch_kwargs["attention_mask"])
                                if n_frozen_bottom_layers == 0 or use_reentrant_ckpt:
                                    emb = emb.requires_grad_(True)
                                kw_forward = {
                                    "inputs_embeds": emb,
                                    "attention_mask": batch_kwargs["attention_mask"],
                                }
                                if "token_type_ids" in batch_kwargs:
                                    kw_forward["token_type_ids"] = batch_kwargs["token_type_ids"]
                                logits = model(**kw_forward).logits
                            else:
                                logits = model(**batch_kwargs).logits
                            logits_f = logits.float()
                        main_loss = criterion(logits_f, batch_labels)
                        fam_logits = compute_family_logits(logits_f, temp=4.0)
                        batch_fam_labels = label_to_fam_tensor[batch_labels]
                        aux_loss = family_criterion(fam_logits, batch_fam_labels)
                        loss = (main_loss + FAMILY_AUX_WEIGHT * aux_loss) / grad_accum
                        if batch_groups is not None:
                            repulsion_loss = compute_same_prompt_sibling_repulsion(logits_f, batch_groups, batch_labels, margin=0.15)
                            loss = loss + (0.04 * repulsion_loss) / grad_accum
                        branch_var = getattr(getattr(model, "classifier", None), "last_branch_var", None)
                        if branch_var is not None:
                            loss = loss + (0.03 * branch_var) / grad_accum

                        if use_amp:
                            scaler.scale(loss).backward()
                        else:
                            loss.backward()

                        loss_val = loss.item() * grad_accum
                        if math.isnan(loss_val) or math.isinf(loss_val):
                            n_skipped_micros += 1
                            print(f"  [{model_name}] Warning: NaN/Inf loss in micro-step {micro_step}. Skipping micro-batch ({n_skipped_micros} total skips).", flush=True)
                            if n_skipped_micros > 0.15 * max(n_total_micros * (epoch + 1), 1):
                                raise FloatingPointError(f"[{model_name}] >15% micro-batches ({n_skipped_micros}) produced NaN/Inf loss. Unstable FP16 training detected; discarding fold.")
                            optimizer.zero_grad(set_to_none=True)
                            accumulated_steps = 0
                            continue

                    except torch.cuda.OutOfMemoryError:
                        n_oom_skips += 1
                        print(f"  [{model_name}] CUDA OOM detected at micro-step {micro_step} (OOM #{n_oom_skips}). Resetting partial grads...", flush=True)
                        if n_oom_skips > 5:
                            raise RuntimeError(f"[{model_name}] Exceeded 5 CUDA OOM errors in fold; aborting to prevent data starvation.")
                        set_model_pooler_mask(model, None)
                        if pooler_inst is not None:
                            pooler_inst._extra_layers = None
                        try:
                            del logits, loss, batch_kwargs, batch_labels
                        except Exception:
                            pass
                        optimizer.zero_grad(set_to_none=True)
                        accumulated_steps = 0
                        if cuda_ok and str(dev).startswith("cuda"):
                            dev_idx = torch.device(dev).index if torch.device(dev).index is not None else 0
                            with torch.cuda.device(dev_idx):
                                torch.cuda.synchronize(dev_idx)
                                torch.cuda.empty_cache()
                        if not ckpt_active and hasattr(model, "gradient_checkpointing_enable"):
                            try:
                                model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
                            except TypeError:
                                use_reentrant_ckpt = True
                                model.gradient_checkpointing_enable()
                            ckpt_active = True
                            print(f"  [{model_name}] Auto-enabled gradient checkpointing after CUDA OOM to protect remaining fold.", flush=True)
                        continue

                    running_loss += loss_val
                    n_micro += 1
                    n_total_micros_executed += 1
                    n_b = int(batch_labels.numel())
                    ep_samples += n_b
                    if not (epoch == 0 and micro_step == 0):
                        steady_samples_executed += n_b
                        ep_steady_samples += n_b
                    accumulated_steps += 1

                    if accumulated_steps == grad_accum:
                        if use_amp:
                            scaler.unscale_(optimizer)
                            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                            old_scale = scaler.get_scale()
                            scaler.step(optimizer)
                            scaler.update()
                            if scaler.get_scale() >= old_scale:
                                scheduler.step()
                        else:
                            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                            optimizer.step()
                            scheduler.step()
                        optimizer.zero_grad(set_to_none=True)
                        accumulated_steps = 0

                    if micro_step == 0 and epoch == 0:
                        t_step1_done = time.perf_counter()
                        first_step_overhead = t_step1_done - t_batch_start

                    log_interval = max(n_total_micros // 5, 25) if n_total_micros >= 20 else n_total_micros
                    if (micro_step + 1) % log_interval == 0 or (micro_step + 1) == n_total_micros:
                        if t_step1_done is not None and micro_step > 0:
                            step_elapsed = max(time.perf_counter() - t_step1_done, 0.001)
                            step_sps = ep_steady_samples / step_elapsed
                        else:
                            step_elapsed = max(time.perf_counter() - t_batch_start, 0.001)
                            step_sps = ep_samples / step_elapsed
                        step_loss = running_loss / max(n_micro, 1)
                        pct = ((micro_step + 1) / n_total_micros) * 100.0
                        print(f"  [{model_name}] Ep {epoch+1}/{epochs} | Step {micro_step+1}/{n_total_micros} ({pct:3.0f}%) | Loss: {step_loss:.4f} | Speed: {step_sps:5.2f} s/s {get_mem_status(dev)}", flush=True)
        finally:
            prefetcher.close()

        pure_train_sec += (time.perf_counter() - t_batch_start)

        if accumulated_steps > 0:
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
            else:
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                optimizer.step()
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            accumulated_steps = 0

        avg_loss = running_loss / max(n_micro, 1)

        if stop_early and n_total_micros_executed == 0:
            print(f"  [{model_name}] Time Guard triggered before any gradient step in Ep {epoch+1}; skipping untrained fold eval.", flush=True)
            break

        val_logits = evaluate_transformer_split(model, enc_va, tokenizer, dev, use_amp, len(labels))
        val_preds = np.argmax(val_logits, axis=1)
        val_score = balanced_accuracy_score(y_va_int, val_preds)
        print(f"  [{model_name}] Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f} | Val Bal Acc: {val_score:.4f} {get_mem_status(dev)}", flush=True)

        if val_score > best_val_score:
            best_val_score = val_score
            best_val_logits = val_logits
            best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if k in trainable_names}

        # Collect trainable states for Stochastic Weight Averaging (SWA) over the final converged epochs
        if epoch >= max(epochs - 2, 0):
            epoch_states.append({k: v.detach().cpu().clone() for k, v in model.named_parameters() if k in trainable_names and v.is_floating_point()})

        if stop_early:
            break

    train_dur = time.perf_counter() - t_train_start
    if n_total_micros_executed > 1 and steady_samples_executed > 0 and first_step_overhead > 0.0:
        steady_sec = max(pure_train_sec - first_step_overhead, 0.001)
        pure_throughput = steady_samples_executed / steady_sec
    else:
        pure_throughput = (len(texts_tr) * max(epoch + 1, 1)) / max(pure_train_sec, 0.01)
    train_transformer_fold.last_throughput = pure_throughput
    if "deberta" in model_name.lower():
        train_deberta_fold.last_throughput = pure_throughput
    elif "bge" in model_name.lower():
        train_bge_fold.last_throughput = pure_throughput
    print(f"  [{model_name}] Training finished: {train_dur:.1f}s ({pure_throughput:.2f} samples/sec step throughput). Best Val Bal Acc: {best_val_score:.4f}", flush=True)

    if len(epoch_states) >= 2:
        try:
            swa_w = torch.linspace(0.5, 1.0, len(epoch_states))
            swa_w = swa_w / swa_w.sum()
            swa_state = {}
            for k in epoch_states[0]:
                acc = epoch_states[0][k] * float(swa_w[0])
                for idx_s in range(1, len(epoch_states)):
                    acc = acc + epoch_states[idx_s][k] * float(swa_w[idx_s])
                swa_state[k] = acc
            model.load_state_dict({k: v.to(dev) for k, v in swa_state.items()}, strict=False)
            swa_val_logits = evaluate_transformer_split(model, enc_va, tokenizer, dev, use_amp, len(labels))
            swa_val_score = balanced_accuracy_score(y_va_int, np.argmax(swa_val_logits, axis=1))
            print(f"  [{model_name}] SWA Checkpoint Averaged ({len(epoch_states)} epochs) | Val Bal Acc: {swa_val_score:.4f} (vs Best Single {best_val_score:.4f})", flush=True)
            if swa_val_score >= best_val_score:
                best_val_score = swa_val_score
                best_val_logits = swa_val_logits
                best_state = swa_state
            elif best_state is not None:
                model.load_state_dict({k: v.to(dev) for k, v in best_state.items()}, strict=False)
        except Exception as e:
            print(f"  [{model_name}] SWA averaging fallback: {e}", flush=True)
            if best_state is not None:
                model.load_state_dict({k: v.to(dev) for k, v in best_state.items()}, strict=False)
    elif best_state is not None:
        model.load_state_dict({k: v.to(dev) for k, v in best_state.items()}, strict=False)

    test_logits = evaluate_transformer_split(model, enc_te, tokenizer, dev, use_amp, len(labels), enc_data_tta=cached_enc_te_tta) if best_val_logits is not None else None

    set_model_pooler_mask(model, None)
    del model, optimizer, scheduler, epoch_states
    if "swa_state" in locals():
        del swa_state
    if best_state is not None:
        del best_state
    gc.collect()
    if cuda_ok and str(dev).startswith("cuda"):
        dev_idx = torch.device(dev).index if torch.device(dev).index is not None else 0
        with torch.cuda.device(dev_idx):
            torch.cuda.synchronize(dev_idx)
            torch.cuda.empty_cache()

    return best_val_logits, test_logits


def train_transformer_fold(
    model_name,
    texts_tr, y_tr_str, texts_va, y_va_str, texts_te,
    model_dir, labels,
    device="cuda:0",
    batch_size=12,
    grad_accum=1,
    max_length=448,
    epochs=3,
    lr_backbone=1.5e-5,
    lr_head=1.2e-4,
    weight_decay=0.01,
    label_smoothing=0.01,
    layer_decay=0.95,
    enable_grad_checkpointing=True,
    deadline=None,
    cached_enc_tr=None,
    cached_enc_va=None,
    cached_enc_te=None,
    cached_enc_te_tta=None,
    cached_tokenizer=None,
    group_ids_tr=None,
    use_topic_masking=True,
    use_sift=True,
    n_frozen_bottom_layers=4,
):
    """Exception-safe wrapper ensuring GPU memory is reclaimed even if a fold aborts mid-epoch."""
    try:
        return _train_transformer_fold_impl(
            model_name, texts_tr, y_tr_str, texts_va, y_va_str, texts_te, model_dir, labels,
            device=device, batch_size=batch_size, grad_accum=grad_accum, max_length=max_length,
            epochs=epochs, lr_backbone=lr_backbone, lr_head=lr_head, weight_decay=weight_decay,
            label_smoothing=label_smoothing, layer_decay=layer_decay,
            enable_grad_checkpointing=enable_grad_checkpointing, deadline=deadline,
            cached_enc_tr=cached_enc_tr, cached_enc_va=cached_enc_va, cached_enc_te=cached_enc_te,
            cached_enc_te_tta=cached_enc_te_tta, cached_tokenizer=cached_tokenizer,
            group_ids_tr=group_ids_tr, use_topic_masking=use_topic_masking, use_sift=use_sift,
            n_frozen_bottom_layers=n_frozen_bottom_layers,
        )
    finally:
        gc.collect()
        if torch.cuda.is_available() and str(device).startswith("cuda"):
            dev_idx = torch.device(device).index if torch.device(device).index is not None else 0
            with torch.cuda.device(dev_idx):
                torch.cuda.synchronize(dev_idx)
                torch.cuda.empty_cache()

def train_deberta_fold(texts_tr, y_tr_str, texts_va, y_va_str, texts_te, model_dir, labels, group_ids_tr=None, **kwargs):
    kwargs.setdefault("batch_size", 16)
    kwargs.setdefault("grad_accum", 1)
    kwargs.setdefault("epochs", 4)
    kwargs.setdefault("lr_backbone", 1.8e-5)
    kwargs.setdefault("lr_head", 1.5e-4)
    kwargs.setdefault("layer_decay", 0.90)
    kwargs.setdefault("enable_grad_checkpointing", True)
    return train_transformer_fold(
        "DeBERTa FT", texts_tr, y_tr_str, texts_va, y_va_str, texts_te, model_dir, labels,
        group_ids_tr=group_ids_tr, **kwargs
    )

# --- 7. BGE-LARGE FULL FINE-TUNE WRAPPER (GPU 1) ---

def train_bge_fold(texts_tr, y_tr_str, texts_va, y_va_str, texts_te, model_dir, labels, group_ids_tr=None, **kwargs):
    kwargs.setdefault("batch_size", 20)
    kwargs.setdefault("grad_accum", 1)
    kwargs.setdefault("epochs", 5)
    kwargs.setdefault("lr_backbone", 2.2e-5)
    kwargs.setdefault("lr_head", 1.2e-4)
    kwargs.setdefault("layer_decay", 0.85)
    kwargs.setdefault("enable_grad_checkpointing", True)
    return train_transformer_fold(
        "BGE FT", texts_tr, y_tr_str, texts_va, y_va_str, texts_te, model_dir, labels,
        group_ids_tr=group_ids_tr, **kwargs
    )


# --- 8. QWEN 3 8B / 2.5 7B LORA FINE-TUNE PER FOLD (DUAL GPU) ---

def _align_device_map_for_module(module: nn.Module, dev_map: dict) -> dict:
    """Align a saved hf_device_map to exact module names (accounting for PEFT base_model prefixes)."""
    if not dev_map or not isinstance(dev_map, dict):
        return {}
    mod_names = {name for name, _ in module.named_modules()}
    aligned = {}
    for k, v in dev_map.items():
        if k in mod_names:
            aligned[k] = v
        elif f"base_model.model.{k}" in mod_names:
            aligned[f"base_model.model.{k}"] = v
        elif f"base_model.{k}" in mod_names:
            aligned[f"base_model.{k}"] = v
        elif k == "":
            aligned[""] = v
    return aligned


def _remove_accelerate_hooks_safe(module: nn.Module) -> None:
    """Strip accelerate device hooks using sub.__dict__ only, bypassing PEFT __getattr__ delegation."""
    try:
        from accelerate.hooks import _accelerate_added_attributes
    except Exception:
        _accelerate_added_attributes = ("to", "cuda", "npu", "xpu", "mlu", "sdaa", "musa")
    for sub in list(module.modules()):
        if "_hf_hook" in sub.__dict__:
            hook = sub.__dict__["_hf_hook"]
            try:
                if hook is not None:
                    hook.detach_hook(sub)
            finally:
                sub.__dict__.pop("_hf_hook", None)
        if "_old_forward" in sub.__dict__:
            old_fwd = sub.__dict__.pop("_old_forward")
            if "GraphModuleImpl" in str(type(sub)):
                sub.__class__.forward = old_fwd
            else:
                sub.forward = old_fwd
        for attr in _accelerate_added_attributes:
            sub.__dict__.pop(attr, None)
        sub.__dict__.pop("hf_device_map", None)


def _offload_qwen_backbone_to_cpu_cache(backbone: nn.Module) -> nn.Module:
    """Safely detach accelerate hooks (PEFT-safe) and move resident backbone to Host RAM."""
    _remove_accelerate_hooks_safe(backbone)
    cpu_backbone = backbone.to("cpu")
    GLOBAL_RAM_MODEL_CACHE["qwen_backbone"] = cpu_backbone
    return cpu_backbone


def _train_qwen_lora_fold_impl(
    texts_tr, y_tr_str, texts_va, y_va_str, texts_te,
    model_dir, labels,
    batch_size=16,
    grad_accum=1,
    max_length=448,
    epochs=3,
    lr_lora=1.5e-4,
    lr_head=3.5e-4,
    lora_r=48,
    lora_alpha=96,
    lora_dropout=0.08,
    deadline=None,
    cached_enc_tr=None,
    cached_enc_va=None,
    cached_enc_te=None,
    cached_enc_te_tta=None,
    cached_tokenizer=None,
    group_ids_tr=None,
    retain_backbone: bool = False,
):
    """
    LoRA fine-tune of Qwen 3 8B / 2.5 7B across dual GPUs (device_map="auto").
    Supports zero-disk resident backbone reuse via Host RAM caching & PCIe staging.
    Context length aligned to 448 tokens (covering 98.7% of all passage lengths).
    LoRA rank 48 with alpha 96 targeting all linear projections (~120M trainable parameters).
    FastBatchPrefetcher with host RAM pinned transfers for zero GPU bubbles.
    Upfront RAM token cache reuse (eliminates redundant tokenizations across folds).
    Returns (val_logits, test_logits) each (N, 18) float64.
    """
    try:
        from peft import LoraConfig, get_peft_model, TaskType
    except ImportError:
        print("  [Qwen LoRA] peft not installed -> skipping Qwen stream.", flush=True)
        return None, None

    label2id = {l: i for i, l in enumerate(labels)}
    texts_tr = list(texts_tr)
    texts_va = list(texts_va)
    texts_te = list(texts_te)
    y_tr_int = np.array([label2id[y] for y in y_tr_str], dtype=np.int64)
    y_va_int = np.array([label2id[y] for y in y_va_str], dtype=np.int64)

    cuda_ok = torch.cuda.is_available()
    print(f"  [Qwen LoRA] Loading from {model_dir}...", flush=True)
    if cached_tokenizer is not None:
        tokenizer = cached_tokenizer
    else:
        try:
            tokenizer = AutoTokenizer.from_pretrained(
                model_dir, local_files_only=True, trust_remote_code=True, fix_mistral_regex=True
            )
        except TypeError:
            tokenizer = AutoTokenizer.from_pretrained(
                model_dir, local_files_only=True, trust_remote_code=True
            )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"   # decoder-style: pad left

    n_gpus = torch.cuda.device_count() if cuda_ok else 0
    qwen_desc = "Qwen 3 8B" if ("qwen-3" in str(model_dir).lower() or "qwen3" in str(model_dir).lower()) else "Qwen 2.5 7B"
    if "qwen_backbone" in GLOBAL_RAM_MODEL_CACHE:
        print(f"  [Qwen LoRA] Loading resident backbone from Host RAM cache (0 disk I/O, fast PCIe transfer)...", flush=True)
        backbone = GLOBAL_RAM_MODEL_CACHE["qwen_backbone"]
        dev_map = GLOBAL_RAM_MODEL_CACHE.get("qwen_device_map")
        if n_gpus >= 2:
            try:
                from accelerate import dispatch_model, infer_auto_device_map
                aligned_map = _align_device_map_for_module(backbone, dev_map) if dev_map is not None else {}
                if not aligned_map:
                    aligned_map = infer_auto_device_map(
                        backbone, no_split_module_classes=getattr(backbone, "_no_split_modules", None)
                    )
                backbone = dispatch_model(backbone, device_map=aligned_map)
            except Exception as e:
                print(f"  [Qwen LoRA] accelerate dispatch notice: {e}; recomputing balanced auto map", flush=True)
                try:
                    from accelerate import dispatch_model, infer_auto_device_map
                    auto_map = infer_auto_device_map(
                        backbone, no_split_module_classes=getattr(backbone, "_no_split_modules", None)
                    )
                    backbone = dispatch_model(backbone, device_map=auto_map)
                except Exception as e2:
                    raise RuntimeError(f"Dual-GPU Qwen dispatch failed across {n_gpus} GPUs: {e2}") from e2
        elif cuda_ok:
            backbone = backbone.to("cuda:0")

        # Reset LoRA adapter parameters in 0.01s (clean Kaiming/zeros re-initialization)
        for name, module in backbone.named_modules():
            if hasattr(module, "reset_lora_parameters"):
                try:
                    module.reset_lora_parameters("default", init_lora_weights=True)
                except TypeError:
                    try:
                        module.reset_lora_parameters("default")
                    except TypeError:
                        module.reset_lora_parameters()
        print("  [Qwen LoRA] Reset LoRA adapter weights to initial distribution in 0.01s [OK]", flush=True)
    else:
        with HeartbeatTimer(f"Reading {qwen_desc} weights and sharding across Dual GPUs"):
            kw = {"local_files_only": True, "trust_remote_code": True}
            if n_gpus >= 2:
                print(f"  [Qwen LoRA] device_map=auto across {n_gpus} GPUs", flush=True)
                kw["device_map"] = "auto"
            elif cuda_ok:
                kw["device_map"] = {"": "cuda:0"}

            try:
                backbone = AutoModel.from_pretrained(model_dir, torch_dtype=torch.float16, **kw)
            except TypeError:
                backbone = AutoModel.from_pretrained(model_dir, dtype=torch.float16, **kw)

        if hasattr(backbone, "enable_input_require_grads"):
            backbone.enable_input_require_grads()
        try:
            backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            backbone.gradient_checkpointing_enable()
        backbone.config.use_cache = False

        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_r, lora_alpha=lora_alpha,
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=lora_dropout, bias="none",
        )
        backbone = get_peft_model(backbone, lora_cfg)
        backbone.print_trainable_parameters()
        if hasattr(backbone, "hf_device_map") and backbone.hf_device_map:
            GLOBAL_RAM_MODEL_CACHE["qwen_device_map"] = copy.deepcopy(
                _align_device_map_for_module(backbone, backbone.hf_device_map)
            )

    first_dev = next(backbone.parameters()).device
    last_dev = first_dev
    if hasattr(backbone, "hf_device_map") and backbone.hf_device_map:
        for k in ("model.norm", "norm", "base_model.model.norm"):
            if k in backbone.hf_device_map:
                dev_val = backbone.hf_device_map[k]
                last_dev = torch.device(f"cuda:{dev_val}" if isinstance(dev_val, int) else dev_val)
                break
        else:
            int_devs = [v for v in backbone.hf_device_map.values() if isinstance(v, int)]
            if int_devs:
                last_dev = torch.device(f"cuda:{max(int_devs)}")
    else:
        try:
            base_m = getattr(backbone, "base_model", backbone)
            inner_m = getattr(base_m, "model", base_m)
            norm_layer = getattr(inner_m, "norm", None)
            if norm_layer is not None and hasattr(norm_layer, "weight"):
                last_dev = norm_layer.weight.device
            else:
                last_dev = list(backbone.parameters())[-1].device
        except Exception:
            last_dev = first_dev

    head_dev = last_dev if cuda_ok else torch.device("cpu")
    hidden_size = backbone.config.hidden_size
    head = SpikeAndGapHierarchicalHead(hidden_size, len(labels), sub_centers=3).to(head_dev)

    lora_params = [p for _, p in backbone.named_parameters() if p.requires_grad]
    for p in lora_params:
        if p.dtype != torch.float32:
            p.data = p.data.to(torch.float32)
    head_params = list(head.parameters())
    all_trainable_params = lora_params + head_params
    head_decay = [p for p in head_params if p.ndim >= 2]
    head_no_decay = [p for p in head_params if p.ndim < 2]
    optimizer = AdamW(
        [
            {"params": lora_params, "lr": lr_lora, "weight_decay": 0.01},
            {"params": head_decay, "lr": lr_head, "weight_decay": 0.01},
            {"params": head_no_decay, "lr": lr_head, "weight_decay": 0.0},
        ],
    )
    cw = compute_smoothed_class_weights(y_tr_int, num_classes=len(labels), gamma=0.5)
    weight_tensor = torch.tensor(cw, dtype=torch.float32, device=head_dev)
    criterion = FocalLoss(weight=weight_tensor, gamma=1.0, label_smoothing=0.01)
    family_criterion = nn.CrossEntropyLoss().to(head_dev)
    FAMILY_AUX_WEIGHT = 0.25
    label_to_fam_tensor = torch.tensor(LABEL_TO_FAMILY_IDX, dtype=torch.long, device=head_dev)

    n_opt_steps_per_epoch = max((len(texts_tr) + batch_size * grad_accum - 1) // (batch_size * grad_accum), 1)
    total_steps = n_opt_steps_per_epoch * epochs
    scheduler = get_cosine_schedule_with_warmup_and_min_lr(
        optimizer,
        num_warmup_steps=max(int(0.1 * total_steps), 1),
        num_training_steps=total_steps,
        min_lr_ratio=0.10,
    )

    if cached_enc_tr is not None:
        enc_tr = cached_enc_tr
        enc_va = cached_enc_va
        enc_te = cached_enc_te
        print(f"  [Qwen LoRA] Using RAM-cached token encodings ({len(enc_tr['input_ids'])} train / {len(enc_va['input_ids'])} val / {len(enc_te['input_ids'])} test passages).", flush=True)
    else:
        print(f"  [Qwen LoRA] Tokenizing {len(texts_tr)} train / {len(texts_va)} val / {len(texts_te)} test passages (Head+Tail L={max_length}, native BPE newlines preserved)...", flush=True)
        enc_tr = tokenize_with_head_tail(tokenizer, texts_tr, max_length=max_length, preserve_newlines=False)
        enc_va = tokenize_with_head_tail(tokenizer, texts_va, max_length=max_length, preserve_newlines=False)
        enc_te = tokenize_with_head_tail(tokenizer, texts_te, max_length=max_length, preserve_newlines=False)

    scaler = torch.cuda.amp.GradScaler(enabled=cuda_ok, init_scale=4096.0)

    best_val_score = -1.0
    best_val_logits = None
    best_lora_state = None
    best_head_state = None
    epoch_lora_states = []
    epoch_head_states = []
    tr_lengths = np.array([len(x) for x in enc_tr["input_ids"]])
    n_total_micros = (len(texts_tr) + batch_size - 1) // batch_size
    mega_size = batch_size * 8
    t_qwen_train_start = time.perf_counter()
    pure_qwen_sec = 0.0
    first_step_overhead = 0.0
    n_total_micros_executed = 0
    steady_samples_executed = 0
    stop_early = False
    n_skipped_micros = 0
    n_oom_skips = 0

    for epoch in range(epochs):
        t_qwen_ep_start = time.perf_counter()
        t_step1_done = None
        backbone.train()
        head.train()
        if group_ids_tr is not None:
            epoch_indices = make_prompt_sibling_paired_indices(group_ids_tr, y_tr_int, tr_lengths, batch_size, np.random.RandomState(42 + epoch))
        else:
            perm = np.random.RandomState(42 + epoch).permutation(len(texts_tr))
            sorted_indices = []
            for m_start in range(0, len(texts_tr), mega_size):
                m_chunk = perm[m_start:m_start + mega_size]
                sorted_indices.extend(m_chunk[np.argsort(tr_lengths[m_chunk])])
            epoch_indices = np.array(sorted_indices, dtype=np.int64)

        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0
        n_batches = 0
        accumulated_steps = 0
        ep_samples = 0
        ep_steady_samples = 0

        prefetcher = FastBatchPrefetcher(
            tokenizer, enc_tr, y_tr_int, epoch_indices, batch_size,
            dev=first_dev, label_dev=head_dev, cuda_ok=cuda_ok, group_ids=group_ids_tr
        )
        try:
            for micro_step, (batch_kwargs, batch_labels) in enumerate(prefetcher):
                if deadline is not None and (micro_step % 10 == 0) and time.perf_counter() > deadline:
                    print(f"  [Qwen LoRA] [Time Guard] Deadline reached mid-epoch at step {micro_step}/{n_total_micros}. Stopping training.", flush=True)
                    stop_early = True
                    break

                try:
                    batch_groups = batch_kwargs.pop("_batch_groups", getattr(prefetcher, "current_groups", None))
                    autocast_ctx = amp_autocast() if cuda_ok else contextlib.nullcontext()
                    with autocast_ctx:
                        out = backbone(**batch_kwargs, output_hidden_states=False)
                        h_input = getattr(out, "last_hidden_state", out[0] if isinstance(out, tuple) else out)
                        del out
                        if isinstance(head, SpikeAndGapHierarchicalHead):
                            logits = head(h_input, batch_kwargs["attention_mask"])
                        else:
                            last_h = h_input[0] if isinstance(h_input, (tuple, list)) else h_input
                            if getattr(tokenizer, "padding_side", "left") == "left":
                                pooled = last_h[:, -1, :].float()
                            else:
                                attn_mask = batch_kwargs["attention_mask"].to(last_h.device)
                                seq_lens = torch.clamp(attn_mask.sum(dim=1) - 1, min=0)
                                batch_idx = torch.arange(last_h.size(0), device=last_h.device)
                                pooled = last_h[batch_idx, seq_lens].float()
                            logits = head(pooled.to(head_dev))
                        del h_input
                    logits_f = logits.float()
                    main_loss = criterion(logits_f, batch_labels)
                    fam_logits = compute_family_logits(logits_f, temp=4.0)
                    batch_fam_labels = label_to_fam_tensor[batch_labels]
                    aux_loss = family_criterion(fam_logits, batch_fam_labels)
                    loss = (main_loss + FAMILY_AUX_WEIGHT * aux_loss) / grad_accum
                    if batch_groups is not None:
                        repulsion_loss = compute_same_prompt_sibling_repulsion(logits_f, batch_groups, batch_labels, margin=0.15)
                        loss = loss + (0.04 * repulsion_loss) / grad_accum

                    if cuda_ok:
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()

                    loss_val = loss.item() * grad_accum
                    if math.isnan(loss_val) or math.isinf(loss_val):
                        n_skipped_micros += 1
                        print(f"  [Qwen LoRA] Warning: NaN/Inf loss at micro-step {micro_step}. Skipping ({n_skipped_micros} total skips).", flush=True)
                        if n_skipped_micros > 0.15 * max(n_total_micros * (epoch + 1), 1):
                            raise FloatingPointError(f"[Qwen LoRA] >15% micro-batches ({n_skipped_micros}) produced NaN/Inf loss. Discarding fold.")
                        optimizer.zero_grad(set_to_none=True)
                        accumulated_steps = 0
                        continue

                except torch.cuda.OutOfMemoryError:
                    n_oom_skips += 1
                    print(f"  [Qwen LoRA] CUDA OOM at micro-step {micro_step} (OOM #{n_oom_skips}). Resetting partial grads...", flush=True)
                    if n_oom_skips > 5:
                        raise RuntimeError("[Qwen LoRA] Exceeded 5 CUDA OOM errors in fold; aborting to prevent data starvation.")
                    out = h_input = logits = loss = batch_kwargs = batch_labels = None
                    optimizer.zero_grad(set_to_none=True)
                    accumulated_steps = 0
                    if cuda_ok:
                        for gi in (first_dev, head_dev):
                            if str(gi).startswith("cuda"):
                                dev_idx = torch.device(gi).index if torch.device(gi).index is not None else 0
                                with torch.cuda.device(dev_idx):
                                    torch.cuda.synchronize(dev_idx)
                                    torch.cuda.empty_cache()
                    continue

                epoch_loss += loss_val
                n_batches += 1
                n_total_micros_executed += 1
                n_b = int(batch_labels.numel())
                ep_samples += n_b
                if not (epoch == 0 and micro_step == 0):
                    steady_samples_executed += n_b
                    ep_steady_samples += n_b
                accumulated_steps += 1

                if accumulated_steps == grad_accum:
                    if cuda_ok:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(all_trainable_params, max_norm=1.0)
                        old_scale = scaler.get_scale()
                        scaler.step(optimizer)
                        scaler.update()
                        if scaler.get_scale() >= old_scale:
                            scheduler.step()
                    else:
                        torch.nn.utils.clip_grad_norm_(all_trainable_params, max_norm=1.0)
                        optimizer.step()
                        scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                    accumulated_steps = 0

                if micro_step == 0 and epoch == 0:
                    t_step1_done = time.perf_counter()
                    first_step_overhead = t_step1_done - t_qwen_ep_start

                log_interval = max(n_total_micros // 5, 25) if n_total_micros >= 20 else n_total_micros
                if (micro_step + 1) % log_interval == 0 or (micro_step + 1) == n_total_micros:
                    if t_step1_done is not None and micro_step > 0:
                        step_elapsed = max(time.perf_counter() - t_step1_done, 0.001)
                        step_sps = ep_steady_samples / step_elapsed
                    else:
                        step_elapsed = max(time.perf_counter() - t_qwen_ep_start, 0.001)
                        step_sps = ep_samples / step_elapsed
                    step_loss = epoch_loss / max(n_batches, 1)
                    pct = ((micro_step + 1) / n_total_micros) * 100.0
                    print(f"  [Qwen LoRA] Ep {epoch+1}/{epochs} | Step {micro_step+1}/{n_total_micros} ({pct:3.0f}%) | Loss: {step_loss:.4f} | Speed: {step_sps:5.2f} s/s {get_mem_status()}", flush=True)
        finally:
            prefetcher.close()

        pure_qwen_sec += (time.perf_counter() - t_qwen_ep_start)

        if accumulated_steps > 0:
            if cuda_ok:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(all_trainable_params, max_norm=1.0)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
            else:
                torch.nn.utils.clip_grad_norm_(all_trainable_params, max_norm=1.0)
                optimizer.step()
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            accumulated_steps = 0

        avg_loss = epoch_loss / max(n_batches, 1)
        if stop_early and n_total_micros_executed == 0:
            print(f"  [Qwen LoRA] Time Guard triggered before any gradient step in Ep {epoch+1}; skipping untrained fold eval.", flush=True)
            break
        val_logits = evaluate_qwen_split(backbone, head, enc_va, tokenizer, first_dev, head_dev, cuda_ok, len(labels))
        val_preds = np.argmax(val_logits, axis=1)
        val_score = balanced_accuracy_score(y_va_int, val_preds)
        print(f"  [Qwen LoRA] Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f} | Val Bal Acc: {val_score:.4f} {get_mem_status()}", flush=True)

        if val_score > best_val_score:
            best_val_score = val_score
            best_val_logits = val_logits
            best_lora_state = {k: v.detach().cpu().clone() for k, v in backbone.named_parameters() if "lora_" in k}
            best_head_state = {k: v.cpu().clone() for k, v in head.state_dict().items()}

        # Collect states for Stochastic Weight Averaging (SWA) over the final converged epochs
        if epoch >= max(epochs - 2, 0):
            epoch_lora_states.append({k: v.detach().cpu().clone() for k, v in backbone.named_parameters() if "lora_" in k and v.is_floating_point()})
            epoch_head_states.append({k: v.cpu().clone() for k, v in head.state_dict().items() if v.is_floating_point()})

        if stop_early:
            break

    qwen_dur = time.perf_counter() - t_qwen_train_start
    if n_total_micros_executed > 1 and steady_samples_executed > 0 and first_step_overhead > 0.0:
        steady_sec = max(pure_qwen_sec - first_step_overhead, 0.001)
        qwen_sps = steady_samples_executed / steady_sec
    else:
        qwen_sps = (len(texts_tr) * max(epoch + 1, 1)) / max(pure_qwen_sec, 0.01)
    train_qwen_lora_fold.last_throughput = qwen_sps
    print(f"  [Qwen LoRA] Training finished: {qwen_dur:.1f}s ({qwen_sps:.2f} samples/sec). Best Val Bal Acc: {best_val_score:.4f}", flush=True)

    if len(epoch_lora_states) >= 2:
        try:
            swa_w = torch.linspace(0.5, 1.0, len(epoch_lora_states))
            swa_w = swa_w / swa_w.sum()
            swa_lora = {}
            for k in epoch_lora_states[0]:
                acc = epoch_lora_states[0][k] * float(swa_w[0])
                for idx_s in range(1, len(epoch_lora_states)):
                    acc = acc + epoch_lora_states[idx_s][k] * float(swa_w[idx_s])
                swa_lora[k] = acc
            swa_head = {}
            for k in epoch_head_states[0]:
                acc = epoch_head_states[0][k] * float(swa_w[0])
                for idx_s in range(1, len(epoch_head_states)):
                    acc = acc + epoch_head_states[idx_s][k] * float(swa_w[idx_s])
                swa_head[k] = acc

            backbone.load_state_dict(swa_lora, strict=False)
            head.load_state_dict(swa_head)
            swa_val_logits = evaluate_qwen_split(backbone, head, enc_va, tokenizer, first_dev, head_dev, cuda_ok, len(labels))
            swa_val_score = balanced_accuracy_score(y_va_int, np.argmax(swa_val_logits, axis=1))
            print(f"  [Qwen LoRA] SWA Checkpoint Averaged ({len(epoch_lora_states)} epochs) | Val Bal Acc: {swa_val_score:.4f} (vs Best Single {best_val_score:.4f})", flush=True)
            if swa_val_score >= best_val_score:
                best_val_score = swa_val_score
                best_val_logits = swa_val_logits
                best_lora_state = swa_lora
                best_head_state = swa_head
            else:
                if best_lora_state is not None:
                    backbone.load_state_dict(best_lora_state, strict=False)
                if best_head_state is not None:
                    head.load_state_dict(best_head_state)
        except Exception as e:
            print(f"  [Qwen LoRA] SWA averaging fallback: {e}", flush=True)
            if best_lora_state is not None:
                backbone.load_state_dict(best_lora_state, strict=False)
            if best_head_state is not None:
                head.load_state_dict(best_head_state)
    else:
        if best_lora_state is not None:
            backbone.load_state_dict(best_lora_state, strict=False)
        if best_head_state is not None:
            head.load_state_dict(best_head_state)

    test_logits = evaluate_qwen_split(backbone, head, enc_te, tokenizer, first_dev, head_dev, cuda_ok, len(labels), enc_data_tta=cached_enc_te_tta) if best_val_logits is not None else None

    del head, optimizer, scheduler, epoch_lora_states, epoch_head_states
    if "swa_lora" in locals():
        del swa_lora
    if "swa_head" in locals():
        del swa_head
    if best_lora_state is not None:
        del best_lora_state
    if best_head_state is not None:
        del best_head_state

    if retain_backbone:
        # Offload backbone to host RAM so GPU 0 and GPU 1 are 100% free for DeBERTa/BGE next fold!
        # Host RAM has >20 GB free; moving Qwen (15.2 GB) to CPU RAM takes ~1.3s via PCIe
        # and eliminates 180s Kaggle FUSE network disk re-read!
        try:
            _offload_qwen_backbone_to_cpu_cache(backbone)
            print("  [Qwen LoRA] Stripped device hooks & offloaded resident backbone to Host RAM (Dual GPUs 100% free)", flush=True)
        except Exception as e:
            print(f"  [Qwen LoRA] Host RAM offload fallback ({e}); releasing backbone.", flush=True)
            GLOBAL_RAM_MODEL_CACHE.pop("qwen_backbone", None)
            GLOBAL_RAM_MODEL_CACHE.pop("qwen_device_map", None)
        del backbone
    else:
        _remove_accelerate_hooks_safe(backbone)
        GLOBAL_RAM_MODEL_CACHE.pop("qwen_backbone", None)
        GLOBAL_RAM_MODEL_CACHE.pop("qwen_device_map", None)
        del backbone

    gc.collect()
    if cuda_ok:
        for dev_idx in range(torch.cuda.device_count()):
            with torch.cuda.device(dev_idx):
                torch.cuda.synchronize(dev_idx)
                torch.cuda.empty_cache()

    return best_val_logits, test_logits


def train_qwen_lora_fold(
    texts_tr, y_tr_str, texts_va, y_va_str, texts_te,
    model_dir, labels,
    batch_size=16,
    grad_accum=1,
    max_length=448,
    epochs=3,
    lr_lora=1.5e-4,
    lr_head=3.5e-4,
    lora_r=48,
    lora_alpha=96,
    lora_dropout=0.08,
    deadline=None,
    cached_enc_tr=None,
    cached_enc_va=None,
    cached_enc_te=None,
    cached_enc_te_tta=None,
    cached_tokenizer=None,
    group_ids_tr=None,
    retain_backbone: bool = False,
):
    """Exception-safe wrapper guaranteeing Qwen hooks & GPU allocations are cleanly offloaded/freed on abort."""
    try:
        return _train_qwen_lora_fold_impl(
            texts_tr, y_tr_str, texts_va, y_va_str, texts_te, model_dir, labels,
            batch_size=batch_size, grad_accum=grad_accum, max_length=max_length, epochs=epochs,
            lr_lora=lr_lora, lr_head=lr_head, lora_r=lora_r, lora_alpha=lora_alpha,
            lora_dropout=lora_dropout, deadline=deadline, cached_enc_tr=cached_enc_tr,
            cached_enc_va=cached_enc_va, cached_enc_te=cached_enc_te,
            cached_enc_te_tta=cached_enc_te_tta, cached_tokenizer=cached_tokenizer,
            group_ids_tr=group_ids_tr, retain_backbone=retain_backbone,
        )
    except Exception:
        if retain_backbone and "qwen_backbone" in GLOBAL_RAM_MODEL_CACHE:
            try:
                _offload_qwen_backbone_to_cpu_cache(GLOBAL_RAM_MODEL_CACHE["qwen_backbone"])
            except Exception:
                GLOBAL_RAM_MODEL_CACHE.pop("qwen_backbone", None)
                GLOBAL_RAM_MODEL_CACHE.pop("qwen_device_map", None)
        else:
            GLOBAL_RAM_MODEL_CACHE.pop("qwen_backbone", None)
            GLOBAL_RAM_MODEL_CACHE.pop("qwen_device_map", None)
        raise
    finally:
        gc.collect()
        if torch.cuda.is_available():
            for dev_idx in range(torch.cuda.device_count()):
                with torch.cuda.device(dev_idx):
                    torch.cuda.synchronize(dev_idx)
                    torch.cuda.empty_cache()

# --- 9. CONSENSUS BLENDING, CALIBRATION & SINKHORN BALANCING ---

def fit_temperature(margins: np.ndarray, y_true_int: np.ndarray, default_t: float = 1.0) -> float:
    """
    Fits optimal Platt/temperature scaling parameter T by minimizing NLL on valid OOF margins.
    Searches over [0.05, 8.0] using numerical log-sum-exp; guaranteed non-zero and stable.
    """
    if margins is None:
        return default_t
    valid = np.isfinite(margins).all(axis=1) & (y_true_int >= 0)
    if np.sum(valid) < 20:
        return default_t
    m_val = margins[valid]
    y_val = y_true_int[valid]

    best_t = default_t
    best_nll = float("inf")
    for t_cand in np.geomspace(0.05, 8.0, 80):
        scaled = m_val / t_cand
        max_s = np.max(scaled, axis=1, keepdims=True)
        exp_s = np.exp(scaled - max_s)
        sum_exp = np.sum(exp_s, axis=1, keepdims=True)
        log_probs = (scaled - max_s) - np.log(sum_exp + 1e-12)
        nll = -np.mean(log_probs[np.arange(len(y_val)), y_val])
        if nll < best_nll:
            best_nll = nll
            best_t = float(t_cand)
    return round(best_t, 3)


def margins_to_probs(margins: np.ndarray, temp: float = 1.0) -> np.ndarray:
    """
    Convert unbounded margins/logits into calibrated probability distribution.
    Hardened against NaNs: any row containing non-finite values is replaced with a uniform prior (1/K).
    """
    m = np.asarray(margins, dtype=np.float64)
    bad = ~np.isfinite(m).all(axis=1)
    m = np.nan_to_num(m, nan=0.0, posinf=50.0, neginf=-50.0)
    z = np.clip(m, -50.0, 50.0) / max(temp, 1e-4)
    z -= z.max(axis=1, keepdims=True)
    p = np.exp(z)
    p /= np.maximum(p.sum(axis=1, keepdims=True), 1e-12)
    if np.any(bad):
        p[bad] = 1.0 / p.shape[1]
    return p


def sinkhorn_knopp_balance(
    probs: np.ndarray, max_iters: int = 50, tol: float = 1e-3
) -> np.ndarray:
    """
    Sinkhorn-Knopp matrix balancing directly on probability distribution.
    Guarantees uniform column sums (N/K) while maintaining row stochasticity.
    Uses relative tolerance convergence check.
    Automatically converts raw margins to calibrated probabilities if negative values are present.
    """
    if np.any(probs < 0.0) or not np.all(np.isfinite(probs)):
        probs = margins_to_probs(probs)
    assert np.all(np.isfinite(probs)), "Non-finite probabilities reached Sinkhorn balancing"
    N, K = probs.shape
    P = np.copy(probs) + 1e-12
    target_col = np.full(K, N / K, dtype=np.float64)
    r = np.ones(N, dtype=np.float64)
    c = np.ones(K, dtype=np.float64)

    for _ in range(max_iters):
        c_prev = c.copy()
        c = target_col / (np.dot(r, P) + 1e-12)
        r = 1.0 / (np.dot(P, c) + 1e-12)
        rel_diff = np.max(np.abs(c - c_prev)) / (np.max(np.abs(c)) + 1e-12)
        if rel_diff < tol:
            break

    P_bal = r[:, None] * P * c[None, :]
    return P_bal / np.maximum(np.sum(P_bal, axis=1, keepdims=True), 1e-12)


def hierarchical_tree_ensemble(
    logits_qwen: np.ndarray = None,
    logits_cpu: np.ndarray = None,
    logits_deberta: np.ndarray = None,
    logits_bge: np.ndarray = None,
    router_weights=(0.35, 0.30, 0.20, 0.15),  # (Qwen, CPU, DeBERTa, BGE) for Stage 1 Family
    expert_weights=(0.55, 0.25, 0.15, 0.05),  # (Qwen, CPU, DeBERTa, BGE) for Stage 2 Sibling
    hierarchy_alpha: float = 0.65,             # Blend between flat and hierarchical (prevents hard cliff)
    family_groups: dict = None,
) -> np.ndarray:
    """
    Differentiable Bayesian Hierarchical Neural-Tree Probability Fusion:
      P(Class_i) = P_router(Family_k) * P_expert(Class_i | Family_k)

    Stage 1 Router (router_weights) isolates coarse macro-family regions (>92% family accuracy).
    Stage 2 Sibling Expert (expert_weights) isolates conditional probabilities strictly within family siblings.
    Smooth interpolation with flat expert distribution via hierarchy_alpha prevents hard-decision cliffs.
    Uses canonical FAMILY_TO_INDICES by default to prevent label index scrambling across families.
    Gracefully handles None streams (skips missing models and renormalizes weights).
    """
    groups = family_groups if family_groups is not None else FAMILY_TO_INDICES
    models = [logits_qwen, logits_cpu, logits_deberta, logits_bge]
    first_m = next((m for m in models if m is not None), None)
    if first_m is None:
        raise ValueError("At least one model stream must be provided to hierarchical_tree_ensemble.")

    N = first_m.shape[0]
    C = first_m.shape[1]

    w_r = [w if m is not None else 0.0 for w, m in zip(router_weights, models)]
    w_e = [w if m is not None else 0.0 for w, m in zip(expert_weights, models)]
    sum_r = max(sum(w_r), 1e-6)
    sum_e = max(sum(w_e), 1e-6)
    w_r = [w / sum_r for w in w_r]
    w_e = [w / sum_e for w in w_e]

    z_router = np.zeros((N, C), dtype=np.float64)
    z_expert = np.zeros((N, C), dtype=np.float64)

    for m, wr, we in zip(models, w_r, w_e):
        if m is not None:
            z_router += wr * m
            z_expert += we * m

    p_router_18 = margins_to_probs(z_router)
    p_hierarchical = np.zeros((N, C), dtype=np.float64)

    for fam_name, idxs in groups.items():
        p_family = np.sum(p_router_18[:, idxs], axis=-1, keepdims=True)
        if len(idxs) > 1:
            p_sibling_given_fam = margins_to_probs(z_expert[:, idxs])
        else:
            p_sibling_given_fam = np.ones((N, 1), dtype=np.float64)
        p_hierarchical[:, idxs] = p_family * p_sibling_given_fam

    p_flat = margins_to_probs(z_expert)
    p_final = hierarchy_alpha * p_hierarchical + (1.0 - hierarchy_alpha) * p_flat
    return p_final / np.maximum(np.sum(p_final, axis=-1, keepdims=True), 1e-12)


def compute_hierarchical_tree_probs(
    p_svm: np.ndarray,
    p_deb: np.ndarray = None,
    p_qwen: np.ndarray = None,
    p_bge: np.ndarray = None,
    router_weights=(0.30, 0.25, 0.30, 0.15),
    expert_weights=(1.0, 1.0, 1.0, 1.0),
    hierarchy_alpha: float = 0.65,
) -> np.ndarray:
    """
    Differentiable Bayesian Hierarchical Neural-Tree Probability Fusion:
      P(Class_i) = P_router(Family_k) * P_expert(Class_i | Family_k)
    Stage 1 Router (router_weights) isolates coarse macro-family regions (>92% family accuracy).
    Stage 2 Sibling Expert (expert_weights) isolates conditional probabilities strictly within family siblings.
    Smooth interpolation with flat expert distribution via hierarchy_alpha prevents hard-decision cliffs.
    """
    models = [p_svm, p_deb, p_qwen, p_bge]
    w_r = [w if m is not None else 0.0 for w, m in zip(router_weights, models)]
    w_e = [w if m is not None else 0.0 for w, m in zip(expert_weights, models)]
    sum_r = max(sum(w_r), 1e-6)
    sum_e = max(sum(w_e), 1e-6)
    w_r = [w / sum_r for w in w_r]
    w_e = [w / sum_e for w in w_e]

    ref_m = next((m for m in models if m is not None), None)
    if ref_m is None:
        return np.zeros((0, NUM_CLASSES), dtype=np.float64)
    N = ref_m.shape[0]
    p_router = np.zeros((N, NUM_CLASSES), dtype=np.float64)
    p_expert = np.zeros((N, NUM_CLASSES), dtype=np.float64)

    for m, wr, we in zip(models, w_r, w_e):
        if m is not None:
            p_router += wr * m
            p_expert += we * m

    p_router = p_router / np.maximum(np.sum(p_router, axis=-1, keepdims=True), 1e-12)
    p_expert = p_expert / np.maximum(np.sum(p_expert, axis=-1, keepdims=True), 1e-12)

    if hierarchy_alpha <= 0.0:
        return p_expert

    p_hierarchical = np.zeros((N, NUM_CLASSES), dtype=np.float64)
    for fam, idxs in FAMILY_TO_INDICES.items():
        # Stage 1: Router probability mass allocated to Family k
        p_family = np.sum(p_router[:, idxs], axis=-1, keepdims=True)
        # Stage 2: Sibling conditional probability strictly within Family k
        if len(idxs) > 1:
            sub_mass = np.sum(p_expert[:, idxs], axis=-1, keepdims=True)
            p_sibling = p_expert[:, idxs] / np.maximum(sub_mass, 1e-12)
        else:
            p_sibling = np.ones((N, 1), dtype=np.float64)
        p_hierarchical[:, idxs] = p_family * p_sibling

    p_final = hierarchy_alpha * p_hierarchical + (1.0 - hierarchy_alpha) * p_expert
    return p_final / np.maximum(np.sum(p_final, axis=-1, keepdims=True), 1e-12)


def quad_rank_blend(
    svm_m: np.ndarray = None,
    deberta_m: np.ndarray = None,
    qwen_m: np.ndarray = None,
    bge_m: np.ndarray = None,
    weights=(1.0, 1.0, 1.0, 1.0),
    temperatures=(0.20, 1.2, 1.5, 1.2),
    balance_test: bool = False,
    apply_family_gating: bool = True,
    family_alpha: float = 0.65,
    router_weights=(0.30, 0.25, 0.30, 0.15),
    return_probs: bool = False,
):
    """
    Calibrated Probability Consensus across 4 model streams with Hierarchical Soft-Tree Fusion.
    Stage 1 Router (router_weights) + Stage 2 Sibling Expert (weights).
    Blends smooth probability distributions, applies hierarchical tree routing, and balances test priors via Sinkhorn.
    SVM margins use temp=0.20 to match transformer logit dynamic range and entropy.
    """
    ts, td, tq, tb = temperatures
    p_svm = margins_to_probs(svm_m, temp=ts) if svm_m is not None else None
    p_deb = margins_to_probs(deberta_m, temp=td) if deberta_m is not None else None
    p_qwen = margins_to_probs(qwen_m, temp=tq) if qwen_m is not None else None
    p_bge = margins_to_probs(bge_m, temp=tb) if bge_m is not None else None

    probs = compute_hierarchical_tree_probs(
        p_svm=p_svm,
        p_deb=p_deb,
        p_qwen=p_qwen,
        p_bge=p_bge,
        router_weights=router_weights,
        expert_weights=weights,
        hierarchy_alpha=family_alpha if apply_family_gating else 0.0,
    )

    if balance_test and len(probs) >= 36:
        probs = sinkhorn_knopp_balance(probs)

    if return_probs:
        return probs
    return np.argmax(probs, axis=1)


def optimize_blend_weights(oof_svm, oof_deb, oof_qwen, oof_bge, y_true, temperatures=(0.20, 1.2, 1.5, 1.2)):
    """
    Grid search over expert weight candidates and hierarchical tree alpha maximising OOF Balanced Accuracy on valid rows only.
    Tests hierarchy_alpha in [0.0, 0.20, 0.35, 0.50, 0.65, 0.75, 0.85] with guaranteed monotonic improvement over flat blend.
    """
    best_w = (
        1.0 if oof_svm is not None else 0.0,
        1.0 if oof_deb is not None else 0.0,
        1.0 if oof_qwen is not None else 0.0,
        1.0 if oof_bge is not None else 0.0,
    )
    best_alpha = 0.65
    best_score = -1.0

    streams = [s for s in (oof_svm, oof_deb, oof_bge, oof_qwen) if s is not None]
    if not streams:
        return best_w, best_alpha, 0.0
    valid_mask = np.ones(len(streams[0]), dtype=bool)
    for s in streams:
        valid_mask &= ~np.isnan(s).any(axis=1)

    if not np.any(valid_mask):
        return best_w, best_alpha, 0.0

    label2id = {l: i for i, l in enumerate(LABELS)}
    y_true_int = np.array([label2id[y] if isinstance(y, str) else int(y) for y in y_true], dtype=np.int64)
    y_val = y_true_int[valid_mask]

    ts, td, tq, tb = temperatures
    svm_p = margins_to_probs(oof_svm[valid_mask], temp=ts) if oof_svm is not None else None
    deb_p = margins_to_probs(oof_deb[valid_mask], temp=td) if oof_deb is not None else None
    qwn_p = margins_to_probs(oof_qwen[valid_mask], temp=tq) if oof_qwen is not None else None
    bge_p = margins_to_probs(oof_bge[valid_mask], temp=tb) if oof_bge is not None else None

    s_cands = [0.2, 0.5, 0.8, 1.0, 1.2, 1.5] if svm_p is not None else [0.0]
    d_cands = [0.0, 0.2, 0.5, 0.8, 1.0, 1.2, 1.5] if deb_p is not None else [0.0]
    q_cands = [0.0, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5] if qwn_p is not None else [0.0]
    b_cands = [0.0, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0] if bge_p is not None else [0.0]

    # Precompute invariant Stage 1 Router family probabilities across the entire grid search
    r_weights = (0.30, 0.25, 0.30, 0.15)
    r_models = [svm_p, deb_p, qwn_p, bge_p]
    ref_p = next(m for m in r_models if m is not None)
    w_r_eff = [w if m is not None else 0.0 for w, m in zip(r_weights, r_models)]
    sum_r = max(sum(w_r_eff), 1e-6)
    p_router = np.zeros_like(ref_p)
    for m, wr in zip(r_models, w_r_eff):
        if m is not None:
            p_router += (wr / sum_r) * m
    p_router /= np.maximum(p_router.sum(axis=-1, keepdims=True), 1e-12)
    p_family_dict = {fam: np.sum(p_router[:, idxs], axis=-1, keepdims=True) for fam, idxs in FAMILY_TO_INDICES.items()}
    N_val, K_cls = ref_p.shape
    do_sinkhorn = N_val >= 36

    def _fast_bal_acc(y_true_arr, pred_arr, safe_counts_arr, n_present_cls):
        hits = np.bincount(y_true_arr[y_true_arr == pred_arr], minlength=K_cls)
        return float(np.sum(hits / safe_counts_arr) / n_present_cls)

    raw_counts = np.bincount(y_val, minlength=K_cls).astype(np.float64)
    present = raw_counts > 0
    n_present = int(np.sum(present))
    safe_counts = np.where(present, raw_counts, 1.0)

    # Preallocate reusable buffers to eliminate heap churn across 2,352 grid steps
    p_expert = np.empty_like(ref_p)
    p_hier = np.empty_like(ref_p)
    blend_p = np.empty_like(ref_p)
    target_col = np.full(K_cls, N_val / K_cls, dtype=np.float64)
    ones_N = np.ones(N_val, dtype=np.float64)
    multi_fams = [(idxs, p_family_dict[fam]) for fam, idxs in FAMILY_TO_INDICES.items() if len(idxs) > 1]
    single_fams = [(idxs, p_family_dict[fam]) for fam, idxs in FAMILY_TO_INDICES.items() if len(idxs) == 1]

    for ws in s_cands:
        for wd in d_cands:
            for wq in q_cands:
                for wb in b_cands:
                    if ws + wd + wq + wb <= 0.0:
                        continue
                    cand_weights = (ws, wd, wq, wb)
                    w_e_eff = [w if m is not None else 0.0 for w, m in zip(cand_weights, r_models)]
                    sum_e = max(sum(w_e_eff), 1e-6)
                    p_expert.fill(0.0)
                    for m, we in zip(r_models, w_e_eff):
                        if m is not None and we > 0.0:
                            p_expert += (we / sum_e) * m
                    row_s = p_expert.sum(axis=-1, keepdims=True)
                    np.maximum(row_s, 1e-12, out=row_s)
                    p_expert /= row_s

                    for idxs, p_fam in multi_fams:
                        sub = p_expert[:, idxs]
                        sub_mass = np.maximum(sub.sum(axis=-1, keepdims=True), 1e-12)
                        p_hier[:, idxs] = p_fam * (sub / sub_mass)
                    for idxs, p_fam in single_fams:
                        p_hier[:, idxs] = p_fam

                    np.multiply(p_hier, 0.65, out=blend_p)
                    blend_p += 0.35 * p_expert
                    b_s = blend_p.sum(axis=-1, keepdims=True)
                    np.maximum(b_s, 1e-12, out=b_s)
                    blend_p /= b_s

                    if do_sinkhorn:
                        blend_p += 1e-12
                        r = ones_N.copy()
                        c = np.ones(K_cls, dtype=np.float64)
                        for _ in range(20):
                            c_prev = c.copy()
                            c = target_col / (np.dot(r, blend_p) + 1e-12)
                            r = 1.0 / (np.dot(blend_p, c) + 1e-12)
                            if np.max(np.abs(c - c_prev)) / (np.max(np.abs(c)) + 1e-12) < 2e-3:
                                break
                        blend_p *= r[:, None]
                        blend_p *= c[None, :]
                        b_s2 = blend_p.sum(axis=-1, keepdims=True)
                        np.maximum(b_s2, 1e-12, out=b_s2)
                        blend_p /= b_s2

                    pred_idx = np.argmax(blend_p, axis=1)
                    score = _fast_bal_acc(y_val, pred_idx, safe_counts, n_present)
                    if score > best_score:
                        best_score = score
                        best_w = cand_weights

    # Fine-tune hierarchical tree alpha on the optimal expert weights
    best_score_final = -1.0
    for cand_alpha in [0.0, 0.20, 0.35, 0.50, 0.65, 0.75, 0.85]:
        tree_p = compute_hierarchical_tree_probs(
            p_svm=svm_p, p_deb=deb_p, p_qwen=qwn_p, p_bge=bge_p,
            router_weights=r_weights,
            expert_weights=best_w,
            hierarchy_alpha=cand_alpha,
        )
        if do_sinkhorn:
            tree_p = sinkhorn_knopp_balance(tree_p, max_iters=50, tol=1e-3)
        pred_idx = np.argmax(tree_p, axis=1)
        score = _fast_bal_acc(y_val, pred_idx, safe_counts, n_present)
        if score > best_score_final:
            best_score_final = score
            best_alpha = cand_alpha
    best_score = best_score_final

    return best_w, best_alpha, best_score


def optimize_cpu_subblend(sparse_probs: np.ndarray, dense_probs: np.ndarray, y_true_int: np.ndarray) -> float:
    """
    Find optimal alpha such that alpha*sparse + (1-alpha)*dense maximizes OOF balanced accuracy.
    Searches over alpha in [0.5, 1.0] anchored at pure sparse (1.0) to preserve sparse dominance.
    """
    valid = np.isfinite(sparse_probs).all(axis=1) & np.isfinite(dense_probs).all(axis=1)
    if np.sum(valid) < 10:
        return 0.85
    base_score = balanced_accuracy_score(y_true_int[valid], sparse_probs[valid].argmax(axis=1))
    best_score = base_score
    best_alpha = 1.0
    for alpha in np.arange(0.95, 0.45, -0.05):
        blend = alpha * sparse_probs[valid] + (1.0 - alpha) * dense_probs[valid]
        score = balanced_accuracy_score(y_true_int[valid], blend.argmax(axis=1))
        if score > best_score:
            best_score = score
            best_alpha = float(alpha)
    return round(best_alpha, 2)


def calibrate_logits(logits: np.ndarray, temp: float = 1.0) -> np.ndarray:
    """Applies temperature scaling to raw model logits or margins."""
    if logits is None:
        return None
    return logits / max(float(temp), 1e-4)


def bag_and_match_logit_dispersion(test_folds_list: list[np.ndarray], oof_reference: np.ndarray) -> np.ndarray | None:
    """
    Averages test logits across folds and rescales row-wise standard deviation to match
    the single-fold OOF reference distribution. Prevents 5-fold averaging from artificially
    inflating test-time Shannon entropy inside CPUGradientMetaCombiner's confidence gates.
    """
    if not test_folds_list:
        return None
    valid_folds = [f for f in test_folds_list if f is not None]
    if not valid_folds:
        return None
    # Zero-center each fold's rows before averaging to remove fold-level constant offsets
    centered_folds = [f - np.mean(f, axis=1, keepdims=True) for f in valid_folds]
    bagged = np.mean(centered_folds, axis=0)

    if oof_reference is not None:
        valid = np.isfinite(oof_reference).all(axis=1)
        if np.sum(valid) >= 20:
            oof_row_std = float(np.mean(np.std(oof_reference[valid], axis=1)))
            test_row_std = float(np.mean(np.std(bagged, axis=1)))
            if test_row_std > 1e-6 and oof_row_std > 1e-6:
                scale = np.clip(oof_row_std / test_row_std, 0.85, 1.35)
                bagged = bagged * scale
    return bagged


class TransductivePromptCenterer:
    """
    Precomputes the top-K lexical prompt neighbor graph once from sparse TF-IDF (excluding dense style columns)
    and tunes the centering strength eta on OOF to guarantee monotonic accuracy gain before applying to test.
    """
    def __init__(self, k_neighbors: int = 17, sim_threshold: float = 0.22):
        self.k_neighbors = k_neighbors
        self.sim_threshold = sim_threshold
        self.nbr_lists_ = None
        self.nbr_weights_ = None

    def fit_graph(self, X_sparse):
        if X_sparse is None or X_sparse.shape[0] <= self.k_neighbors:
            return self
        try:
            # Exclude the last 145 dense stylometric columns so similarity reflects pure prompt topic
            X_lex = X_sparse[:, :-145] if X_sparse.shape[1] > 145 else X_sparse
            X_lex = X_lex.astype(np.float32, copy=False)
            row_norms = np.sqrt(np.asarray(X_lex.power(2).sum(axis=1)).ravel())
            row_norms = np.maximum(row_norms, 1e-12).astype(np.float32)
            X_norm = sp.diags(1.0 / row_norms, dtype=np.float32) @ X_lex

            n = X_norm.shape[0]
            if sp.issparse(X_norm) and X_norm.shape[1] > 500:
                col_df = np.bincount(X_norm.indices, minlength=X_norm.shape[1])
                max_df = max(int(0.40 * n), 2)
                active_cols = np.flatnonzero((col_df >= 2) & (col_df <= max_df))
                if len(active_cols) > 0:
                    X_norm = X_norm[:, active_cols]

            S = (X_norm @ X_norm.T).toarray()
            np.fill_diagonal(S, -1.0)

            k = min(self.k_neighbors, n - 1)
            top_k_idx = np.argpartition(-S, k, axis=1)[:, :k]

            nbr_idx_mat = np.zeros((n, k), dtype=np.int64)
            nbr_w_mat = np.zeros((n, k), dtype=np.float64)
            has_nbrs = np.zeros(n, dtype=bool)
            self.nbr_lists_ = []
            self.nbr_weights_ = []
            for i in range(n):
                nbrs = top_k_idx[i]
                sims = S[i, nbrs]
                valid = sims >= self.sim_threshold
                n_valid = int(np.sum(valid))
                if n_valid >= 3:
                    v_nbrs = nbrs[valid]
                    w = sims[valid].astype(np.float64)
                    w_norm = w / max(float(w.sum()), 1e-12)
                    self.nbr_lists_.append(v_nbrs)
                    self.nbr_weights_.append(w_norm[:, None])
                    nbr_idx_mat[i, :n_valid] = v_nbrs
                    nbr_w_mat[i, :n_valid] = w_norm
                    has_nbrs[i] = True
                else:
                    self.nbr_lists_.append(None)
                    self.nbr_weights_.append(None)
            self._nbr_idx_mat = nbr_idx_mat
            self._nbr_w_mat = nbr_w_mat
            self._has_nbrs = has_nbrs
        except Exception as e:
            print(f"  [Prompt Centerer] Graph fit notice: {e}", flush=True)
            self.nbr_lists_ = None
            self.nbr_weights_ = None
            self._nbr_idx_mat = None
            self._nbr_w_mat = None
            self._has_nbrs = None
        return self

    def transform(self, logits: np.ndarray, eta: float = 0.15) -> np.ndarray:
        if logits is None or self.nbr_lists_ is None or eta <= 0.0 or len(logits) != len(self.nbr_lists_):
            return logits
        try:
            valid_rows = np.isfinite(logits).all(axis=1)
            z_centered = logits.copy()
            z_centered[valid_rows] -= np.mean(logits[valid_rows], axis=1, keepdims=True)
            out = z_centered.copy()

            if getattr(self, "_nbr_idx_mat", None) is not None and np.all(valid_rows):
                active = self._has_nbrs
                if np.any(active):
                    nbr_z = z_centered[self._nbr_idx_mat[active]]              # [Na, k, C]
                    nbr_mean = np.einsum("nkc,nk->nc", nbr_z, self._nbr_w_mat[active])
                    out[active] = z_centered[active] - eta * nbr_mean
                return out

            for i in range(len(logits)):
                if not valid_rows[i] or self.nbr_lists_[i] is None:
                    continue
                nbrs = self.nbr_lists_[i]
                v_mask = valid_rows[nbrs]
                if np.sum(v_mask) >= 3:
                    w = self.nbr_weights_[i][v_mask]
                    w = w / w.sum()
                    nbr_mean = np.sum(z_centered[nbrs[v_mask]] * w, axis=0)
                    out[i] = z_centered[i] - eta * nbr_mean
            return out
        except Exception as e:
            print(f"  [Prompt Centerer] Transform notice: {e}", flush=True)
            return logits

    def optimize_eta(self, oof_logits: np.ndarray | list, y_true_int: np.ndarray, candidates=(0.0, 0.06, 0.12, 0.18, 0.25)) -> float:
        """
        Tunes eta across candidate values to maximize out-of-fold balanced accuracy.
        Candidate 0.0 guarantees monotonic non-decreasing performance.
        Accepts either a single stream (np.ndarray) or list of streams.
        """
        if self.nbr_lists_ is None or oof_logits is None:
            return 0.0
        ref_stream = oof_logits[0] if isinstance(oof_logits, list) else oof_logits
        if ref_stream is None or len(ref_stream) != len(self.nbr_lists_):
            return 0.0
        valid_y = np.isfinite(ref_stream).all(axis=1)
        if np.sum(valid_y) < 5:
            return 0.0
        best_eta = 0.0
        best_score = float(balanced_accuracy_score(y_true_int[valid_y], np.argmax(ref_stream[valid_y], axis=1)))
        for cand_eta in candidates:
            if cand_eta <= 0.0:
                continue
            cand_m = self.transform(ref_stream, eta=cand_eta)
            score = float(balanced_accuracy_score(y_true_int[valid_y], np.argmax(cand_m[valid_y], axis=1)))
            if score > best_score:
                best_score = score
                best_eta = float(cand_eta)
        return best_eta


class CPUGradientMetaCombiner:
    """
    PyTorch CPU Autograd L-BFGS Meta-Combiner.
    Learns optimal per-model, per-class specialization weights W[M, C],
    per-model, per-family router weights W_rot[M, 6],
    sample-level entropy confidence gates, and Bayesian hierarchical tree fusion.
    Executes in under 2 seconds on CPU with exact analytical autograd.
    """
    def __init__(self, n_models: int = 4, n_classes: int = 18, l2_reg: float = 0.05, family_groups=None):
        self.M = n_models
        self.C = n_classes
        self.l2_reg = l2_reg
        self.family_groups = family_groups if family_groups is not None else [
            FAMILY_TO_INDICES[fam] for fam in FAMILIES
        ]
        self.W_exp = None
        self.W_rot = None
        self.gamma = None
        self.alpha_raw = None
        self.valid_indices = None

    @staticmethod
    def _compute_entropy(probs: torch.Tensor) -> torch.Tensor:
        p = probs.clamp(min=1e-12)
        ent = -torch.sum(p * torch.log(p), dim=-1, keepdim=True)
        return ent / math.log(max(probs.shape[-1], 2))

    def _forward(self, logits_stack: torch.Tensor) -> torch.Tensor:
        # logits_stack: [M, N, C]
        M, N, C = logits_stack.shape
        probs_m = F.softmax(logits_stack, dim=-1)
        ent_m = self._compute_entropy(probs_m)  # [M, N, 1]

        g = torch.exp(torch.clamp(self.gamma, -5.0, 5.0)).view(M, 1, 1)
        conf_gate = torch.exp(-g * ent_m)
        gated = logits_stack * conf_gate  # [M, N, C]

        # 1. Expert logits
        z_expert = torch.sum(self.W_exp.view(M, 1, C) * gated, dim=0)  # [N, C]

        # 2. Router logits
        n_fams = len(self.family_groups)
        fam_logits_m = torch.zeros((M, N, n_fams), dtype=logits_stack.dtype, device=logits_stack.device)
        for f_idx, c_idxs in enumerate(self.family_groups):
            fam_logits_m[:, :, f_idx] = torch.logsumexp(gated[:, :, c_idxs], dim=-1)
        z_router = torch.sum(self.W_rot.view(M, 1, n_fams) * fam_logits_m, dim=0)  # [N, n_fams]
        p_family = F.softmax(z_router, dim=-1)  # [N, n_fams]

        # 3. Hierarchical Bayesian Tree Fusion
        p_tree = torch.zeros_like(z_expert)
        for f_idx, c_idxs in enumerate(self.family_groups):
            p_fam_slice = p_family[:, [f_idx]]
            if len(c_idxs) > 1:
                p_sib = F.softmax(z_expert[:, c_idxs], dim=-1)
            else:
                p_sib = torch.ones((N, 1), dtype=logits_stack.dtype, device=logits_stack.device)
            p_tree[:, c_idxs] = p_fam_slice * p_sib

        p_flat = F.softmax(z_expert, dim=-1)
        alpha = torch.sigmoid(self.alpha_raw)
        p_fused = (alpha * p_tree + (1.0 - alpha) * p_flat).clamp(min=1e-12)
        return p_fused / p_fused.sum(dim=-1, keepdim=True)

    def fit(self, oof_logits_list: list, y_true: np.ndarray, prior_weights: list = None, class_weights: np.ndarray = None, max_iter: int = 45):
        valid_indices = [i for i, l in enumerate(oof_logits_list) if l is not None]
        self.valid_indices = valid_indices
        self.M = len(valid_indices)
        if self.M == 0:
            return self
        logits_stack = torch.tensor(np.stack([oof_logits_list[i] for i in valid_indices], axis=0), dtype=torch.float32)
        y = torch.tensor(y_true, dtype=torch.long)
        N = y.shape[0]

        if prior_weights is None:
            prior_w = torch.full((self.M,), 1.0 / self.M)
        else:
            if len(prior_weights) == len(oof_logits_list):
                pw = [prior_weights[i] for i in valid_indices]
            else:
                pw = [prior_weights[j] for j in range(min(len(prior_weights), self.M))]
            prior_w = torch.tensor(pw, dtype=torch.float32)
            prior_w = prior_w / prior_w.sum()

        n_fams = len(self.family_groups)
        self.W_exp = torch.nn.Parameter(prior_w.view(self.M, 1).repeat(1, self.C))
        self.W_rot = torch.nn.Parameter(torch.full((self.M, n_fams), 1.0 / self.M))
        self.gamma = torch.nn.Parameter(torch.zeros(self.M))
        self.alpha_raw = torch.nn.Parameter(torch.tensor([0.8]))

        exp_target = self.W_exp.data.clone()
        rot_target = self.W_rot.data.clone()

        if class_weights is None:
            counts = np.bincount(y_true, minlength=self.C)
            cw = N / (self.C * np.maximum(counts, 1).astype(np.float32))
            cw_tensor = torch.tensor(cw, dtype=torch.float32)
        else:
            cw_tensor = torch.tensor(class_weights, dtype=torch.float32)
        sample_w = cw_tensor[y]

        optimizer = torch.optim.LBFGS(
            [self.W_exp, self.W_rot, self.gamma, self.alpha_raw],
            max_iter=max_iter, lr=0.15, history_size=10
        )

        arange_N = torch.arange(N)

        def closure():
            optimizer.zero_grad()
            probs = self._forward(logits_stack)
            nll = -torch.mean(sample_w * torch.log(probs[arange_N, y]))
            reg = self.l2_reg * (
                torch.sum((self.W_exp - exp_target) ** 2) +
                torch.sum((self.W_rot - rot_target) ** 2) +
                torch.sum(self.gamma ** 2)
            )
            loss = nll + reg
            loss.backward()
            return loss

        optimizer.step(closure)
        return self

    def fit_nested_cv(
        self,
        oof_logits_list: list,
        y_true: np.ndarray,
        groups: np.ndarray = None,
        prior_weights: list = None,
        class_weights: np.ndarray = None,
        n_splits: int = 5,
        max_iter: int = 35,
    ) -> tuple[np.ndarray, float]:
        """
        Nested Prompt-Grouped Cross-Validation.
        Fits meta-combiner across n_splits prompt groups, producing true out-of-sample CV probabilities.
        Averages parameter tensors across folds for bagged test inference (predict_proba).
        Returns (oof_cv_probs, oof_cv_score).
        """
        valid_indices = [i for i, l in enumerate(oof_logits_list) if l is not None]
        self.valid_indices = valid_indices
        self.M = len(valid_indices)
        N = len(y_true)
        if self.M == 0 or N == 0:
            return np.ones((N, self.C)) / self.C, 0.0

        n_unique_groups = len(np.unique(groups)) if groups is not None else 0
        if groups is not None and n_unique_groups >= n_splits:
            cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
            splits = list(cv.split(np.arange(N), y_true, groups=groups))
        else:
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
            splits = list(cv.split(np.arange(N), y_true))

        cv_oof_probs = np.zeros((N, self.C), dtype=np.float64)
        fold_W_exp, fold_W_rot, fold_gamma, fold_alpha = [], [], [], []

        for tr_idx, va_idx in splits:
            fold_tr_streams = [oof_logits_list[i][tr_idx] if oof_logits_list[i] is not None else None for i in range(len(oof_logits_list))]
            fold_va_streams = [oof_logits_list[i][va_idx] if oof_logits_list[i] is not None else None for i in range(len(oof_logits_list))]
            y_tr_fold = y_true[tr_idx]

            fold_comb = CPUGradientMetaCombiner(
                n_models=self.M, n_classes=self.C, l2_reg=self.l2_reg, family_groups=self.family_groups
            )
            fold_comb.valid_indices = self.valid_indices
            fold_comb.fit(fold_tr_streams, y_tr_fold, prior_weights=prior_weights, class_weights=class_weights, max_iter=max_iter)

            cv_oof_probs[va_idx] = fold_comb.predict_proba(fold_va_streams)

            fold_W_exp.append(fold_comb.W_exp.data.clone())
            fold_W_rot.append(fold_comb.W_rot.data.clone())
            fold_gamma.append(fold_comb.gamma.data.clone())
            fold_alpha.append(fold_comb.alpha_raw.data.clone())

        # Bag parameters across all folds for superior test generalization
        self.W_exp = torch.nn.Parameter(torch.stack(fold_W_exp).mean(dim=0))
        self.W_rot = torch.nn.Parameter(torch.stack(fold_W_rot).mean(dim=0))
        self.gamma = torch.nn.Parameter(torch.stack(fold_gamma).mean(dim=0))
        self.alpha_raw = torch.nn.Parameter(torch.stack(fold_alpha).mean(dim=0))

        if len(cv_oof_probs) >= 36:
            cv_oof_probs_eval = sinkhorn_knopp_balance(cv_oof_probs)
        else:
            cv_oof_probs_eval = cv_oof_probs
        cv_preds = np.argmax(cv_oof_probs_eval, axis=1)
        cv_score = float(balanced_accuracy_score(y_true, cv_preds))
        return cv_oof_probs_eval, cv_score

    def predict_proba(self, test_logits_list: list) -> np.ndarray:
        if self.valid_indices is not None and len(test_logits_list) >= max(self.valid_indices) + 1:
            valid_streams = [test_logits_list[i] for i in self.valid_indices if test_logits_list[i] is not None]
        else:
            valid_streams = [l for l in test_logits_list if l is not None]
        if not valid_streams or self.W_exp is None or len(valid_streams) != self.M:
            fallback_n = next((len(s) for s in test_logits_list if s is not None), 1)
            return np.ones((fallback_n, self.C)) / self.C
        logits_stack = torch.tensor(np.stack(valid_streams, axis=0), dtype=torch.float32)
        with torch.no_grad():
            probs = self._forward(logits_stack)
        return probs.cpu().numpy()

# --- 10. PROMPT DSU & STRATIFIED GROUP K-FOLD ---

class DSU:
    """Disjoint Set Union with iterative two-pass path compression (immune to recursion limits)."""
    def __init__(self, elements):
        self.p = {x: x for x in elements}

    def find(self, x):
        root = x
        while self.p[root] != root:
            root = self.p[root]
        curr = x
        while curr != root:
            nxt = self.p[curr]
            self.p[curr] = root
            curr = nxt
        return root

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.p[ry] = rx


def make_prompt_groups(df):
    """DSU to ensure identical texts across different prompts never cross folds."""
    if "prompt_id" in df.columns:
        raw_pids = df["prompt_id"].tolist()
        prompt_vals = [
            f"__missing_pid_{i}" if (p is None or pd.isna(p) or str(p).strip() == "") else str(p)
            for i, p in enumerate(raw_pids)
        ]
    else:
        prompt_vals = [f"__row_{i}" for i in range(len(df))]
    dsu = DSU(set(prompt_vals))
    seen = {}
    for pid, text in zip(prompt_vals, df["text"]):
        h = hashlib.sha256(" ".join(str(text).split()).encode()).hexdigest()
        if h in seen:
            dsu.union(pid, seen[h])
        else:
            seen[h] = pid
    return np.array([dsu.find(p) for p in prompt_vals])


# --- 11. SUBMISSION VALIDATION ---

def validate_and_save_submission(test_ids, predictions, output_path):
    """Atomic parquet write with schema, null, and integrity validation."""
    if len(predictions) != len(test_ids):
        raise ValueError(f"Row count mismatch: expected {len(test_ids)} IDs, got {len(predictions)} predictions")
    sub = pd.DataFrame({"id": test_ids, "model": predictions})
    if list(sub.columns) != ["id", "model"]:
        raise ValueError("Submission columns must be exactly ['id', 'model']")
    if sub.isna().any().any():
        raise ValueError("Submission contains null values")
    if not set(sub["model"]).issubset(set(LABELS)):
        bad = set(sub["model"]) - set(LABELS)
        raise ValueError(f"Invalid predicted class labels: {bad}")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = output_path.parent / f".tmp_sub_{time.time_ns()}.parquet"
    sub.to_parquet(temp_path, index=False)
    pd.testing.assert_frame_equal(sub, pd.read_parquet(temp_path))
    temp_path.replace(output_path)
    print(f"\nSUCCESS: submission saved to {output_path}", flush=True)
    print(f"Total rows: {len(sub):,}", flush=True)
    print("Class distribution:", flush=True)
    print(sub["model"].value_counts().to_string(), flush=True)
    return sub

# --- 12. MAIN PIPELINE ---

def run_quad_ensemble():
    start_time = time.perf_counter()
    print("=" * 65, flush=True)
    print("NAME THAT LLM: QUAD-MODEL FRONTIER ENSEMBLE (V6 Production Pipeline)", flush=True)
    print("DeBERTa Full FT (GPU0) + BGE Full FT (GPU1) + Qwen LoRA (Dual GPU) + CPU Dual-Linear", flush=True)
    print("=" * 65 + "\n", flush=True)

    data_dir, qwen_dir, deberta_dir, bge_dir, ext_path, working_dir = discover_paths()
    print(flush=True)

    seed_everything(42)

    train_df = pd.read_csv(data_dir / "train.csv", dtype={"id": str, "prompt_id": str})
    test_df = pd.read_csv(data_dir / "test.csv", dtype={"id": str}, keep_default_na=False)
    train_df["text"] = train_df["text"].fillna("")
    test_df["text"] = test_df["text"].fillna("")

    # Preserve 100% raw unaltered text for CPU stylometrics & character TF-IDF
    raw_texts_tr = train_df["text"].tolist()
    raw_texts_te = test_df["text"].tolist()

    tr_lens = train_df["text"].str.len()
    print(f"Train: {len(train_df):,} passages | {train_df['prompt_id'].nunique()} unique prompts (length: mean={tr_lens.mean():.0f}, min={tr_lens.min()}, max={tr_lens.max()} chars)", flush=True)
    print(f"Test:  {len(test_df):,} passages\n", flush=True)

    # Ingest and prepare mass-conserved external data if available
    ext_df = None
    X_ext_all = None
    V_dense_ext = None
    ext_prompt_ids = None

    if ext_path is not None and Path(ext_path).is_file():
        try:
            print(f"  [External Data] Ingesting mass-balanced corpus from {ext_path}...", flush=True)
            comb_ext_df, _ = build_mass_balanced_external_corpus(
                train_df, ext_path, LABELS, ext_mass_ratio=0.55, prefer_multi_model=True
            )
            ext_slice = comb_ext_df.iloc[len(train_df):].reset_index(drop=True)
            del comb_ext_df
            ext_df = ext_slice
            ext_texts = ext_slice["text"].tolist()
            ext_prompt_ids = ext_slice["prompt_id"].to_numpy()
            print(f"  [External Data] Loaded {len(ext_texts):,} passages across {ext_slice['model'].nunique()} classes (Mass ratio=0.55, 0.0% prior skew) [OK]", flush=True)
        except Exception as e:
            print(f"  [External Data] Warning: Failed to load external data: {e}. Falling back to competition data.", flush=True)
            ext_df = None

    n_train = len(train_df)
    texts_test = raw_texts_te
    y_true = train_df["model"].to_numpy()

    # --- Prompt-grouped 5-fold CV split ---
    group_ids = make_prompt_groups(train_df)
    n_splits = 5
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    folds = np.full(n_train, -1, dtype=int)
    for fold_i, (_, val_idx) in enumerate(
        splitter.split(train_df["text"], train_df["model"], group_ids)
    ):
        folds[val_idx] = fold_i

    # --- OOF / test accumulators (NaN initialized to prevent zero-poisoning) ---
    oof_svm = np.full((n_train, NUM_CLASSES), np.nan, dtype=np.float64)
    oof_deb = np.full((n_train, NUM_CLASSES), np.nan, dtype=np.float64) if deberta_dir else None
    oof_bge = np.full((n_train, NUM_CLASSES), np.nan, dtype=np.float64) if bge_dir else None
    oof_qwen = np.full((n_train, NUM_CLASSES), np.nan, dtype=np.float64) if qwen_dir else None

    test_svm_folds = []
    test_deb_folds = []
    test_bge_folds = []
    test_qwen_folds = []

    # Detect GPU configuration
    cuda_ok = torch.cuda.is_available()
    n_gpus = torch.cuda.device_count() if cuda_ok else 0
    deb_dev = "cuda:0" if cuda_ok else "cpu"
    bge_dev = "cuda:1" if n_gpus >= 2 else ("cuda:0" if cuda_ok else "cpu")
    print(f"Hardware setup: CUDA={cuda_ok} | GPUs={n_gpus} | DeBERTa={deb_dev} | BGE={bge_dev} | Qwen=device_map=auto\n", flush=True)

    # ── STEP 0: MULTI-MODEL UPFRONT PRE-TOKENIZATION & RAM CACHING ──
    print("=" * 65, flush=True)
    print("STEP 0: MULTI-MODEL UPFRONT PRE-TOKENIZATION & RAM CACHING", flush=True)
    print("=" * 65, flush=True)
    token_caches = precompute_token_caches(
        raw_texts_tr, raw_texts_te, deberta_dir, bge_dir, qwen_dir,
        raw_texts_ext=None,
        max_length=448,
    )
    print(flush=True)

    # ── STEP 1: CPU MULTIVIEW FEATURE UNION & IMMEDIATE BASELINE SUBMISSION ──
    print("=" * 65, flush=True)
    print("STEP 1: FITTING CPU FEATURE UNION & IMMEDIATE BASELINE SUBMISSION", flush=True)
    print("=" * 65, flush=True)
    print("Train class distribution:")
    print(train_df["model"].value_counts().to_string(), flush=True)

    # Compute dense stylometrics once upfront (reused in both FeatureUnion and dense stream; zero duplicate regex work)
    t_style_start = time.perf_counter()
    print("  Precomputing dense stylometric features upfront across all passages...", flush=True)
    style_extractor = RichStyleFeatures()
    V_dense_all = style_extractor.fit_transform(raw_texts_tr + texts_test)
    n_tr = len(raw_texts_tr)
    print(f"  Dense stylometrics matrix shape: {V_dense_all.shape} computed in {time.perf_counter() - t_style_start:.2f}s [OK]", flush=True)

    V_dense_ext = None
    if ext_df is not None:
        t_ext_style = time.perf_counter()
        print("  Precomputing dense stylometrics on external passages...", flush=True)
        V_dense_ext = style_extractor.transform(ext_texts)
        print(f"  External dense stylometrics: {V_dense_ext.shape} computed in {time.perf_counter() - t_ext_style:.2f}s [OK]", flush=True)

    tfidf_extractor = build_cpu_feature_extractor(style_weight=0.2, include_style=False)
    style_scaler = StandardScaler()
    all_raw_texts = raw_texts_tr + texts_test
    with threadpool_limits(limits=4):
        X_tfidf_all = tfidf_extractor.fit_transform(all_raw_texts)
        V_scaled_all = (0.2 * style_scaler.fit_transform(V_dense_all)).astype(np.float32)
        X_all = sp.hstack([X_tfidf_all, sp.csr_matrix(V_scaled_all)], format="csr")
        del X_tfidf_all, V_scaled_all
        X_train_all = X_all[:n_tr]
        X_test_all = X_all[n_tr:]
        if ext_df is not None:
            print("  [External Data] Transforming external texts via competition-locked TF-IDF FeatureUnion...", flush=True)
            X_tfidf_ext = tfidf_extractor.transform(ext_texts)
            V_scaled_ext = (0.2 * style_scaler.transform(V_dense_ext)).astype(np.float32)
            X_ext_all = sp.hstack([X_tfidf_ext, sp.csr_matrix(V_scaled_ext)], format="csr")
            del X_tfidf_ext, V_scaled_ext
            print(f"  [External Data] Sparse external matrix: {X_ext_all.shape} [OK]", flush=True)
    del all_raw_texts, X_all, style_scaler, style_extractor
    for _, tr in tfidf_extractor.transformer_list:
        if hasattr(tr, "stop_words_"):
            tr.stop_words_ = None
    del tfidf_extractor
    gc.collect()

    # Write immediate baseline submission (guarantees non-empty submission in first 30 seconds via fast L2 SVM)
    pdf_mask_quick = compute_prompt_df_mask(X_train_all, group_ids, min_prompts=3)
    keep_cols_quick = np.flatnonzero(pdf_mask_quick)
    quick_clf = LinearSVC(
        penalty="l2", loss="squared_hinge", C=1.5, dual="auto",
        tol=1e-2, max_iter=400, class_weight="balanced", random_state=42,
    )
    with threadpool_limits(limits=4):
        quick_clf.fit(X_train_all[:, keep_cols_quick], y_true)
        quick_test_m = quick_clf.decision_function(X_test_all[:, keep_cols_quick])
        if quick_test_m.ndim == 1:
            quick_test_m = np.column_stack([-quick_test_m, quick_test_m])
        quick_test_m = align_margins(quick_test_m, list(quick_clf.classes_), LABELS)
    del quick_clf, keep_cols_quick, pdf_mask_quick
    quick_preds_idx = np.argmax(quick_test_m, axis=1)
    baseline_sub_path = working_dir / "submission.parquet"
    validate_and_save_submission(test_df["id"].tolist(), [LABELS[i] for i in quick_preds_idx], baseline_sub_path)
    print("Immediate baseline submission saved successfully (safety guarantee verified).\n", flush=True)

    # ── STEP 2: 5-FOLD PROMPT-GROUPED CROSS-VALIDATION ──
    print("=" * 65, flush=True)
    print("STEP 2: 5-FOLD PROMPT-GROUPED CROSS-VALIDATION", flush=True)
    print("=" * 65, flush=True)

    is_qwen3 = qwen_dir and ("qwen-3" in str(qwen_dir).lower() or "qwen3" in str(qwen_dir).lower())
    N_EPOCHS_DEB = 4
    N_EPOCHS_BGE = 5
    N_EPOCHS_QWEN = 2 if is_qwen3 else 3
    MAX_WALLCLOCK_HOURS = 8.2
    SAFETY_MARGIN_SEC = 1200  # 20 min buffer for final bagging, Sinkhorn, and submission
    pipeline_t0 = SCRIPT_T0
    est_inference_sec = 900.0  # 15 min reserve for final inference, calibration and blending
    pipeline_deadline = SCRIPT_T0 + (MAX_WALLCLOCK_HOURS * 3600) - SAFETY_MARGIN_SEC - est_inference_sec
    model_durations = {"deb": [], "bge": [], "qwen": []}

    for fold in range(n_splits):
        seed_everything(42 + fold)
        fold_start = time.perf_counter()
        elapsed_hours = (time.perf_counter() - pipeline_t0) / 3600.0
        if elapsed_hours > (MAX_WALLCLOCK_HOURS - 1.2):
            print(f"\n[Time Guard] {elapsed_hours:.2f}h elapsed. Starting another fold would risk the 9h limit.", flush=True)
            print(f"[Time Guard] Gracefully stopping at {fold} completed folds to finalize ensemble.", flush=True)
            break

        print(f"\n{'='*50}\nFOLD {fold+1}/{n_splits} (elapsed: {elapsed_hours:.2f}h)\n{'='*50}", flush=True)

        tr_idx = np.where(folds != fold)[0]
        va_idx = np.where(folds == fold)[0]
        texts_tr = [raw_texts_tr[i] for i in tr_idx]
        texts_va = [raw_texts_tr[i] for i in va_idx]
        y_tr_str = y_true[tr_idx].tolist()
        y_va_str = y_true[va_idx].tolist()

        # Slice precomputed token caches from RAM in 0.001s
        deb_cache = token_caches.get("deberta")
        deb_kw = {}
        if deb_cache is not None:
            deb_kw["cached_enc_tr"] = {k: [deb_cache["tr"][k][i] for i in tr_idx] for k in deb_cache["tr"]}
            deb_kw["cached_enc_va"] = {k: [deb_cache["tr"][k][i] for i in va_idx] for k in deb_cache["tr"]}
            deb_kw["cached_enc_te"] = deb_cache["te"]
            deb_kw["cached_enc_te_tta"] = deb_cache.get("te_tta")
            deb_kw["cached_tokenizer"] = deb_cache["tokenizer"]

        bge_cache = token_caches.get("bge")
        bge_kw = {}
        if bge_cache is not None:
            bge_kw["cached_enc_tr"] = {k: [bge_cache["tr"][k][i] for i in tr_idx] for k in bge_cache["tr"]}
            bge_kw["cached_enc_va"] = {k: [bge_cache["tr"][k][i] for i in va_idx] for k in bge_cache["tr"]}
            bge_kw["cached_enc_te"] = bge_cache["te"]
            bge_kw["cached_enc_te_tta"] = bge_cache.get("te_tta")
            bge_kw["cached_tokenizer"] = bge_cache["tokenizer"]

        qwen_cache = token_caches.get("qwen")
        qwen_kw = {}
        if qwen_cache is not None:
            qwen_kw["cached_enc_tr"] = {k: [qwen_cache["tr"][k][i] for i in tr_idx] for k in qwen_cache["tr"]}
            qwen_kw["cached_enc_va"] = {k: [qwen_cache["tr"][k][i] for i in va_idx] for k in qwen_cache["tr"]}
            qwen_kw["cached_enc_te"] = qwen_cache["te"]
            qwen_kw["cached_enc_te_tta"] = qwen_cache.get("te_tta")
            qwen_kw["cached_tokenizer"] = qwen_cache["tokenizer"]

        # Sequential Fine-Tuning: DeBERTa (GPU 0) -> BGE (GPU 1) -> Qwen LoRA (Dual GPU)
        deb_expected = np.mean(model_durations["deb"]) * 1.15 if model_durations["deb"] else 1150.0
        bge_expected = np.mean(model_durations["bge"]) * 1.15 if model_durations["bge"] else 550.0
        run_deb = deberta_dir is not None and (time.perf_counter() + deb_expected < pipeline_deadline)

        # ── 1. DeBERTa-v3-large fine-tune (GPU 0) ───────────
        if run_deb:
            try:
                print(f"\n[Fold {fold+1}] DeBERTa-v3-large full fine-tune ({deb_dev}, BS=16, GA=1, L=448, ckpt=ON)", flush=True)
                m_t0 = time.perf_counter()
                val_deb, test_deb = train_deberta_fold(
                    texts_tr, y_tr_str, texts_va, y_va_str, texts_test,
                    deberta_dir, LABELS,
                    device=deb_dev,
                    batch_size=16, grad_accum=1, max_length=448,
                    epochs=N_EPOCHS_DEB, lr_backbone=1.8e-5, lr_head=1.5e-4, layer_decay=0.90, enable_grad_checkpointing=True,
                    deadline=pipeline_deadline,
                    group_ids_tr=group_ids[tr_idx],
                    **deb_kw,
                )
                if val_deb is not None and np.all(np.isfinite(val_deb)):
                    check_finite_logits(val_deb, f"DeBERTa Fold {fold+1} Val")
                    oof_deb[va_idx] = val_deb
                if test_deb is not None and np.all(np.isfinite(test_deb)):
                    check_finite_logits(test_deb, f"DeBERTa Fold {fold+1} Test")
                    test_deb_folds.append(test_deb)
                if val_deb is not None:
                    model_durations["deb"].append(time.perf_counter() - m_t0)
            except Exception as e:
                import traceback
                print(f"  [DeBERTa FT] Fold {fold+1} error: {e}", flush=True)
                traceback.print_exc()

        gc.collect()
        if cuda_ok:
            for gi in range(n_gpus):
                with torch.cuda.device(gi):
                    torch.cuda.synchronize(gi)
                    torch.cuda.empty_cache()

        # ── 2. BGE-large full fine-tune (GPU 1) ─────────────
        run_bge = bge_dir is not None and (time.perf_counter() + bge_expected < pipeline_deadline)
        if run_bge:
            try:
                print(f"\n[Fold {fold+1}] BGE-large full fine-tune ({bge_dev}, BS=20, GA=1, L=448, ckpt=ON)", flush=True)
                m_t0 = time.perf_counter()
                val_bge, test_bge = train_bge_fold(
                    texts_tr, y_tr_str, texts_va, y_va_str, texts_test,
                    bge_dir, LABELS,
                    device=bge_dev,
                    batch_size=20, grad_accum=1, max_length=448,
                    epochs=N_EPOCHS_BGE, lr_backbone=2.0e-5, lr_head=1.2e-4, layer_decay=0.85, enable_grad_checkpointing=True,
                    deadline=pipeline_deadline,
                    group_ids_tr=group_ids[tr_idx],
                    **bge_kw,
                )
                if val_bge is not None and np.all(np.isfinite(val_bge)):
                    check_finite_logits(val_bge, f"BGE Fold {fold+1} Val")
                    oof_bge[va_idx] = val_bge
                if test_bge is not None and np.all(np.isfinite(test_bge)):
                    check_finite_logits(test_bge, f"BGE Fold {fold+1} Test")
                    test_bge_folds.append(test_bge)
                if val_bge is not None:
                    model_durations["bge"].append(time.perf_counter() - m_t0)
            except Exception as e:
                import traceback
                print(f"  [BGE FT] Fold {fold+1} error: {e}", flush=True)
                traceback.print_exc()

        gc.collect()
        if cuda_ok:
            for gi in range(n_gpus):
                with torch.cuda.device(gi):
                    torch.cuda.synchronize(gi)
                    torch.cuda.empty_cache()

        # ── 3. Qwen LoRA fine-tune (Dual GPU) ───────────
        qwen_bs = 8 if is_qwen3 else 16
        qwen_ga = 2 if is_qwen3 else 1
        qwen_lr_lora = 1.5e-4 if is_qwen3 else 1.2e-4
        qwen_lr_head = 2.2e-4 if is_qwen3 else 1.8e-4
        qwen_desc = "Qwen 3 8B" if is_qwen3 else "Qwen 2.5 7B"
        qwen_expected = np.mean(model_durations["qwen"]) * 1.15 if model_durations["qwen"] else (3300.0 if is_qwen3 else 2400.0)
        if qwen_dir is not None and (time.perf_counter() + qwen_expected < pipeline_deadline):
            try:
                print(f"\n[Fold {fold+1}] {qwen_desc} LoRA fine-tune (device_map=auto, r=48, BS={qwen_bs}, GA={qwen_ga}, epochs={N_EPOCHS_QWEN}, L=448)", flush=True)
                m_t0 = time.perf_counter()
                retain_qwen = False
                val_qwen, test_qwen = train_qwen_lora_fold(
                    texts_tr, y_tr_str, texts_va, y_va_str, texts_test,
                    qwen_dir, LABELS,
                    batch_size=qwen_bs, grad_accum=qwen_ga, max_length=448, epochs=N_EPOCHS_QWEN,
                    lr_lora=qwen_lr_lora, lr_head=qwen_lr_head, lora_r=48, lora_alpha=96, lora_dropout=0.08,
                    deadline=pipeline_deadline,
                    group_ids_tr=group_ids[tr_idx],
                    retain_backbone=retain_qwen,
                    **qwen_kw,
                )
                if val_qwen is not None and np.all(np.isfinite(val_qwen)):
                    check_finite_logits(val_qwen, f"Qwen Fold {fold+1} Val")
                    oof_qwen[va_idx] = val_qwen
                if test_qwen is not None and np.all(np.isfinite(test_qwen)):
                    check_finite_logits(test_qwen, f"Qwen Fold {fold+1} Test")
                    test_qwen_folds.append(test_qwen)
                if val_qwen is not None:
                    model_durations["qwen"].append(time.perf_counter() - m_t0)
            except Exception as e:
                import traceback
                print(f"  [Qwen LoRA] Fold {fold+1} error: {e}", flush=True)
                traceback.print_exc()

        gc.collect()
        if cuda_ok:
            for gi in range(n_gpus):
                with torch.cuda.device(gi):
                    torch.cuda.synchronize()
                    torch.cuda.empty_cache()

        # ── 4. CPU Multiview Dual-Linear (Sparse CSR + PromptDF Filter + Sibling Specialists) ──
        try:
            print(f"\n[Fold {fold+1}] CPU Multiview Dual-Linear (Sparse CSR + PromptDF Filter + Sibling Specialists) {get_mem_status(None)}", flush=True)
            pdf_mask = compute_prompt_df_mask(X_train_all[tr_idx], group_ids[tr_idx], min_prompts=3)
            keep_cols = np.flatnonzero(pdf_mask)
            X_tr_fold = X_train_all[tr_idx][:, keep_cols]
            X_va_fold = X_train_all[va_idx][:, keep_cols]
            X_te_fold = X_test_all[:, keep_cols]

            cpu_clf = DualLinearClassifier(c_svc=1.5, alpha_ridge=1.0, svc_weight=0.70, seed=42 + fold, canonical_labels=LABELS)
            clique_specialist = IntraFamilyCliqueSpecialist(alpha_ridge=1.0, beta_refine=0.25, seed=42 + fold)
            with threadpool_limits(limits=4):
                if X_ext_all is not None:
                    # Bounded-variance mass conservation & regularization invariance (Sum(w) == n_comp)
                    w_comp_fold, w_ext_fold = compute_fold_variance_capped_weights(
                        y_true[tr_idx], ext_df["model"].to_numpy(), ext_df["_n_models"].to_numpy(),
                        n_classes=NUM_CLASSES, ext_mass_ratio=0.55, max_single_ext_weight=0.35,
                    )
                    X_ext_fold = X_ext_all[:, keep_cols]
                    X_tr_comb = sp.vstack([X_tr_fold, X_ext_fold])
                    y_tr_comb = np.concatenate([y_tr_str, ext_df["model"].to_numpy()])
                    w_tr_comb = np.concatenate([w_comp_fold, w_ext_fold])

                    # Domain covariate moment alignment for dense stylometrics
                    V_dense_tr = V_dense_all[tr_idx]
                    V_dense_ext_aligned = align_external_dense_moments(V_dense_tr, V_dense_ext, w_ext_fold)
                    comb_V_fold = np.vstack([V_dense_tr, V_dense_ext_aligned])
                    comb_pids_fold = np.concatenate([group_ids[tr_idx], ext_prompt_ids])

                    subspace_proj = StyleInvariantSubspaceProjector(n_topic_components=16, attenuation=0.75)
                    subspace_proj.fit(comb_V_fold, comb_pids_fold, y_labels=y_tr_comb)
                    V_dense_tr_comb = subspace_proj.transform(comb_V_fold)
                    V_dense_va = subspace_proj.transform(V_dense_all[va_idx])
                    V_dense_te = subspace_proj.transform(V_dense_all[n_tr:])

                    cpu_clf.fit(X_tr_comb, y_tr_comb, sample_weight=w_tr_comb)
                    clique_specialist.fit(X_tr_comb, y_tr_comb, sample_weight=w_tr_comb)
                else:
                    X_tr_comb = X_tr_fold
                    y_tr_comb = y_tr_str
                    w_tr_comb = None
                    subspace_proj = StyleInvariantSubspaceProjector(n_topic_components=16, attenuation=0.75)
                    subspace_proj.fit(V_dense_all[tr_idx], group_ids[tr_idx], y_labels=y_tr_str)
                    V_dense_tr_comb = subspace_proj.transform(V_dense_all[tr_idx])
                    V_dense_va = subspace_proj.transform(V_dense_all[va_idx])
                    V_dense_te = subspace_proj.transform(V_dense_all[n_tr:])
                    cpu_clf.fit(X_tr_comb, y_tr_comb)
                    clique_specialist.fit(X_tr_comb, y_tr_comb)

                val_sparse_m = cpu_clf.decision_function(X_va_fold)
                test_sparse_m = cpu_clf.decision_function(X_te_fold)

                # Intra-family sibling tie-breaking refinement with validation-tuned beta (single delta pass per split)
                y_va_int_fold = np.array([LABEL_TO_IDX[y] for y in y_va_str], dtype=np.int64)
                deltas_va = clique_specialist._compute_sibling_deltas(X_va_fold, val_sparse_m.shape[0], val_sparse_m.shape[1])
                opt_beta = clique_specialist.optimize_beta(val_sparse_m, X_va_fold, y_va_int_fold, precomputed_deltas=deltas_va)
                val_sparse_m = clique_specialist.refine_margins(val_sparse_m, X_va_fold, beta=opt_beta, precomputed_deltas=deltas_va)
                test_sparse_m = clique_specialist.refine_margins(test_sparse_m, X_te_fold, beta=opt_beta)

                # Dense stylometric calibrated stream (strictly leak-free fold projection)
                dense_clf = DenseStylometricClassifier(c=0.2, seed=42 + fold, canonical_labels=LABELS)
                if w_tr_comb is not None:
                    dense_clf.fit(V_dense_tr_comb, y_tr_comb, sample_weight=w_tr_comb)
                else:
                    dense_clf.fit(V_dense_tr_comb, y_tr_comb)
                p_dense_va = dense_clf.predict_proba(V_dense_va)
                p_dense_te = dense_clf.predict_proba(V_dense_te)

                # Calibrated consensus: optimal sparse TF-IDF + scaled dense stylometrics
                p_sparse_va = margins_to_probs(val_sparse_m, temp=0.20)
                p_sparse_te = margins_to_probs(test_sparse_m, temp=0.20)
                cpu_alpha = optimize_cpu_subblend(p_sparse_va, p_dense_va, y_va_int_fold)
                p_cpu_va = cpu_alpha * p_sparse_va + (1.0 - cpu_alpha) * p_dense_va
                p_cpu_te = cpu_alpha * p_sparse_te + (1.0 - cpu_alpha) * p_dense_te
                val_svm_m = 0.20 * np.log(p_cpu_va + 1e-12)
                test_svm_m = 0.20 * np.log(p_cpu_te + 1e-12)
                print(f"  [CPU Linear] Fold {fold+1} optimal sparse/dense alpha={cpu_alpha:.2f}", flush=True)

                check_finite_logits(val_svm_m, f"CPU Linear Fold {fold+1} Val")
                check_finite_logits(test_svm_m, f"CPU Linear Fold {fold+1} Test")

                oof_svm[va_idx] = val_svm_m
                test_svm_folds.append(test_svm_m)
                svm_bal_acc = balanced_accuracy_score(y_va_str, [LABELS[i] for i in np.argmax(val_svm_m, axis=1)])
                print(f"  [CPU Linear] Fold {fold+1} bal_acc={svm_bal_acc*100:.2f}% (PromptDF min3 + Sibling Specialists active)", flush=True)

                del cpu_clf, dense_clf, clique_specialist, subspace_proj, X_tr_fold, X_va_fold, X_te_fold, pdf_mask, keep_cols
                if X_ext_all is not None:
                    del X_ext_fold, X_tr_comb, V_dense_tr_comb, V_dense_ext_aligned, comb_V_fold
                gc.collect()
        except Exception as e:
            import traceback
            print(f"  [CPU Linear] Fold {fold+1} error: {e}", flush=True)
            traceback.print_exc()

        # ── Safe Incremental Submission Checkpoint ────────────
        # Guarantees a fully valid, competition-compliant submission.parquet exists on disk after EVERY fold
        try:
            interim_svm_m = np.mean(test_svm_folds, axis=0) if test_svm_folds else None
            interim_deb_m = np.mean(test_deb_folds, axis=0) if test_deb_folds else None
            interim_bge_m = np.mean(test_bge_folds, axis=0) if test_bge_folds else None
            interim_qwen_m = np.mean(test_qwen_folds, axis=0) if test_qwen_folds else None

            if any(m is not None for m in (interim_svm_m, interim_deb_m, interim_bge_m, interim_qwen_m)):
                interim_weights = (
                    1.0 if interim_svm_m is not None else 0.0,
                    1.0 if interim_deb_m is not None else 0.0,
                    1.0 if interim_qwen_m is not None else 0.0,
                    1.0 if interim_bge_m is not None else 0.0,
                )
                interim_preds_idx = quad_rank_blend(
                    interim_svm_m, interim_deb_m, interim_qwen_m, interim_bge_m,
                    weights=interim_weights,
                    temperatures=(0.20, 1.2, 1.5, 1.2),
                    balance_test=True,
                    apply_family_gating=True,
                )
                validate_and_save_submission(
                    test_df["id"].tolist(),
                    [LABELS[i] for i in interim_preds_idx],
                    working_dir / "submission.parquet",
                )
                print(f"  [Safety Checkpoint] Incremental submission.parquet saved after Fold {fold+1}/{n_splits} [OK]", flush=True)
            else:
                print(f"  [Safety Checkpoint] No completed fold test logits yet at Fold {fold+1}/{n_splits}; retaining Step 1 baseline submission.parquet.", flush=True)
        except Exception as e:
            import traceback
            print(f"  [Safety Checkpoint] Warning during interim submission write: {e}", flush=True)
            traceback.print_exc()

        print(f"\n[Fold {fold+1}] Done in {time.perf_counter()-fold_start:.1f}s {get_mem_status(None)}", flush=True)
        gc.collect()
        if cuda_ok:
            for gi in range(n_gpus):
                with torch.cuda.device(gi):
                    torch.cuda.synchronize()
                    torch.cuda.empty_cache()
                    try:
                        torch.cuda.ipc_collect()
                    except Exception:
                        pass

    # ── STEP 3: OUT-OF-FOLD EVALUATION & BLEND OPTIMIZATION ──
    print("\n" + "=" * 65, flush=True)
    print("STEP 3: OUT-OF-FOLD EVALUATION & BLEND OPTIMIZATION", flush=True)
    print("=" * 65, flush=True)

    eval_mask = ~np.isnan(oof_svm).any(axis=1) if oof_svm is not None else np.zeros(n_train, dtype=bool)
    if np.any(eval_mask) and bool(test_svm_folds):
        y_eval = y_true[eval_mask]
        svm_score = balanced_accuracy_score(y_eval, [LABELS[i] for i in np.argmax(oof_svm[eval_mask], axis=1)])
        print(f"  CPU Dual-Linear: {svm_score*100:.2f}%", flush=True)
    else:
        oof_svm = None
        test_svm_folds = []
        svm_score = 0.0
        print("  CPU Dual-Linear: (skipped / no valid folds)", flush=True)

    # Universal Neural Stream Quality Gate (relative to CPU baseline)
    min_acceptable_score = max(2.0 / NUM_CLASSES, 0.35 * svm_score)

    if oof_deb is not None and np.any(~np.isnan(oof_deb).any(axis=1)) and bool(test_deb_folds):
        deb_mask = ~np.isnan(oof_deb).any(axis=1)
        deb_score = balanced_accuracy_score(y_true[deb_mask], [LABELS[i] for i in np.argmax(oof_deb[deb_mask], axis=1)])
        if deb_score < min_acceptable_score:
            print(f"  [DeBERTa FT] Warning: OOF score ({deb_score*100:.2f}%) below minimum quality threshold ({min_acceptable_score*100:.2f}%). Dropping stream from blend.", flush=True)
            oof_deb = None
            test_deb_folds = []
        else:
            print(f"  DeBERTa FT:      {deb_score*100:.2f}% (folds evaluated: {np.sum(deb_mask)//max(n_train//n_splits, 1)}/{n_splits})", flush=True)
    else:
        oof_deb = None
        test_deb_folds = []
        print("  DeBERTa FT:      (skipped)", flush=True)

    if oof_bge is not None and np.any(~np.isnan(oof_bge).any(axis=1)) and bool(test_bge_folds):
        bge_mask = ~np.isnan(oof_bge).any(axis=1)
        bge_score = balanced_accuracy_score(y_true[bge_mask], [LABELS[i] for i in np.argmax(oof_bge[bge_mask], axis=1)])
        if bge_score < min_acceptable_score:
            print(f"  [BGE FT] Warning: OOF score ({bge_score*100:.2f}%) below minimum quality threshold ({min_acceptable_score*100:.2f}%). Dropping stream from blend.", flush=True)
            oof_bge = None
            test_bge_folds = []
        else:
            print(f"  BGE FT:          {bge_score*100:.2f}% (folds evaluated: {np.sum(bge_mask)//max(n_train//n_splits, 1)}/{n_splits})", flush=True)
    else:
        oof_bge = None
        test_bge_folds = []
        print("  BGE FT:          (skipped)", flush=True)

    if oof_qwen is not None and np.any(~np.isnan(oof_qwen).any(axis=1)) and bool(test_qwen_folds):
        qwen_mask = ~np.isnan(oof_qwen).any(axis=1)
        qwen_score = balanced_accuracy_score(y_true[qwen_mask], [LABELS[i] for i in np.argmax(oof_qwen[qwen_mask], axis=1)])
        if qwen_score < min_acceptable_score:
            print(f"  [Qwen LoRA] Warning: OOF score ({qwen_score*100:.2f}%) below minimum quality threshold ({min_acceptable_score*100:.2f}%). Dropping stream from blend.", flush=True)
            oof_qwen = None
            test_qwen_folds = []
        else:
            print(f"  Qwen LoRA:       {qwen_score*100:.2f}% (folds evaluated: {np.sum(qwen_mask)//max(n_train//n_splits, 1)}/{n_splits})", flush=True)
    else:
        oof_qwen = None
        test_qwen_folds = []
        print("  Qwen LoRA:       (skipped)", flush=True)

    if all(s is None for s in (oof_svm, oof_deb, oof_bge, oof_qwen)):
        print("  [Ensemble Warning] All CV streams produced empty/rejected OOF results; preserving Step 1 baseline submission.parquet.", flush=True)
        elapsed = time.perf_counter() - start_time
        print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.2f} min)", flush=True)
        return

    # Automated Temperature Calibration (minimizing NLL on valid OOF rows)
    lbl2id = {l: i for i, l in enumerate(LABELS)}
    y_true_int = np.array([lbl2id[y] for y in y_true], dtype=np.int64)

    # Preserve pre-centering OOF references for exact apples-to-apples test dispersion matching (tiny <1.5 MB)
    oof_svm_uncentered = oof_svm
    oof_deb_uncentered = oof_deb
    oof_qwen_uncentered = oof_qwen
    oof_bge_uncentered = oof_bge

    # ── Transductive Same-Prompt Neighbor Centering (OOF Graph & Monotonic Tuning) ──
    oof_centerer = TransductivePromptCenterer(k_neighbors=17, sim_threshold=0.22).fit_graph(X_train_all)
    eta_svm = oof_centerer.optimize_eta(oof_svm, y_true_int) if oof_svm is not None else 0.0
    eta_deb = oof_centerer.optimize_eta(oof_deb, y_true_int) if oof_deb is not None else 0.0
    eta_qwn = oof_centerer.optimize_eta(oof_qwen, y_true_int) if oof_qwen is not None else 0.0
    eta_bge = oof_centerer.optimize_eta(oof_bge, y_true_int) if oof_bge is not None else 0.0
    print(f"  [Prompt Centering] Optimal per-stream etas: SVM={eta_svm:.2f} DeBERTa={eta_deb:.2f} Qwen={eta_qwn:.2f} BGE={eta_bge:.2f}", flush=True)
    if oof_svm is not None and eta_svm > 0: oof_svm = oof_centerer.transform(oof_svm, eta=eta_svm)
    if oof_deb is not None and eta_deb > 0: oof_deb = oof_centerer.transform(oof_deb, eta=eta_deb)
    if oof_qwen is not None and eta_qwn > 0: oof_qwen = oof_centerer.transform(oof_qwen, eta=eta_qwn)
    if oof_bge is not None and eta_bge > 0: oof_bge = oof_centerer.transform(oof_bge, eta=eta_bge)

    T_svm = 0.20  # Analytically calibrated to match the 0.20 * log(p_cpu) consensus scaling
    T_deb = fit_temperature(oof_deb, y_true_int, default_t=1.20) if oof_deb is not None else 1.20
    T_qwn = fit_temperature(oof_qwen, y_true_int, default_t=1.50) if oof_qwen is not None else 1.50
    T_bge = fit_temperature(oof_bge, y_true_int, default_t=1.20) if oof_bge is not None else 1.20
    calibrated_temps = (T_svm, T_deb, T_qwn, T_bge)
    print(f"\n  Calibrated Temperatures (NLL-optimal): SVM={T_svm} DeBERTa={T_deb} Qwen={T_qwn} BGE={T_bge}", flush=True)

    blend_score = 0.0
    try:
        best_w, best_alpha, blend_score = optimize_blend_weights(
            oof_svm,
            oof_deb,
            oof_qwen,
            oof_bge,
            y_true,
            temperatures=calibrated_temps,
        )
        ws, wd, wq, wb = best_w
        print(f"\n  Blend weights: CPU={ws} DeBERTa={wd} Qwen={wq} BGE={wb} | Family Alpha={best_alpha:.2f}", flush=True)
        print(f"  OOF QUAD BLEND: {blend_score*100:.2f}%\n", flush=True)
    except Exception as e:
        import traceback
        print(f"  [Blend Tuning] Error during blend optimization: {e}", flush=True)
        traceback.print_exc()
        best_w = (
            1.0 if (oof_svm is not None and test_svm_folds) else 0.0,
            1.0 if (oof_deb is not None and test_deb_folds) else 0.0,
            1.0 if (oof_qwen is not None and test_qwen_folds) else 0.0,
            1.0 if (oof_bge is not None and test_bge_folds) else 0.0,
        )
        best_alpha = 0.35

    # ── CPU Gradient-Optimized Meta-Combiner (L-BFGS Matrix + Dynamic Entropy Gate) ──
    use_meta_combiner = False
    meta_combiner = None
    try:
        raw_oof_streams = [oof_svm, oof_deb, oof_qwen, oof_bge]
        avail_streams = [s for s in raw_oof_streams if s is not None and np.any(~np.isnan(s).any(axis=1))]
        if len(avail_streams) >= 2:
            print("=" * 65, flush=True)
            print("FITTING CPU GRADIENT META-COMBINER (L-BFGS Matrix + Dynamic Entropy Gate)", flush=True)
            calib_oof = [
                calibrate_logits(oof_svm, temp=T_svm) if oof_svm is not None else None,
                calibrate_logits(oof_deb, temp=T_deb) if oof_deb is not None else None,
                calibrate_logits(oof_qwen, temp=T_qwn) if oof_qwen is not None else None,
                calibrate_logits(oof_bge, temp=T_bge) if oof_bge is not None else None,
            ]
            valid_oof_mask = np.ones(n_train, dtype=bool)
            for s in calib_oof:
                if s is not None:
                    valid_oof_mask = valid_oof_mask & ~np.isnan(s).any(axis=1)

            if np.sum(valid_oof_mask) >= 36:
                y_meta = y_true_int[valid_oof_mask]
                groups_meta = group_ids[valid_oof_mask] if group_ids is not None else None
                meta_oof_inputs = [s[valid_oof_mask] if s is not None else None for s in calib_oof]
                meta_combiner = CPUGradientMetaCombiner(n_models=sum(s is not None for s in meta_oof_inputs), n_classes=NUM_CLASSES, l2_reg=0.05)
                meta_priors = [0.25, 0.20, 0.45, 0.10]
                meta_oof_probs, meta_score = meta_combiner.fit_nested_cv(
                    meta_oof_inputs, y_meta, groups=groups_meta, prior_weights=meta_priors, max_iter=30
                )
                print(f"  Nested 5-Fold CV Meta-Combiner OOF: {meta_score*100:.2f}% (vs Flat/Grid: {blend_score*100:.2f}%)", flush=True)
                if meta_score >= blend_score:
                    use_meta_combiner = True
                    print("  -> Meta-Combiner achieved superior out-of-sample OOF score. Selected for test predictions! [OK]\n", flush=True)
                else:
                    use_meta_combiner = False
                    print("  -> Flat/Grid blend achieved superior score; using Quad-Rank blend for test.\n", flush=True)
    except Exception as e:
        print(f"  [Meta-Combiner] Note: {e}. Falling back to Quad-Rank blend.\n", flush=True)
        use_meta_combiner = False

    # Diagnostic: Ridge Stacker Comparison (5-Fold CV out-of-sample)
    try:
        from sklearn.linear_model import RidgeClassifier
        from sklearn.model_selection import cross_val_predict
        # Compute mask directly from raw OOF arrays before margins_to_probs replaces NaNs
        raw_streams = [s for s in (oof_svm, oof_deb, oof_qwen, oof_bge) if s is not None]
        if len(raw_streams) > 1:
            stack_mask = ~np.isnan(raw_streams[0]).any(axis=1)
            for raw_st in raw_streams[1:]:
                stack_mask = stack_mask & ~np.isnan(raw_st).any(axis=1)
            if np.any(stack_mask):
                y_stack = y_true_int[stack_mask]
                avail_probs = []
                if oof_svm is not None: avail_probs.append(margins_to_probs(oof_svm[stack_mask], temp=T_svm))
                if oof_deb is not None: avail_probs.append(margins_to_probs(oof_deb[stack_mask], temp=T_deb))
                if oof_qwen is not None: avail_probs.append(margins_to_probs(oof_qwen[stack_mask], temp=T_qwn))
                if oof_bge is not None: avail_probs.append(margins_to_probs(oof_bge[stack_mask], temp=T_bge))
                X_oof_stack = np.hstack(avail_probs)
                groups_stack = group_ids[stack_mask]
                n_groups_stack = len(np.unique(groups_stack)) if groups_stack is not None else 0
                if n_groups_stack >= 5:
                    cv_stack = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
                    ridge_preds = cross_val_predict(RidgeClassifier(alpha=1.0), X_oof_stack, y_stack, cv=cv_stack, groups=groups_stack)
                else:
                    cv_stack = StratifiedKFold(n_splits=min(5, max(2, len(y_stack) // 18)), shuffle=True, random_state=42)
                    ridge_preds = cross_val_predict(RidgeClassifier(alpha=1.0), X_oof_stack, y_stack, cv=cv_stack)
                ridge_score = balanced_accuracy_score(y_stack, ridge_preds)
                print(f"  OOF RIDGE STACKER (5-Fold CV): {ridge_score*100:.2f}% (out-of-sample diagnostic)\n", flush=True)
    except Exception as e:
        print(f"  [Diagnostic Stacker] Note: {e}", flush=True)

    # ── STEP 4: BAGGED TEST INFERENCE & FINAL SUBMISSION ──
    try:
        print("=" * 65, flush=True)
        print("STEP 4: COMPUTING BAGGED TEST PREDICTIONS & FINAL SUBMISSION", flush=True)
        print("=" * 65, flush=True)
        test_svm_m = bag_and_match_logit_dispersion(test_svm_folds, oof_svm_uncentered) if test_svm_folds else None
        test_deb_m = bag_and_match_logit_dispersion(test_deb_folds, oof_deb_uncentered) if test_deb_folds else None
        test_bge_m = bag_and_match_logit_dispersion(test_bge_folds, oof_bge_uncentered) if test_bge_folds else None
        test_qwen_m = bag_and_match_logit_dispersion(test_qwen_folds, oof_qwen_uncentered) if test_qwen_folds else None

        # Transductive prompt-topic centering on unseen test passages using learned per-stream OOF etas
        test_centerer = TransductivePromptCenterer(k_neighbors=17, sim_threshold=0.22).fit_graph(X_test_all)
        if test_svm_m is not None and eta_svm > 0: test_svm_m = test_centerer.transform(test_svm_m, eta=eta_svm)
        if test_deb_m is not None and eta_deb > 0: test_deb_m = test_centerer.transform(test_deb_m, eta=eta_deb)
        if test_bge_m is not None and eta_bge > 0: test_bge_m = test_centerer.transform(test_bge_m, eta=eta_bge)
        if test_qwen_m is not None and eta_qwn > 0: test_qwen_m = test_centerer.transform(test_qwen_m, eta=eta_qwn)

        if use_meta_combiner and meta_combiner is not None:
            try:
                calib_test = [
                    calibrate_logits(test_svm_m, temp=T_svm) if test_svm_m is not None else None,
                    calibrate_logits(test_deb_m, temp=T_deb) if test_deb_m is not None else None,
                    calibrate_logits(test_qwen_m, temp=T_qwn) if test_qwen_m is not None else None,
                    calibrate_logits(test_bge_m, temp=T_bge) if test_bge_m is not None else None,
                ]
                test_probs = meta_combiner.predict_proba(calib_test)
                if np.allclose(test_probs, 1.0 / NUM_CLASSES, atol=1e-4):
                    raise ValueError("Meta-Combiner produced uniform fallback distribution; switching to Quad-Rank blend.")
                if len(test_probs) >= 36:
                    test_probs = sinkhorn_knopp_balance(test_probs)
                test_pred_idx = np.argmax(test_probs, axis=1)
                print("  [Final Blend] Generated test predictions via CPU Gradient Meta-Combiner + Sinkhorn [OK]", flush=True)
            except Exception as e:
                print(f"  [Final Blend] Meta-Combiner test inference fallback: {e}; using Quad-Rank blend.", flush=True)
                test_pred_idx = quad_rank_blend(
                    test_svm_m, test_deb_m, test_qwen_m, test_bge_m,
                    weights=best_w,
                    temperatures=calibrated_temps,
                    balance_test=True,
                    apply_family_gating=True,
                    family_alpha=best_alpha,
                )
                print("  [Final Blend] Generated test predictions via Quad-Rank Blend [OK]", flush=True)
        else:
            test_pred_idx = quad_rank_blend(
                test_svm_m, test_deb_m, test_qwen_m, test_bge_m,
                weights=best_w,
                temperatures=calibrated_temps,
                balance_test=True,
                apply_family_gating=True,
                family_alpha=best_alpha,
            )
            print("  [Final Blend] Generated test predictions via Quad-Rank Blend [OK]", flush=True)

        test_predictions = [LABELS[i] for i in test_pred_idx]

        sub_path = working_dir / "submission.parquet"
        validate_and_save_submission(test_df["id"].tolist(), test_predictions, sub_path)
    except Exception as e:
        import traceback
        print(f"  [Final Blend] Error during final test prediction / submission write: {e}", flush=True)
        traceback.print_exc()
        print("  [Final Blend] Preserving last successful incremental submission.parquet on disk.", flush=True)

    elapsed = time.perf_counter() - start_time
    print(f"\nTotal time: {elapsed:.1f}s ({elapsed/60:.2f} min)", flush=True)


if __name__ == "__main__":
    run_quad_ensemble()
